# Classificação Hiperespectral com MLOps

Um único notebook, autocontido, para comparar CNNs 1D, 2D e 3D em imagens hiperespectrais. O roteiro segue o fluxo real de trabalho: configurar, preparar os dados, treinar, rastrear e observar os resultados.

## Como usar este notebook

Para executar um experimento, rode as seções **1 a 7** em ordem e, depois, habilite `RUN_EXPERIMENT` na seção 8. As células de implementação ficam agrupadas por responsabilidade: você pode recolhê-las depois de uma primeira leitura e concentrar-se nas células de configuração e execução.

- Dados esperados: arquivos `.mat` em `data/` (por padrão, Indian Pines).
- Dependências: `requirements.txt` (versões pinadas e conferidas por `check_versions()` na seção 2). Instale com `uv pip install -r requirements.txt`.
- Saídas: `outputs/runs/` contém métricas, configuração, figuras, checkpoints e manifestos; `outputs/registry/` contém os modelos promovidos.

### Roteiro de navegação

1. **Base e configuração** — seções 1 e 2: caminhos, ambiente, GPU, determinismo e meta de treinamento.
2. **Dados e segmentação** — seção 3: PCA, segmentadores, diagnóstico visual e glossário.
3. **Modelos e treino** — seções 4 e 5: CNN1D/2D/3D, early stopping, interpretabilidade e HPO.
4. **Rastreabilidade** — seções 6 e 7: artefatos, campanhas e variantes comparáveis.
5. **Execução** — seção 8: habilite somente o painel/campanha desejado.
6. **Resultados** — seção 9: histórico auditável e comparação entre execuções.


## 1. Base do experimento

### 1.1 Caminhos, contrato de configuração e ambiente

Esta célula define caminhos, a configuração tipada e utilitários de ambiente. Em geral, não precisa ser alterada: a configuração prática está na próxima seção.

In [ ]:
# ============================================================================
# PATHS
# ============================================================================
"""Resolução de caminhos — funciona igual no Jupyter, no CLI e numa instância AWS.

O notebook original montava o Google Drive e usava `os.getcwd()`, o que quebrava
conforme o diretório em que o Jupyter tinha sido iniciado. Aqui a raiz do
repositório é descoberta subindo a árvore de diretórios.
"""


import os
from pathlib import Path

__all__ = ["repo_root", "data_dir", "output_dir"]


def repo_root(start: Path | None = None) -> Path:
    """Raiz do repositório, independente de onde o processo foi iniciado."""
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        if (base / ".git").is_dir() or (base / "scripts" / "download_data.py").exists():
            return base
    return here


def data_dir(root: Path | None = None) -> Path:
    """Diretório dos `.mat`.

    `HSI_DATA_DIR` tem precedência (útil para apontar um EBS/EFS na AWS); sem ela,
    `data/dataHSI/` dentro do repositório. O caminho do ambiente pode estar fora do
    repositório, por isso a raiz é descoberta à parte e não derivada daqui.
    """
    from_env = os.environ.get("HSI_DATA_DIR")
    if from_env:
        return Path(from_env).expanduser().resolve()
    return (root or repo_root()) / "data" / "dataHSI"


def output_dir(root: Path | None = None, create: bool = True) -> Path:
    """Diretório de saídas (curvas, checkpoints, relatórios)."""
    out = (root or repo_root()) / "outputs"
    if create:
        out.mkdir(parents=True, exist_ok=True)
    return out

# ============================================================================
# CONFIG
# ============================================================================
"""Configuração do experimento — tudo o que antes eram variáveis globais soltas.

No notebook original os hiperparâmetros ficavam espalhados por várias células
(`dataset`, `test_ratio`, `two_input`, `Kb`, `NUM_EPOCHS`, `BATCH_SIZE`, ...) e
eram lidos implicitamente por funções que dependiam do escopo global. Aqui viram
um objeto único, explícito e serializável.
"""


from dataclasses import dataclass, field, asdict

__all__ = ["ExperimentConfig", "DATASETS", "MODEL_TYPES", "SEGMENTERS"]

# Cenas suportadas: código curto -> nome legível.
DATASETS = {
    "IP": "Indian Pines",
    "PU": "Pavia University",
    "SA": "Salinas Valley",
}

# Famílias disponíveis. As strings ficam aqui (em vez de importar os módulos de
# modelo/segmentação) para que ExperimentConfig continue leve e sem dependências
# circulares.
#: Arquitetura do trabalho. O TCC é sobre CNNs 1D; 2D e 3D foram medidas e
#: retiradas do escopo em 24/09 (ver entrada 11 do diário de experimentos).
MODEL_TYPES = ("cnn1d",)
SEGMENTERS = ("seeds", "slic", "lsc", "watershed")

# Componentes de PCA por cena. O original usava `30 if dataset == 'IP' else 15`;
# Indian Pines precisa de mais componentes porque tem 200 bandas e 16 classes
# espectralmente próximas.
PCA_COMPONENTS = {"IP": 30, "PU": 15, "SA": 15}


@dataclass(frozen=True)
class ExperimentConfig:
    """Parâmetros de uma execução completa.

    Os valores padrão reproduzem o resultado de referência documentado em
    `SETUP.md` (PaviaU, duas entradas, SEED=42, modo determinístico).
    """

    dataset: str = "PU"
    #: True -> embedding + id do superpixel; False -> somente a entrada da CNN.
    two_input: bool = True
    #: Mantido como campo para que `config.json` continue registrando a
    #: arquitetura explicitamente, mesmo havendo só uma.
    model_type: str = "cnn1d"
    #: Algoritmo de superpixel usado quando `two_input=True`.
    segmenter: str = "seeds"
    #: Ajustes específicos rastreados do segmentador (preenchidos pelo HPO).
    segmenter_overrides: dict = field(default_factory=dict)
    #: Como o id da região entra na cabeça densa quando `two_input=True`.
    #: "scalar" concatena o id cru — é o baseline reproduzível, mas trata um
    #: rótulo nominal como número contínuo e depende da magnitude bruta do id.
    #: "embedding" aprende um vetor por região: sem ordem falsa entre ids e sem
    #: depender da escala. Ver a ablação da seção 5.7.
    fusion: str = "scalar"
    #: Dimensão do vetor aprendido por região quando `fusion="embedding"`.
    segment_embedding_dim: int = 8

    #: "pixel" = split aleatório estratificado por pixel. É o protocolo da
    #: literatura de HSI, mas deixa o mesmo superpixel presente em treino e em
    #: teste: um modelo com fusão pode memorizar "região R -> classe C".
    #: "region" = cada superpixel inteiro vai para uma partição só; elimina esse
    #: vazamento por construção e é o modo correto para auditar o ganho da fusão.
    #: "block" = blocos retangulares da cena; independe do segmentador, mas a
    #: disjunção é apenas aproximada (regiões cruzam a fronteira do bloco).
    split_mode: str = "pixel"
    #: Lado do bloco em pixels quando `split_mode="block"`. 0 usa 1/8 do menor
    #: lado da cena — bem maior que o superpixel típico, para que poucas regiões
    #: cruzem a fronteira.
    split_block: int = 0
    #: Validação cruzada por região. 0 mantém a partição aleatória única (que deixa
    #: poucas regiões no teste e produz desvio alto). Com k > 1, as regiões são
    #: divididas em k grupos: o grupo `split_fold` é o teste, o seguinte é a
    #: validação, e o resto treina. Percorrer `split_fold` de 0 a k-1 testa **cada
    #: região exatamente uma vez** — é o que torna o número confiável.
    split_folds: int = 0
    #: Qual grupo é o conjunto de teste, quando `split_folds > 1`.
    split_fold: int = 0

    #: Fração do conjunto de TREINO efetivamente usada, amostrada de forma
    #: estratificada por classe. 1.0 usa tudo; 0.10 usa um décimo. Validação e
    #: teste ficam intactos, então as métricas continuam comparáveis entre
    #: frações — muda o quanto o modelo vê, não como ele é avaliado.
    #: Treinar com poucas amostras rotuladas é, além de mais barato, o protocolo
    #: usual da literatura de HSI, onde rotular pixel é caro.
    train_fraction: float = 1.0

    #: Fração da base que sai do treino (vai para teste+validação).
    test_ratio: float = 0.15
    #: Fração de teste+validação que vira validação.
    valid_ratio: float = 0.5

    epochs: int = 60
    #: Batch físico enviado à GPU. Mantenha 32 para reproduzir o baseline; use o
    #: benchmark de batches para escolher um valor maior para a sua GPU.
    batch_size: int = 32
    #: Batch de validação/teste. Zero reutiliza `batch_size`; como não há
    #: gradientes nessa fase, este valor normalmente pode ser maior.
    eval_batch_size: int = 0
    #: Quantos batches físicos acumulam gradiente antes de `optimizer.step()`.
    #: O batch efetivo é `batch_size * gradient_accumulation_steps`.
    gradient_accumulation_steps: int = 1
    learning_rate: float = 1e-3
    #: Regularização L2 do Adam. 0.0 reproduz o baseline; o HPO varia este eixo
    #: para testar se penalizar pesos grandes ajuda em cenas com poucas amostras.
    weight_decay: float = 0.0
    #: O padrão do PyTorch é 1e-8; 1e-7 é o valor da implementação de referência
    #: e é mantido para que os resultados sejam comparáveis com os dela.
    adam_eps: float = 1e-7
    dropout: float = 0.5
    #: Janela de épocas consecutivas sem ganho significativo antes de parar.
    #: Ex.: 10 encerra após 10 épocas sem superar a melhor validação pelo Δ mínimo.
    patience: int = 10
    #: A métrica de validação que governa a parada; sua regra é salva no run.
    early_stopping_monitor: str = "val_accuracy"
    #: Ganho absoluto mínimo. 0.001 equivale a 0,1 ponto percentual de acurácia.
    early_stopping_min_delta: float = 1e-3
    #: Meta de validação. None usa somente patience; a receita fixa 0.95.
    early_stopping_target: float | None = None
    #: Recupera os pesos da melhor época, e não os da última época executada.
    restore_best_weights: bool = True

    seed: int = 42
    #: cudnn.deterministic + TF32 off. Troque para False só ao medir desempenho.
    deterministic: bool = True

    #: num_workers=0 é intencional por padrão: os tensores já estão em RAM e os
    #: batches são pequenos. Só aumente após medir em uma máquina com carga de
    #: I/O/preprocessamento adicional.
    num_workers: int = 0
    prefetch_factor: int = 2
    persistent_workers: bool = False

    #: Preenchido no __post_init__ a partir do dataset, se não for informado.
    pca_components: int = 0

    def __post_init__(self) -> None:
        if self.dataset not in DATASETS:
            raise ValueError(f"dataset inválido: {self.dataset!r}. Use um de {sorted(DATASETS)}.")
        if self.model_type not in MODEL_TYPES:
            raise ValueError(f"model_type inválido: {self.model_type!r}. Use um de {MODEL_TYPES}.")
        if self.segmenter not in SEGMENTERS:
            raise ValueError(f"segmenter inválido: {self.segmenter!r}. Use um de {SEGMENTERS}.")
        for name in ("epochs", "batch_size", "gradient_accumulation_steps", "prefetch_factor", "patience"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name} deve ser positivo.")
        if not 0 < self.train_fraction <= 1:
            raise ValueError("train_fraction deve estar em (0, 1].")
        if self.weight_decay < 0:
            raise ValueError("weight_decay não pode ser negativo.")
        if self.fusion not in {"scalar", "embedding"}:
            raise ValueError("fusion deve ser 'scalar' ou 'embedding'.")
        if self.segment_embedding_dim < 1:
            raise ValueError("segment_embedding_dim deve ser positivo.")
        if self.split_mode not in {"pixel", "region", "block"}:
            raise ValueError("split_mode deve ser 'pixel', 'region' ou 'block'.")
        if self.split_block < 0:
            raise ValueError("split_block não pode ser negativo.")
        if self.split_folds and self.split_folds < 3:
            raise ValueError("split_folds deve ser 0 (partição única) ou pelo menos 3.")
        if self.split_folds and not 0 <= self.split_fold < self.split_folds:
            raise ValueError("split_fold deve estar em [0, split_folds).")
        if self.num_workers < 0:
            raise ValueError("num_workers não pode ser negativo.")
        if self.persistent_workers and not self.num_workers:
            raise ValueError("persistent_workers requer num_workers maior que zero.")
        if self.eval_batch_size < 0:
            raise ValueError("eval_batch_size não pode ser negativo.")
        if self.early_stopping_monitor not in {"val_accuracy", "val_loss"}:
            raise ValueError("early_stopping_monitor deve ser 'val_accuracy' ou 'val_loss'.")
        if self.early_stopping_min_delta < 0:
            raise ValueError("early_stopping_min_delta não pode ser negativo.")
        if self.early_stopping_target is not None:
            if not np.isfinite(self.early_stopping_target):
                raise ValueError("early_stopping_target deve ser finito ou None.")
            if self.early_stopping_monitor == "val_accuracy" and not 0 <= self.early_stopping_target <= 1:
                raise ValueError("A meta de val_accuracy deve estar entre 0 e 1.")
        if not self.pca_components:
            object.__setattr__(self, "pca_components", PCA_COMPONENTS[self.dataset])

    @property
    def scene_name(self) -> str:
        return DATASETS[self.dataset]

    @property
    def model_name(self) -> str:
        """Prefixo dos arquivos de saída (era `TORCH_MODEL` no notebook)."""
        return f"{self.dataset}_CNN"

    @property
    def resolved_eval_batch_size(self) -> int:
        """Batch de inferência efetivamente usado pelos DataLoaders."""
        return self.eval_batch_size or self.batch_size

    @property
    def effective_batch_size(self) -> int:
        """Número de amostras que contribuem para cada atualização do Adam."""
        return self.batch_size * self.gradient_accumulation_steps

    def to_dict(self) -> dict:
        return asdict(self)

    def summary(self) -> str:
        modo = (
            f"{self.model_type} + segmentação ({self.segmenter}, fusão {self.fusion})"
            if self.two_input else f"{self.model_type} (sem fusão)"
        )
        return (
            f"{self.dataset} — {self.scene_name} | {modo}\n"
            f"PCA: {self.pca_components} componentes | "
            f"split: {1 - self.test_ratio:.0%} treino, "
            f"{self.test_ratio * (1 - self.valid_ratio):.0%} teste, "
            f"{self.test_ratio * self.valid_ratio:.0%} validação\n"
            f"épocas: {self.epochs} | early stopping: {self.early_stopping_monitor}, "
f"patience {self.patience}, Δ≥{self.early_stopping_min_delta:g}, "
                f"meta {self.early_stopping_target if self.early_stopping_target is not None else 'desativada'} | "
            f"batch físico/efetivo: {self.batch_size}/{self.effective_batch_size} | "
            f"lr: {self.learning_rate} | dropout: {self.dropout} | "
            f"weight decay: {self.weight_decay:g}\n"
            + (f"treino: {self.train_fraction:.0%} das amostras disponíveis\n"
               if self.train_fraction < 1 else "")
            + f"split: {self.split_mode}"
            + (f" (blocos de {self.split_block or 'auto'} px)" if self.split_mode == "block" else "")
            + (f" (regiões de {self.segmenter}" if self.split_mode == "region" else "")
            + (f", fold {self.split_fold + 1}/{self.split_folds})"
               if self.split_mode == "region" and self.split_folds
               else ")" if self.split_mode == "region" else "")
            + f"\nseed: {self.seed} | determinístico: {self.deterministic}"
        )

# ============================================================================
# ENV
# ============================================================================
"""Ambiente: conferência de versões, seleção de GPU, determinismo e sementes.

Duas armadilhas que estas funções detectam em vez de deixar estourar no meio do
treino:

- **RTX 50xx (Blackwell, `sm_120`)** só roda em build do PyTorch com CUDA 12.8+.
  Um wheel cu121/cu124 importa sem erro e falha no primeiro kernel com
  *"no kernel image is available for execution on the device"*.
- **`opencv-python` em vez de `opencv-contrib`**: `cv.ximgproc.createSuperpixelSEEDS`
  só existe no pacote contrib, e a falha só apareceria na célula de segmentação.
"""


import importlib.metadata as importlib_metadata
import os
import sys
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn

__all__ = ["EXPECTED_VERSIONS", "check_versions", "select_device", "configure_determinism",
           "configure_opencv_determinism", "seed_everything"]

#: Versões com que o projeto foi validado (as mesmas de `requirements.txt`).
EXPECTED_VERSIONS = {
    "python": "3.12.13",
    "torch": "2.13.0",
    "numpy": "2.5.1",
    "scipy": "1.18.0",
    "scikit-learn": "1.9.0",
    "pandas": "3.0.5",
    "matplotlib": "3.11.1",
    "seaborn": "0.13.2",
    "opencv-contrib-python-headless": "5.0.0.93",
}


def _installed_version(package: str) -> str | None:
    if package == "python":
        return sys.version.split()[0]
    try:
        return importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        # A variante com GUI serve igual; só não é a recomendada para servidor.
        if package.endswith("-headless"):
            try:
                return importlib_metadata.version(package[: -len("-headless")]) + " (com GUI)"
            except importlib_metadata.PackageNotFoundError:
                pass
        return None


def check_versions(verbose: bool = True) -> list[str]:
    """Compara o ambiente com `EXPECTED_VERSIONS`. Só avisa — não interrompe.

    Retorna a lista de pacotes divergentes ou ausentes.
    """
    import cv2 as cv

    divergent = []
    if verbose:
        print(f"{'pacote':32s} {'esperado':14s} {'instalado':22s} status")
        print("-" * 80)

    for package, expected in EXPECTED_VERSIONS.items():
        found = _installed_version(package)
        if found is None:
            status = "AUSENTE"
        else:
            base = found.split("+")[0].split(" ")[0]  # ignora '+cu130' e sufixos
            status = "ok" if base == expected else "DIVERGENTE"
        if status != "ok":
            divergent.append(package)
        if verbose:
            print(f"{package:32s} {expected:14s} {str(found):22s} {status}")

    if verbose:
        # O build CUDA vem do índice usado na instalação, não do requirements.txt.
        print("-" * 80)
        cuda_build = torch.version.cuda
        if cuda_build is None:
            cuda_status = "CPU-only (sem GPU)"
        elif tuple(map(int, cuda_build.split("."))) >= (12, 8):
            cuda_status = "ok (suporta sm_120)"
        else:
            cuda_status = "INCOMPATÍVEL com sm_120 (RTX 50xx)"
        print(f"{'torch CUDA build':32s} {'>= 12.8':14s} {str(cuda_build):22s} {cuda_status}")
        print(
            f"{'cv2 / ximgproc (SEEDS)':32s} {'-':14s} {cv.__version__:22s} "
            f"{'ok' if hasattr(cv, 'ximgproc') else 'AUSENTE: instale opencv-contrib'}"
        )
        if divergent:
            print("\nDivergências:", ", ".join(divergent))
            print("Para reproduzir exatamente: uv pip install -r requirements.txt   (ver SETUP.md)")
        else:
            print("\nAmbiente idêntico ao de validação.")

    return divergent


def _arch_number(arch: str, prefix: str) -> int | None:
    value = arch[len(prefix):]
    return int(value) if arch.startswith(prefix) and value.isdigit() else None


def _build_supports_gpu(capability: tuple[int, int], arch_list: list[str]) -> tuple[str, str]:
    """O build do PyTorch atende esta GPU? Retorna (nível, explicação).

    Três formas de atender, em ordem de preferência:
      1. cubin exato para sm_XY;
      2. cubin sm_XZ do mesmo major com Z <= Y — a compatibilidade binária do CUDA
         vale entre minors do mesmo major (sm_86 numa L4 sm_89), nunca entre majors;
      3. PTX (compute_*) de versão <= a da GPU, compilado por JIT no primeiro uso.
    """
    major, minor = capability
    sm = f"sm_{major}{minor}"

    if sm in arch_list:
        return "ok", f"Build tem kernels nativos para {sm}."

    numbers = [_arch_number(a, "sm_") for a in arch_list]
    if any(n is not None and n // 10 == major and n % 10 <= minor for n in numbers):
        return "ok", (
            f"Sem cubin {sm} exato, mas há um do mesmo major reutilizável "
            "(compatibilidade binária CUDA)."
        )

    ptx = [_arch_number(a, "compute_") for a in arch_list]
    if any(n is not None and n <= major * 10 + minor for n in ptx):
        return "aviso", f"{sm} não está no build; deve funcionar via JIT de PTX (primeira execução mais lenta)."

    return "erro", f"Este build do PyTorch não suporta {sm}."


#: Prefixo impresso para cada nível de suporte devolvido por `_build_supports_gpu`.
_SUPPORT_TAG = {"ok": "ok", "aviso": "!", "erro": "ERRO"}


def select_device(verbose: bool = True) -> torch.device:
    """Escolhe GPU se houver, reportando (e testando) se ela realmente executa kernels."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if verbose:
        print("Python: ", sys.version.split()[0])
        print("PyTorch:", torch.__version__)
        print("Device: ", device)

    if device.type != "cuda":
        if verbose:
            print("GPU: None — o treino vai rodar em CPU (mais lento, porém funcional).")
        return device

    props = torch.cuda.get_device_properties(0)
    capability = torch.cuda.get_device_capability(0)
    arch_list = torch.cuda.get_arch_list()
    level, message = _build_supports_gpu(capability, arch_list)

    if verbose:
        print("GPU:            ", props.name)
        print("Compute cap.:   ", f"{capability[0]}.{capability[1]}", f"(sm_{capability[0]}{capability[1]})")
        print("SMs / memória:  ", props.multi_processor_count, "SMs |", f"{props.total_memory / 1024**3:.2f} GB")
        print("CUDA runtime:   ", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
        print("Build compilado:", " ".join(arch_list))
        print(f"\n[{_SUPPORT_TAG[level]}] {message}")
        if level == "erro" and capability[0] >= 12:
            print("       RTX 50xx (Blackwell) exige wheel CUDA 12.8+:")
            print("       uv pip install torch --index-url https://download.pytorch.org/whl/cu130")

    # Prova real: um kernel de convolução igual ao usado no modelo.
    try:
        probe = nn.Conv1d(1, 16, 9).to(device)
        out = probe(torch.zeros(2, 1, 32, device=device))
        torch.cuda.synchronize()
        if verbose:
            print("[ok] Convolução de teste executada na GPU:", tuple(out.shape))
        del probe, out
        torch.cuda.empty_cache()
    except RuntimeError as error:
        print("[ERRO] A GPU não executou o kernel de teste:", error)

    return device


def configure_opencv_determinism(verbose: bool = True) -> None:
    """Força o OpenCV a uma thread só, para que a segmentação seja determinística.

    Motivo, medido: `createSuperpixelLSC` é paralelizado e seu desempate depende da
    ordem de execução das threads. Duas chamadas com a MESMA imagem e os MESMOS
    parâmetros devolviam rótulos diferentes em ~390 pixels de Indian Pines. SEEDS,
    SLIC e Watershed não têm o problema.

    A consequência era silenciosa e cara: com o LSC no espaço de busca, o score de
    um mesmo trial variava entre execuções, e as estratégias adaptativas (TPE e o
    algoritmo genético) amplificavam essa variação até trocarem de vencedor. Foi o
    que fez a mesma campanha dar 0,8713 numa execução e 0,7711 em outra.

    O custo é nulo nestas cenas: 38,6 ms por segmentação com uma thread contra
    39,1 ms com 32 — as imagens são pequenas demais para a paralelização compensar.
    """
    import cv2 as cv

    cv.setNumThreads(1)
    if verbose:
        print("OpenCV: 1 thread (determinismo do LSC)")


def configure_determinism(deterministic: bool = True, verbose: bool = True) -> None:
    """Troca explícita entre reprodutibilidade e velocidade.

    `cudnn.benchmark=True` mede kernels e escolhe o mais rápido por shape, mas a
    escolha pode mudar entre execuções e com ela os últimos dígitos do resultado.
    TF32 acelera as densas (Ampere+) truncando a mantissa: é determinístico, porém
    dá números diferentes de FP32 puro.
    """
    if deterministic:
        configure_opencv_determinism(verbose=verbose)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # cuBLAS determinístico
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.use_deterministic_algorithms(True, warn_only=True)
        if verbose:
            print("\nModo determinístico: cudnn.deterministic=True, benchmark=False, TF32 off")
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        if verbose:
            print("\nModo desempenho: cudnn.benchmark=True, TF32 on (resultado pode variar)")


def seed_everything(seed: int = 42) -> None:
    """Semeia NumPy e PyTorch (CPU e todas as GPUs).

    Deve ser chamada imediatamente antes de construir o modelo: é a construção
    que consome o RNG do PyTorch e define os pesos iniciais.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Compatibilidade com os nomes usados nas funções abaixo.
default_data_dir = data_dir


## 2. Escolher ambiente e configuração

### 2.1 Ambiente, GPU e reprodutibilidade

A configuração abaixo é o principal ponto de ajuste. Comece com poucas épocas para validar o fluxo; depois aumente `epochs` e ajuste `batch_size` conforme a memória da GPU.

### 2.2 Meta de treinamento e early stopping

**Meta atual:** atingir `val_accuracy ≥ 0,95`. Caso a meta não seja alcançada, o treino é encerrado depois de **10 épocas** sem melhora de pelo menos **0,001** (0,1 ponto percentual) em `val_accuracy`. A validação governa tanto a meta quanto a parada; o teste permanece reservado para o relato final. Ajuste `early_stopping_target`, `patience` e `early_stopping_min_delta` juntos.


In [ ]:
check_versions()
device = select_device(verbose=True)
configure_determinism(deterministic=True)
seed_everything(seed=42)

config = ExperimentConfig(
    dataset='IP',
    segmenter='seeds',          # opções: seeds, slic, lsc
    two_input=True,             # concatena espectro e segmento no CNN1D
    pca_components=30,
    batch_size=128,
    epochs=60,
    # Meta: 95% na validação OU 10 épocas sem ganho de 0,1 p.p.
    early_stopping_monitor='val_accuracy',
    early_stopping_target=0.95,
    patience=10,
    early_stopping_min_delta=0.001,
    restore_best_weights=True,
    num_workers=0,              # aumente apenas após validar o SO/ambiente
)
print(f'Dispositivo: {device} | arquitetura: {config.model_type} | segmentador: {config.segmenter}')


## 3. Preparar os dados e o contexto espacial

### 3.1 Carregamento, normalização e divisões

A preparação transforma a cena em amostras rotuladas, reduz as bandas com PCA, cria superpixels e, para CNNs 2D/3D, extrai patches. A função `prepare_data_parallel` permite executar PCA e segmentação em paralelo por threads ou processos antes da fusão.

In [ ]:
# ============================================================================
# DATA
# ============================================================================
"""Carga das cenas hiperespectrais e transformação para tabela de pixels."""


import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import scipy.io as sio


__all__ = ["Scene", "SCENE_FILES", "ensure_dataset", "load_scene", "to_pixel_table", "remove_background"]

# Arquivo e chave do dicionário `.mat` de cada cena. O notebook original repetia
# isso num if/elif de três ramos com o caminho montado na mão.
SCENE_FILES = {
    "IP": {
        "image": ("Indian_pines_corrected.mat", "indian_pines_corrected"),
        "labels": ("Indian_pines_gt.mat", "indian_pines_gt"),
    },
    "PU": {
        "image": ("PaviaU.mat", "paviaU"),
        "labels": ("PaviaU_gt.mat", "paviaU_gt"),
    },
    "SA": {
        "image": ("Salinas_corrected.mat", "salinas_corrected"),
        "labels": ("Salinas_gt.mat", "salinas_gt"),
    },
}


@dataclass(frozen=True)
class Scene:
    """Cena hiperespectral crua: cubo (H, W, B) e ground truth (H, W)."""

    code: str
    image: np.ndarray
    labels: np.ndarray

    @property
    def shape(self) -> tuple[int, int]:
        return self.image.shape[0], self.image.shape[1]

    @property
    def n_bands(self) -> int:
        return self.image.shape[2]

    @property
    def n_classes(self) -> int:
        """Classes reais, sem contar o background (rótulo 0)."""
        return int(self.labels.max())

    def __str__(self) -> str:
        return (
            f"{self.code}: cubo {self.image.shape} {self.image.dtype} | "
            f"gt {self.labels.shape} {self.labels.dtype} | {self.n_classes} classes"
        )


def ensure_dataset(dataset: str, directory: Path | None = None) -> Path:
    """Garante que os `.mat` da cena estão em disco, baixando se faltarem.

    Delega para `scripts/download_data.py`, que valida sha256 e o conteúdo de
    cada arquivo. Retorna o diretório onde os dados estão.
    """
    directory = Path(directory) if directory else default_data_dir()
    required = [name for name, _ in SCENE_FILES[dataset].values()]
    missing = [f for f in required if not (directory / f).exists()]
    if not missing:
        return directory

    script = repo_root() / "scripts" / "download_data.py"
    if not script.exists():
        raise FileNotFoundError(
            f"Arquivos ausentes em {directory}: {missing}. "
            f"{script} não encontrado — baixe manualmente (ver SETUP.md) ou defina HSI_DATA_DIR."
        )
    directory.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [sys.executable, str(script), "--dataset", dataset, "--out", str(directory)],
        check=True,
    )
    return directory


def load_scene(dataset: str, directory: Path | None = None, download: bool = True) -> Scene:
    """Carrega cubo e ground truth de uma cena."""
    if dataset not in SCENE_FILES:
        raise ValueError(f"Dataset inválido: {dataset!r}. Use IP, PU ou SA.")
    directory = ensure_dataset(dataset, directory) if download else (Path(directory) if directory else default_data_dir())

    spec = SCENE_FILES[dataset]
    (img_file, img_key), (gt_file, gt_key) = spec["image"], spec["labels"]
    image = sio.loadmat(directory / img_file)[img_key]
    labels = sio.loadmat(directory / gt_file)[gt_key]
    return Scene(code=dataset, image=image, labels=labels)


def to_pixel_table(scene: Scene) -> tuple[np.ndarray, np.ndarray]:
    """Cubo (H, W, B) -> tabela (H*W, B) para classificação por pixel.

    Retorna também os rótulos achatados (H*W,), ainda com o background.
    """
    h, w, bands = scene.image.shape
    return scene.image.reshape(h * w, bands), scene.labels.reshape(h * w)


def remove_background(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Descarta as amostras de background (classe 0) e reindexa as classes para 0..C-1.

    Funciona tanto para a tabela espectral (N, B) quanto para uma coluna auxiliar
    (N, 1) — o filtro é sempre pelo vetor de rótulos `y`.
    """
    keep = y > 0
    x_out, y_out = x[keep], y[keep]
    return x_out, y_out - 1

# ============================================================================
# PREPROCESS
# ============================================================================
"""Redução de dimensionalidade (PCA) e composição RGB para a segmentação."""


import numpy as np
from sklearn.decomposition import PCA

__all__ = ["apply_pca", "mat2gray", "rgb_composite"]


def apply_pca(x: np.ndarray, n_components: int, random_state: int = 42, whiten: bool = True):
    """Reduz o número de bandas mantendo as componentes de maior variância.

    `whiten=True` e `random_state` fixo reproduzem o notebook original — o
    `random_state` importa porque, nestes shapes, o sklearn escolhe o solver
    randomizado.

    Retorna `(X_transformado, modelo_ajustado)`.
    """
    model = PCA(n_components=n_components, whiten=whiten, random_state=random_state)
    model.fit(x)
    return model.transform(x), model


def mat2gray(a: np.ndarray) -> np.ndarray:
    """Normaliza para [0, 1] (equivalente do `mat2gray` do MATLAB)."""
    lo, hi = np.min(a), np.max(a)
    return (a - lo) / (hi - lo)


def rgb_composite(components: np.ndarray, shape: tuple[int, int]) -> np.ndarray:
    """Três primeiras componentes principais -> imagem RGB uint8 (H, W, 3).

    É essa imagem que alimenta o SEEDS: o algoritmo de superpixel trabalha sobre
    uma imagem de 3 canais, não sobre o cubo inteiro.
    """
    if components.shape[1] < 3:
        raise ValueError(f"São necessárias 3 componentes, recebidas {components.shape[1]}.")
    height, width = shape
    channels = [(255 * mat2gray(components[:, i])).astype(np.uint8).reshape(height, width) for i in range(3)]
    return np.dstack(channels)

# ============================================================================
# SEGMENTATION
# ============================================================================
"""Segmentação espacial com SEEDS, SLIC, LSC (OpenCV contrib) e Watershed.

O id da região de cada pixel vira a *segunda entrada* da rede: além do espectro,
o modelo recebe a que região homogênea aquele pixel pertence.

Os quatro algoritmos devolvem o mesmo contrato — `SegmentationResult` com labels
e máscara de contorno — e por isso DataLoader, CNNs e métricas não mudam quando o
segmentador muda. SEEDS/SLIC/LSC são superpixels do `cv.ximgproc` e exigem
`opencv-contrib-python[-headless]`; Watershed usa bacias de gradiente do OpenCV
base, iniciado por uma malha regular de marcadores, e é a alternativa
conceitualmente distinta do conjunto.

A tupla canônica `SEGMENTERS` é definida na seção 1 (contrato de configuração),
porque `ExperimentConfig` a valida; aqui ficam apenas os parâmetros e a execução.
"""


from dataclasses import dataclass

import numpy as np

__all__ = [
    "SeedsParams", "SlicParams", "LscParams", "WatershedParams",
    "SEEDS_PARAMS", "SLIC_PARAMS", "LSC_PARAMS", "WATERSHED_PARAMS",
    "SegmentationResult", "SegmentationArtifacts", "segment_superpixels",
    "superpixel_labels", "contour_overlay", "colorize_segments",
]


@dataclass(frozen=True)
class SeedsParams:
    """Parâmetros do `cv.ximgproc.createSuperpixelSEEDS`.

    num_superpixels é um alvo, não uma garantia: o algoritmo pode devolver menos.
    """

    num_iterations: int = 150
    prior: int = 1  # suavização de forma 3x3 quando > 0; válido em [0, 5]
    double_step: bool = True  # itera cada nível de bloco duas vezes (mais preciso)
    num_superpixels: int = 150
    num_levels: int = 8
    num_histogram_bins: int = 15


# Indian Pines tem 145x145 e classes agrícolas em faixas estreitas: precisa de mais
# superpixels e mais iterações. PaviaU e Salinas usam o mesmo conjunto.
SEEDS_PARAMS = {
    "IP": SeedsParams(
        num_iterations=200, prior=2, double_step=True,
        num_superpixels=350, num_levels=10, num_histogram_bins=5,
    ),
    "PU": SeedsParams(),
    "SA": SeedsParams(),
}


@dataclass(frozen=True)
class SlicParams:
    """Parâmetros do SLICO do OpenCV.

    ``region_size`` é o lado aproximado de cada superpixel. SLICO adapta a
    compacidade automaticamente e, por isso, é uma boa comparação com SEEDS sem
    introduzir mais um hiperparâmetro de cor/forma.
    """

    region_size: int = 12
    ruler: float = 10.0
    num_iterations: int = 10


@dataclass(frozen=True)
class LscParams:
    """Parâmetros do Linear Spectral Clustering (LSC) do OpenCV."""

    region_size: int = 12
    ratio: float = 0.075
    num_iterations: int = 10


@dataclass(frozen=True)
class WatershedParams:
    """Marcadores aproximam o número de regiões desejado no Watershed do OpenCV."""

    markers: int = 350


SLIC_PARAMS = {"IP": SlicParams(region_size=8), "PU": SlicParams(), "SA": SlicParams()}
LSC_PARAMS = {"IP": LscParams(region_size=8), "PU": LscParams(), "SA": LscParams()}
WATERSHED_PARAMS = {
    "IP": WatershedParams(markers=350),
    "PU": WatershedParams(markers=500),
    "SA": WatershedParams(markers=500),
}


@dataclass(frozen=True)
class SegmentationResult:
    """Saída da segmentação, em arrays puros.

    Guardar a máscara de contorno aqui — em vez do objeto SEEDS do OpenCV, que
    não é serializável — é o que permite rodar a segmentação num processo
    separado e trazer o resultado de volta (ver `hsi.parallel`).
    """

    labels: np.ndarray  # (H, W) int32 — id do superpixel de cada pixel
    contour_mask: np.ndarray  # (H, W) uint8 — bordas entre superpixels

    @property
    def n_superpixels(self) -> int:
        return len(np.unique(self.labels))


@dataclass(frozen=True)
class SegmentationArtifacts:
    """Tudo o que o ramo espacial produz, incluindo os intermediários visuais.

    Só os gráficos precisam de `components` e `rgb`; o pipeline em si usa apenas os
    ids de superpixel. Todos são arrays puros, então a estrutura atravessa a
    fronteira de um processo sem problema.
    """

    components: np.ndarray  # (H*W, 3) — as 3 primeiras componentes principais
    rgb: np.ndarray  # (H, W, 3) uint8 — entrada do SEEDS
    result: "SegmentationResult"

    @property
    def labels(self) -> np.ndarray:
        return self.result.labels

    @property
    def contour_mask(self) -> np.ndarray:
        return self.result.contour_mask

    @property
    def n_superpixels(self) -> int:
        return self.result.n_superpixels


def _boundary_mask(labels: np.ndarray) -> np.ndarray:
    """Converte transições entre regiões em máscara de contorno uint8.

    O Watershed do OpenCV não expõe ``getLabelContourMask``; derivar a máscara das
    diferenças entre vizinhos mantém o mesmo contrato visual dos superpixels.
    """
    mask = np.zeros(labels.shape, dtype=np.uint8)
    mask[:, 1:] |= labels[:, 1:] != labels[:, :-1]
    mask[1:, :] |= labels[1:, :] != labels[:-1, :]
    return (255 * mask).astype(np.uint8)


def _watershed_segments(rgb: np.ndarray, params: WatershedParams) -> SegmentationResult:
    """Executa Watershed com uma malha de sementes regulares e reindexa as regiões."""
    import cv2 as cv

    height, width = rgb.shape[:2]
    target = max(1, params.markers)
    rows = max(1, int(round(np.sqrt(target * height / width))))
    cols = max(1, int(round(target / rows)))
    markers = np.zeros((height, width), dtype=np.int32)
    marker_id = 1
    for y in np.linspace(0, height - 1, rows, dtype=int):
        for x in np.linspace(0, width - 1, cols, dtype=int):
            markers[y, x] = marker_id
            marker_id += 1
    raw = cv.watershed(cv.cvtColor(rgb, cv.COLOR_RGB2BGR), markers)
    raw[raw < 0] = 0  # -1 é o contorno especial retornado pelo OpenCV.
    _, labels = np.unique(raw, return_inverse=True)
    labels = labels.reshape(raw.shape).astype(np.int32)
    return SegmentationResult(labels=labels, contour_mask=_boundary_mask(labels))


def segment_superpixels(
    rgb: np.ndarray,
    algorithm: str,
    params: SeedsParams | SlicParams | LscParams | WatershedParams,
) -> SegmentationResult:
    """Segmenta RGB com o algoritmo escolhido e devolve labels/contornos puros.

    Args:
        rgb: Composição uint8 ``(H, W, 3)`` construída das componentes de PCA.
        algorithm: ``seeds``, ``slic`` (SLICO), ``lsc`` ou ``watershed``.
        params: Dataclass correspondente ao algoritmo e à cena.

    A saída uniforme permite comparar algoritmos sem alterar DataLoader, modelo ou
    métricas. Watershed é resolvido antes da checagem do ``ximgproc`` porque é o
    único que não depende do pacote contrib.
    """
    import cv2 as cv

    if algorithm == "watershed":
        if not isinstance(params, WatershedParams):
            raise TypeError("Watershed requer WatershedParams.")
        return _watershed_segments(rgb, params)
    if not hasattr(cv, "ximgproc"):
        raise ImportError(
            "cv.ximgproc ausente: instale opencv-contrib-python-headless "
            "(o pacote opencv-python comum não traz o SEEDS)."
        )
    if algorithm == "seeds":
        if not isinstance(params, SeedsParams):
            raise TypeError("SEEDS requer SeedsParams.")
        height, width, channels = rgb.shape
        segmenter = cv.ximgproc.createSuperpixelSEEDS(
            width, height, channels,
            params.num_superpixels, params.num_levels, params.prior,
            params.num_histogram_bins, params.double_step,
        )
    elif algorithm == "slic":
        if not isinstance(params, SlicParams):
            raise TypeError("SLIC requer SlicParams.")
        segmenter = cv.ximgproc.createSuperpixelSLIC(
            rgb, algorithm=cv.ximgproc.SLICO, region_size=params.region_size, ruler=params.ruler,
        )
    elif algorithm == "lsc":
        if not isinstance(params, LscParams):
            raise TypeError("LSC requer LscParams.")
        segmenter = cv.ximgproc.createSuperpixelLSC(rgb, region_size=params.region_size, ratio=params.ratio)
    else:
        raise ValueError(f"segmentador inválido: {algorithm!r}. Use um de {SEGMENTERS}.")

    # A API do SEEDS recebe a imagem em cada ``iterate``; SLIC/LSC recebem a
    # imagem no construtor e aceitam somente o número de iterações.
    if algorithm == "seeds":
        segmenter.iterate(rgb, params.num_iterations)
    else:
        segmenter.iterate(params.num_iterations)
    return SegmentationResult(
        labels=segmenter.getLabels(), contour_mask=segmenter.getLabelContourMask(False),
    )


def superpixel_labels(rgb: np.ndarray, params: SeedsParams) -> SegmentationResult:
    """Compatibilidade com a API anterior: executa especificamente SEEDS."""
    return segment_superpixels(rgb, "seeds", params)


def contour_overlay(rgb: np.ndarray, segmentation) -> dict[str, np.ndarray]:
    """Imagens intermediárias da segmentação, para inspeção visual.

    Aceita tanto um `SegmentationResult` quanto um `SegmentationArtifacts`.
    """
    import cv2 as cv

    mask = segmentation.contour_mask
    mask_inv = cv.bitwise_not(mask)
    background = cv.bitwise_and(rgb, rgb, mask=mask_inv)
    foreground = cv.bitwise_and(rgb, rgb, mask=mask)
    return {
        "mask_inv": mask_inv,
        "result_bg": background,
        "result_fg": foreground,
        "result": cv.add(background, foreground),
    }


def colorize_segments(labels: np.ndarray, seed: int = 42) -> np.ndarray:
    """Pinta cada superpixel com uma cor aleatória (só para visualização).

    Usa um gerador próprio para não mexer no RNG global do NumPy — no notebook
    original esta visualização consumia o mesmo `np.random` do resto do pipeline.
    """
    rng = np.random.default_rng(seed)
    unique = np.unique(labels)
    palette = rng.integers(0, 256, size=(unique.max() + 1, 3), dtype=np.uint8)
    return palette[labels]

# ============================================================================
# DATASET
# ============================================================================
"""Divisão treino/validação/teste e construção dos DataLoaders.

O notebook original empilhava espectro e segmentação num tensor `(N, Kb, 1, 1, 2)`
só para conseguir passar os dois pelo mesmo `train_test_split`, e depois desfazia
o empilhamento com fatiamento posicional. Aqui divide-se o **índice** uma vez e
cada array é indexado por ele — mesma partição, sem o malabarismo de shape.
"""


from dataclasses import dataclass

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

__all__ = ["Splits", "PreparedData", "split_indices", "validate_prepared_data", "build_loaders"]


@dataclass(frozen=True)
class Splits:
    """Índices das três partições, referentes às linhas de `PreparedData`."""

    train: np.ndarray
    val: np.ndarray
    test: np.ndarray

    def sizes(self) -> dict[str, int]:
        return {"treino": len(self.train), "validação": len(self.val), "teste": len(self.test)}


@dataclass(frozen=True)
class PreparedData:
    """Dados prontos para o modelo, já sem background e com PCA aplicado.

    spectra:  (N, Kb) float64 — componentes principais do espectro
    labels:   (N,)    int64   — classes reindexadas para 0..C-1
    segments: (N,)    ou None — id do superpixel de cada pixel (segunda entrada)
    """

    spectra: np.ndarray
    labels: np.ndarray
    segments: np.ndarray | None
    splits: Splits
    scene_shape: tuple[int, int]
    #: (N, 2) linha/coluna de cada amostra na cena. Necessário para o split por
    #: blocos espaciais e para projetar previsões de volta na imagem.
    coords: np.ndarray | None = None

    @property
    def n_features(self) -> int:
        return self.spectra.shape[1]

    @property
    def n_classes(self) -> int:
        return int(self.labels.max()) + 1

    @property
    def two_input(self) -> bool:
        return self.segments is not None

    @property
    def n_segments(self) -> int:
        """Quantas linhas a tabela de embeddings precisa ter.

        Usa `max + 1`, e não a contagem de valores distintos, porque os ids saem
        do segmentador antes da remoção do background e ficam com buracos: em
        Indian Pines há 63 regiões rotuladas, mas o maior id é 78. Linhas não
        usadas simplesmente nunca recebem gradiente.
        """
        return 0 if self.segments is None else int(np.max(self.segments)) + 1

    def input_shape(self, model_type: str = "cnn1d") -> tuple[int, ...]:
        """Shape de uma amostra no layout channels-first da CNN1D."""
        return (1, self.n_features)


def split_indices(labels: np.ndarray, test_ratio: float, valid_ratio: float, seed: int = 42) -> Splits:
    """Split estratificado em duas etapas, como no original.

    Primeiro separa treino de (validação + teste); depois reparte esse resto.
    `stratify` mantém a proporção de cada classe nas três partições — necessário
    porque as cenas são bastante desbalanceadas.
    """
    index = np.arange(len(labels))
    train, rest = train_test_split(
        index, test_size=test_ratio, random_state=seed, stratify=labels
    )
    # Atenção à ordem: no original o primeiro retorno desta segunda chamada é o
    # conjunto de *teste* e o segundo é o de validação.
    test, val = train_test_split(
        rest, test_size=valid_ratio, random_state=seed, stratify=labels[rest]
    )
    return Splits(train=train, val=val, test=test)


def _split_by_groups(labels, group_id, test_ratio, valid_ratio, seed=42, attempts=64):
    """Sorteia grupos inteiros para cada partição, maximizando classes no treino.

    Um "grupo" é um bloco espacial ou um superpixel: a unidade que não pode ser
    dividida entre treino e teste. Estratificação exata é impossível — uma classe
    rara pode viver dentro de um único grupo — então a busca tenta várias
    permutações e devolve a que cobre mais classes no treino, com um relatório do
    que ficou de fora.
    """
    labels = np.asarray(labels)
    group_id = np.asarray(group_id)
    groups, sizes = np.unique(group_id, return_counts=True)
    size_of = dict(zip(groups.tolist(), sizes.tolist()))
    n_samples, classes = len(labels), np.unique(labels)
    rng = np.random.default_rng(seed)

    best = None
    for _ in range(attempts):
        order = rng.permutation(groups)
        held, accumulated = [], 0
        for candidate in order:
            if accumulated >= test_ratio * n_samples:
                break
            held.append(candidate)
            accumulated += size_of[candidate]
        validation, accumulated_val = [], 0
        for candidate in held:
            if accumulated_val >= valid_ratio * accumulated:
                break
            validation.append(candidate)
            accumulated_val += size_of[candidate]
        train_mask = ~np.isin(group_id, held)
        val_mask = np.isin(group_id, validation)
        test_mask = ~train_mask & ~val_mask
        if not (train_mask.any() and val_mask.any() and test_mask.any()):
            continue
        score = (len(np.unique(labels[train_mask])),
                 -abs(test_mask.mean() + val_mask.mean() - test_ratio))
        if best is None or score > best[0]:
            best = (score, train_mask, val_mask, test_mask, len(held))
    if best is None:
        raise ValueError(
            "Não foi possível formar três partições com grupos disjuntos. "
            "Use grupos menores (reduza `split_block`) ou aumente `test_ratio`."
        )

    _, train_mask, val_mask, test_mask, held_groups = best
    report = {
        "groups_total": int(len(groups)), "groups_held_out": int(held_groups),
        "train_fraction": float(train_mask.mean()), "val_fraction": float(val_mask.mean()),
        "test_fraction": float(test_mask.mean()),
        "classes_missing_from_train": sorted(
            set(classes.tolist()) - set(np.unique(labels[train_mask]).tolist())
        ),
    }
    splits = Splits(train=np.flatnonzero(train_mask), val=np.flatnonzero(val_mask),
                    test=np.flatnonzero(test_mask))
    return splits, report


def split_indices_regions(labels, region_ids, test_ratio, valid_ratio, seed=42) -> tuple[Splits, dict]:
    """Split disjunto por superpixel: uma região inteira vai para uma partição só.

    É o teste exato do vazamento espacial. No split por pixel, o mesmo superpixel
    aparece em treino e em teste, então um modelo com fusão pode memorizar
    "região R -> classe C" e aplicar isso a pixels de teste da mesma região —
    o que infla o ganho atribuído à fusão. Aqui `segment_leakage` devolve zero
    por construção.

    Limitação a declarar no texto: a partição passa a depender do segmentador,
    então comparar segmentadores diferentes sob este modo compara também splits
    diferentes. Para a pergunta "com fusão × sem fusão" isso não é problema, desde
    que os dois braços usem o mesmo segmentador para definir o split.
    """
    splits, report = _split_by_groups(labels, region_ids, test_ratio, valid_ratio, seed)
    return splits, {"mode": "region", **report}


def split_indices_region_folds(labels, region_ids, n_folds: int, fold: int,
                               valid_ratio: float = 0.15, seed: int = 42) -> tuple[Splits, dict]:
    """Validação cruzada por região: o grupo `fold` é o teste, disjunto por região.

    Por que isto existe. `split_indices_regions` sorteia **uma** partição, e com
    `test_ratio=0,15` sobram 3 ou 4 regiões no teste — trocar uma região por outra
    muda quase tudo, e o desvio entre sementes chegou a ±0,25 em Pavia. O número era
    inutilizável. Percorrendo `fold` de 0 a n_folds-1, **cada região é testada
    exatamente uma vez** e a média usa a cena inteira.

    Usa `StratifiedGroupKFold`: mantém cada região inteira numa partição só e ainda
    assim tenta preservar a proporção das classes. Um `GroupKFold` puro ignora as
    classes e deixa mais delas de fora do treino.

    **Decisão deliberada: a validação NÃO é disjunta por região.** Ela é uma amostra
    estratificada por pixel do conjunto de treino. O motivo é medido: separar também
    a validação por região triplicou as classes ausentes do treino (de 4 para 16
    ocorrências ao longo dos 5 folds em Indian Pines), porque ela rouba justamente as
    regiões das classes raras.

    O invariante que protege o número reportado é a disjunção **treino × teste**, e
    essa está garantida. A validação compartilhar regiões com o treino a torna
    otimista, o que pode fazer o early stopping parar num ponto que memoriza mais —
    mas esse modelo então vai **pior** no teste disjunto. O viés é pessimista sobre o
    resultado final, nunca otimista, que é o lado seguro de errar.

    **Limitação que o protocolo não resolve.** Em Indian Pines, 6 das 16 classes vivem
    em uma ou duas regiões apenas — as classes 0, 6, 8 e 12 estão inteiramente dentro
    de *uma* região. Quando essa região cai no teste, a classe é impossível de
    aprender e sua acurácia é zero por construção, derrubando a AA. O relatório
    devolve `classes_missing_from_train` para que a métrica seja lida com essa
    informação, em vez de parecer um fracasso do modelo.
    """
    from sklearn.model_selection import StratifiedGroupKFold

    labels = np.asarray(labels)
    region_ids = np.asarray(region_ids)
    regioes = np.unique(region_ids)
    if len(regioes) < n_folds:
        raise ValueError(
            f"{len(regioes)} regiões não dão para {n_folds} folds. "
            "Aumente o número de superpixels do segmentador ou reduza n_folds."
        )
    vazio = np.zeros(len(labels))
    divisor = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    resto_idx, teste_idx = list(divisor.split(vazio, labels, groups=region_ids))[fold % n_folds]

    # Validação por pixel dentro do treino (ver docstring): estratificada, para não
    # deixar uma classe rara sem amostra de validação.
    rng = np.random.default_rng(seed + 1)
    escolhidos = []
    for klass in np.unique(labels[resto_idx]):
        pool = resto_idx[labels[resto_idx] == klass]
        quantos = min(len(pool) - 1, max(1, int(round(len(pool) * valid_ratio))))
        if quantos > 0:
            escolhidos.append(rng.choice(pool, size=quantos, replace=False))
    val_idx = np.concatenate(escolhidos) if escolhidos else resto_idx[:0]
    treino_idx = np.setdiff1d(resto_idx, val_idx)

    faltando = sorted(set(np.unique(labels).tolist())
                      - set(np.unique(labels[treino_idx]).tolist()))
    relatorio = {
        "mode": "region_folds", "n_folds": int(n_folds), "fold": int(fold),
        "regioes_total": int(len(regioes)),
        "regioes_treino": int(len(np.unique(region_ids[treino_idx]))),
        "regioes_teste": int(len(np.unique(region_ids[teste_idx]))),
        "train_fraction_px": float(len(treino_idx) / len(labels)),
        "test_fraction_px": float(len(teste_idx) / len(labels)),
        "classes_missing_from_train": faltando,
        "classes_in_test": int(len(np.unique(labels[teste_idx]))),
        "val_disjunta_por_regiao": False,
    }
    splits = Splits(train=np.sort(treino_idx), val=np.sort(val_idx), test=np.sort(teste_idx))
    return splits, relatorio


def split_indices_blocks(labels, coords, scene_shape, test_ratio, valid_ratio,
                         block: int = 0, seed: int = 42) -> tuple[Splits, dict]:
    """Split por blocos retangulares da cena, independente do segmentador.

    Vantagem sobre `split_indices_regions`: a mesma partição serve para comparar
    segmentadores diferentes. Desvantagem: a disjunção é **aproximada** — regiões
    que cruzam a fronteira do bloco continuam compartilhadas. Isso importa muito
    em Indian Pines, onde os superpixels do SEEDS têm ~16x16 px e um bloco pequeno
    não os contém. Confira sempre `segment_leakage` em vez de supor.
    """
    coords = np.asarray(coords)
    block = block or max(4, min(scene_shape) // 4)
    grid_cols = int(np.ceil(scene_shape[1] / block))
    block_id = (coords[:, 0] // block) * grid_cols + (coords[:, 1] // block)
    splits, report = _split_by_groups(labels, block_id, test_ratio, valid_ratio, seed)
    return splits, {"mode": "block", "block_size": int(block), **report}


def subsample_train(splits: Splits, labels, fraction: float, seed: int = 42) -> tuple[Splits, dict]:
    """Reduz o conjunto de treino para uma fração, preservando a proporção das classes.

    Amostragem estratificada por classe, e não sorteio uniforme: com 10% uniforme,
    uma classe de 20 pixels teria em média 2 amostras e frequentemente zero, o que
    tornaria a métrica dela indefinida. Estratificar garante ao menos uma amostra
    por classe presente no treino.

    Validação e teste não são tocados. Isso é deliberado: comparar `train_fraction`
    diferentes só faz sentido se o conjunto de avaliação for idêntico.
    """
    labels = np.asarray(labels)
    if fraction >= 1:
        return splits, {"train_fraction": 1.0, "train_size": int(len(splits.train))}
    rng = np.random.default_rng(seed)
    chosen = []
    for klass in np.unique(labels[splits.train]):
        pool = splits.train[labels[splits.train] == klass]
        keep = max(1, int(round(len(pool) * fraction)))
        chosen.append(rng.choice(pool, size=keep, replace=False))
    train = np.sort(np.concatenate(chosen))
    report = {
        "train_fraction": float(fraction),
        "train_size": int(len(train)),
        "train_size_full": int(len(splits.train)),
        "classes_in_train": int(len(np.unique(labels[train]))),
        "min_samples_per_class": int(np.bincount(labels[train]).max(initial=0) and
                                     np.min(np.bincount(labels[train])[np.bincount(labels[train]) > 0])),
    }
    return Splits(train=train, val=splits.val, test=splits.test), report


def segment_leakage(data: PreparedData) -> dict:
    """Quanto do teste compartilha superpixel com o treino — o vazamento residual.

    No split por pixel esse número fica perto de 1,0 por construção. No split por
    blocos ele mede só as regiões que cruzam a fronteira entre blocos, e é o número
    que torna honesta qualquer afirmação sobre o ganho da fusão.
    """
    if data.segments is None:
        return {"applicable": False}
    train_ids = np.unique(data.segments[data.splits.train])
    test_ids = data.segments[data.splits.test]
    unique_test = np.unique(test_ids)
    return {
        "applicable": True,
        "test_samples": int(len(test_ids)),
        "test_samples_sharing_region_with_train": float(np.isin(test_ids, train_ids).mean()),
        "regions_in_test": int(len(unique_test)),
        "regions_also_in_train": int(np.isin(unique_test, train_ids).sum()),
    }


def validate_prepared_data(data: PreparedData) -> dict[str, int | bool]:
    """Aplica o contrato de dados antes de alocar GPU ou iniciar o treino.

    A validação concentra falhas comuns de experimentos: arrays desalinhados,
    NaN/infinito após PCA, classes fora de ``0..C-1`` e splits que se sobrepõem.
    Retorna um resumo serializável para o tracker — incluindo a cobertura, que é
    menor que 1 quando ``train_fraction < 1`` — e levanta ``ValueError`` com uma
    causa objetiva quando o contrato não é atendido.
    """
    n_samples = len(data.labels)
    if data.spectra.ndim != 2 or len(data.spectra) != n_samples:
        raise ValueError("spectra deve ser uma matriz com uma linha por rótulo.")
    if data.labels.ndim != 1 or n_samples == 0:
        raise ValueError("labels deve ser um vetor não vazio.")
    if not np.isfinite(data.spectra).all():
        raise ValueError("spectra contém NaN ou infinito.")
    expected_classes = np.arange(data.n_classes)
    if not np.array_equal(np.unique(data.labels), expected_classes):
        raise ValueError("labels deve conter classes contíguas de 0 até C-1.")
    if data.segments is not None and (len(data.segments) != n_samples or not np.isfinite(data.segments).all()):
        raise ValueError("segments deve ser finito e ter uma entrada por rótulo.")
    all_indices = np.concatenate([data.splits.train, data.splits.val, data.splits.test])
    # O invariante que importa é a DISJUNÇÃO — nenhuma amostra em duas partições.
    # A cobertura total não é exigida: com `train_fraction < 1` o treino é uma
    # amostra estratificada, e sobram pixels rotulados fora das três partições
    # de propósito. Exigir cobertura aqui recusaria um experimento válido.
    if len(np.unique(all_indices)) != len(all_indices):
        raise ValueError("splits se sobrepõem: uma amostra aparece em mais de uma partição.")
    if all_indices.min(initial=0) < 0 or all_indices.max(initial=-1) >= n_samples:
        raise ValueError("splits contém índice fora dos dados preparados.")
    return {
        "valid": True,
        "samples": n_samples,
        "samples_used": int(len(all_indices)),
        "coverage": float(len(all_indices) / n_samples),
        "classes": data.n_classes,
        "has_segments": data.segments is not None,
    }


def _tensors(data: PreparedData, index: np.ndarray, model_type: str = "cnn1d") -> tuple[torch.Tensor, ...]:
    """Fatia uma partição e converte para os tensores que o modelo consome.

    O espectro vira ``(N, 1, Kb)``. A segmentação é um escalar por amostra,
    concatenado depois do achatamento das convoluções.
    """
    features = torch.from_numpy(np.ascontiguousarray(data.spectra[index])).float().unsqueeze(1)
    labels = torch.from_numpy(np.ascontiguousarray(data.labels[index]))
    if data.segments is None:
        return features, labels
    segments = torch.from_numpy(np.ascontiguousarray(data.segments[index])).float().reshape(-1, 1)
    return features, segments, labels


def build_loaders(data: PreparedData, config, device) -> dict[str, DataLoader]:
    """DataLoaders de treino, validação e teste.

    O loader de treino recebe um `generator` semeado explicitamente: a ordem dos
    batches passa a depender só de `config.seed`, e não do estado global do RNG.
    Criar um iterador sobre esse loader (para inspecionar um batch, por exemplo)
    adianta o gerador — por isso `fit` repõe a semente antes da primeira época.

    Em CUDA, ``pin_memory=True`` torna a transferência CPU→GPU não bloqueante em
    ``unpack_batch``. ``prefetch_factor`` e ``persistent_workers`` só são
    passados quando há workers, pois o PyTorch os rejeita/ignora com zero workers.
    Validação e teste podem usar batch maior porque não armazenam gradientes.
    """
    pin = device.type == "cuda"
    datasets = {
        name: TensorDataset(*_tensors(data, index, config.model_type))
        for name, index in (
            ("train", data.splits.train),
            ("val", data.splits.val),
            ("test", data.splits.test),
        )
    }
    worker_options = {}
    if config.num_workers:
        worker_options = {
            "prefetch_factor": config.prefetch_factor,
            "persistent_workers": config.persistent_workers,
        }

    loaders = {
        "train": DataLoader(
            datasets["train"], batch_size=config.batch_size, shuffle=True,
            generator=torch.Generator().manual_seed(config.seed),
            pin_memory=pin, num_workers=config.num_workers, drop_last=False,
            **worker_options,
        )
    }
    for name in ("val", "test"):
        loaders[name] = DataLoader(
            datasets[name], batch_size=config.resolved_eval_batch_size, shuffle=False,
            pin_memory=pin, num_workers=config.num_workers,
            **worker_options,
        )
    return loaders

# ============================================================================
# PARALLEL
# ============================================================================
"""Paralelização inter-etapas da preparação dos dados.

Depois do PCA, os dois ramos do pipeline são independentes até o ponto de fusão:

- **ramo espectral**: PCA para `config.pca_components` bandas sobre os pixels válidos;
- **ramo espacial**: PCA de 3 componentes sobre a cena inteira, composição RGB e
  segmentação SEEDS, SLIC ou LSC.

Nenhum precisa do resultado do outro — só a fusão (montar a segunda entrada da rede)
precisa dos dois. Executá-los concorrentemente troca `T_espectral + T_espacial` por
`max(T_espectral, T_espacial) + overhead`.

Backends disponíveis:

``sequential``
    Um ramo depois do outro. É a linha de base contra a qual o ganho é medido.
``thread``
    `ThreadPoolExecutor`. Funciona apesar do GIL porque o trabalho pesado está em
    código nativo que o libera: a SVD do PCA (BLAS) e o `iterate` do SEEDS (OpenCV).
    Não copia nada — os dois ramos compartilham os arrays de entrada.
``process``
    `ProcessPoolExecutor`, que é o que o projeto do TCC descreve. Imune ao GIL, mas
    paga serialização: a tabela de pixels é enviada ao worker e o mapa de rótulos
    volta. Em cenas grandes esse custo pode consumir o ganho — por isso o
    `scripts/benchmark_parallel.py` mede os três.

O resultado é **idêntico** nos três backends: o paralelismo só muda quando cada
etapa roda, não o que ela calcula.
"""


import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np


__all__ = ["BACKENDS", "StageTimings", "prepare_data_parallel", "spectral_branch", "spatial_branch"]

BACKENDS = ("sequential", "thread", "process")


@dataclass
class StageTimings:
    """Tempo de parede de cada etapa e do bloco paralelo como um todo.

    `stages` guarda o tempo *interno* de cada ramo (o que ele levaria sozinho);
    `wall` é o tempo real do bloco. Num backend paralelo bem-sucedido,
    `wall < soma dos ramos`.
    """

    backend: str
    stages: dict[str, float] = field(default_factory=dict)
    wall: float = 0.0

    @property
    def branch_sum(self) -> float:
        """Soma dos ramos — o que custaria em execução estritamente serial."""
        return self.stages.get("espectral", 0.0) + self.stages.get("espacial", 0.0)

    @property
    def overhead(self) -> float:
        """Sobrecarga de gerenciamento: threads/processos, serialização, sincronização."""
        return self.wall - max(self.stages.get("espectral", 0.0), self.stages.get("espacial", 0.0))

    def speedup_over(self, baseline: "StageTimings") -> float:
        return baseline.wall / self.wall if self.wall else float("nan")

    def summary(self) -> str:
        linhas = [f"backend: {self.backend} | bloco paralelo: {self.wall:.3f}s"]
        for name, value in self.stages.items():
            linhas.append(f"  {name:12s} {value:7.3f}s")
        linhas.append(f"  {'soma ramos':12s} {self.branch_sum:7.3f}s | overhead: {self.overhead:+.3f}s")
        return "\n".join(linhas)


def spectral_branch(spectra: np.ndarray, n_components: int, seed: int) -> np.ndarray:
    """Ramo espectral: reduz as bandas de cada pixel válido por PCA."""
    reduced, _ = apply_pca(spectra, n_components=n_components, random_state=seed)
    return reduced


def spatial_branch(
    pixels: np.ndarray,
    raw_labels: np.ndarray,
    scene_shape: tuple[int, int],
    params: SeedsParams | SlicParams | LscParams,
    seed: int,
    segmenter: str = "seeds",
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Ramo espacial: PCA de 3 componentes -> RGB -> superpixels -> id válido.

    A segmentação roda sobre a cena **inteira**, background incluído, porque os
    superpixels precisam da imagem completa; o filtro de background vem depois.

    Função de módulo (e não closure) porque precisa ser serializável para o backend
    de processos.
    """
    components, _ = apply_pca(pixels, n_components=3, random_state=seed)
    rgb = rgb_composite(components, scene_shape)
    segmentation = superpixel_labels_for(rgb, segmenter, params)

    ids_valid, _ = remove_background(segmentation.labels.reshape(-1, 1), raw_labels)
    return ids_valid.reshape(-1), components, rgb, segmentation


def superpixel_labels_for(rgb, segmenter, params):
    """Indireção fina só para manter o import do OpenCV dentro do worker."""

    return segment_superpixels(rgb, segmenter, params)


def _segmenter_params(config: ExperimentConfig):
    """Seleciona os parâmetros da cena sem acoplar o pipeline a um algoritmo.

    `segmenter_overrides` é preenchido pelo HPO; aplicá-lo aqui garante que o
    parâmetro testado seja exatamente o que vai para o `config.json` do run.
    """
    tables = {
        "seeds": SEEDS_PARAMS,
        "slic": SLIC_PARAMS,
        "lsc": LSC_PARAMS,
        "watershed": WATERSHED_PARAMS,
    }
    return replace(tables[config.segmenter][config.dataset], **config.segmenter_overrides)


def _timed(fn, *args):
    """Executa `fn` medindo seu tempo interno. Retorna `(resultado, segundos)`."""
    started = time.perf_counter()
    return fn(*args), time.perf_counter() - started


def prepare_data_parallel(
    config: ExperimentConfig,
    data_dir: Path | None = None,
    scene: Scene | None = None,
    backend: str = "thread",
) -> tuple[PreparedData, SegmentationArtifacts | None, StageTimings]:
    """`prepare_data` com os dois ramos executados concorrentemente.

    Devolve `(dados, artefatos_da_segmentação, tempos)`. Com `two_input=False` e
    `split_mode` diferente de "region", o ramo espacial é dispensado por completo.
    """
    if backend not in BACKENDS:
        raise ValueError(f"backend inválido: {backend!r}. Use um de {BACKENDS}.")

    scene = scene or load_scene(config.dataset, data_dir)
    pixels, raw_labels = to_pixel_table(scene)
    spectra, labels = remove_background(pixels, raw_labels)
    # `to_pixel_table` achata em row-major, então a posição de cada amostra válida
    # se recupera exatamente do índice achatado. O split por blocos precisa disso.
    valid = np.flatnonzero(raw_labels > 0)
    coords = np.stack(np.divmod(valid, scene.shape[1]), axis=1)

    timings = StageTimings(backend=backend)
    spatial_args = (
        pixels, raw_labels, scene.shape, _segmenter_params(config), config.seed, config.segmenter,
    )
    spectral_args = (spectra, config.pca_components, config.seed)

    started = time.perf_counter()

    # O split por região precisa da segmentação mesmo numa ablação sem fusão —
    # é o que mantém os dois braços (com e sem fusão) sobre a MESMA partição.
    needs_spatial_branch = config.two_input or config.split_mode == "region"

    if not needs_spatial_branch:
        reduced, timings.stages["espectral"] = _timed(spectral_branch, *spectral_args)
        segments, artifacts = None, None

    elif backend == "sequential":
        reduced, timings.stages["espectral"] = _timed(spectral_branch, *spectral_args)
        spatial, timings.stages["espacial"] = _timed(spatial_branch, *spatial_args)
        segments, artifacts = _unpack_spatial(spatial)

    elif backend == "thread":
        # Os dois ramos compartilham os arrays de entrada: nada é copiado.
        with ThreadPoolExecutor(max_workers=2) as pool:
            spatial_future = pool.submit(_timed, spatial_branch, *spatial_args)
            spectral_future = pool.submit(_timed, spectral_branch, *spectral_args)
            reduced, timings.stages["espectral"] = spectral_future.result()
            spatial, timings.stages["espacial"] = spatial_future.result()
        segments, artifacts = _unpack_spatial(spatial)

    else:  # process
        # O ramo espacial vai para outro processo; o espectral roda aqui mesmo,
        # o que evita um segundo round-trip de serialização.
        with ProcessPoolExecutor(max_workers=1) as pool:
            spatial_future = pool.submit(_timed, spatial_branch, *spatial_args)
            reduced, timings.stages["espectral"] = _timed(spectral_branch, *spectral_args)
            spatial, timings.stages["espacial"] = spatial_future.result()
        segments, artifacts = _unpack_spatial(spatial)

    timings.wall = time.perf_counter() - started

    # Com `split_mode="region"` o ramo espacial roda mesmo sem fusão. Nesse caso os
    # artefatos visuais existem, mas o id não entra no TensorDataset nem é
    # registrado como segunda entrada do modelo.
    # `segments` é a segunda entrada do modelo; `region_ids` é o agrupamento usado
    # pelo split. Separá-los permite um braço sem fusão sobre split por região.
    region_ids = segments
    if not config.two_input:
        segments = None

    if config.split_mode == "region":
        if region_ids is None:
            raise ValueError("split_mode='region' exige segmentação disponível.")
        if config.split_folds:
            splits, split_report = split_indices_region_folds(
                labels, region_ids, config.split_folds, config.split_fold,
                # Mesma proporção de validação do protocolo por pixel: `test_ratio`
                # separa 15% e `valid_ratio` fica com metade disso.
                config.test_ratio * config.valid_ratio,
                config.seed,
            )
        else:
            splits, split_report = split_indices_regions(
                labels, region_ids, config.test_ratio, config.valid_ratio, config.seed,
            )
    elif config.split_mode == "block":
        splits, split_report = split_indices_blocks(
            labels, coords, scene.shape, config.test_ratio, config.valid_ratio,
            config.split_block, config.seed,
        )
    else:
        splits, split_report = split_indices(
            labels, config.test_ratio, config.valid_ratio, config.seed
        ), {"mode": "pixel"}
    splits, subsample_report = subsample_train(splits, labels, config.train_fraction, config.seed)
    split_report.update(subsample_report)
    if split_report.get("classes_missing_from_train"):
        # Um grupo inteiro pode conter toda uma classe rara. Avisar é melhor que
        # falhar: o run continua, mas a AA daquela classe será zero por construção.
        print(
            f"Aviso: split por {split_report['mode']} deixou as classes "
            f"{split_report['classes_missing_from_train']} fora do treino."
        )

    data = PreparedData(
        spectra=reduced,
        labels=labels.astype(np.int64),
        segments=segments,
        splits=splits,
        coords=coords,
        scene_shape=scene.shape,
    )
    return data, artifacts, timings


def _unpack_spatial(spatial) -> tuple[np.ndarray, SegmentationArtifacts]:
    ids, components, rgb, segmentation = spatial
    return ids, SegmentationArtifacts(components=components, rgb=rgb, result=segmentation)


### 3.2 Segmentadores disponíveis

Quatro algoritmos: **SEEDS**, **SLIC**, **LSC** (superpixels do `cv.ximgproc`) e **Watershed** (bacias de gradiente do OpenCV base). Todos recebem a mesma composição PCA/RGB e devolvem `SegmentationResult` com labels e contornos no mesmo formato, então trocar de segmentador não altera DataLoader, CNN nem métricas.

Os parâmetros por cena ficam em `SEEDS_PARAMS`, `SLIC_PARAMS`, `LSC_PARAMS` e `WATERSHED_PARAMS`; `config.segmenter_overrides` sobrescreve campos individuais e é o que o HPO varia.

### 3.3 Explorar a cena e comparar segmentadores

Antes de treinar, confirme visualmente a distribuição de classes, assinaturas espectrais, variância do PCA e regiões produzidas por cada superpixel. A pureza aparece em um gráfico de homogeneidade espacial — não usa nem substitui as métricas do classificador.

In [ ]:
# ============================================================================
# DIAGNÓSTICOS VISUAIS — cena, PCA, segmentação, métricas e desempenho
# ============================================================================
"""Visualizações explicáveis para classificação hiperespectral.

As figuras são diagnósticos, não apenas ilustrações: elas ajudam a verificar se a
cena foi lida corretamente, se o PCA preserva estruturas espaciais, se os
superpixels são coerentes e se a qualidade do classificador é uniforme entre as
classes. Todas recebem arrays ou resultados já produzidos pelo pipeline e não
alteram seus dados.
"""

from __future__ import annotations

from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    jaccard_score,
    precision_recall_fscore_support,
)


def _robust_rgb(cube: np.ndarray) -> np.ndarray:
    """Cria uma composição RGB de falsa cor com três bandas bem espaçadas.

    O percentil 2--98 reduz a influência de pixels extremos; ele melhora a leitura
    visual, mas nunca é usado como entrada do modelo ou da segmentação.
    """
    bands = np.linspace(0, cube.shape[2] - 1, 3, dtype=int)
    image = cube[:, :, bands].astype(np.float64)
    low, high = np.percentile(image, (2, 98), axis=(0, 1), keepdims=True)
    return np.clip((image - low) / np.maximum(high - low, 1e-12), 0, 1)


def _label_cmap(n_classes: int):
    """Mapa discreto de cores, deixando o fundo sem rótulo transparente."""
    return plt.colormaps.get_cmap("tab20").resampled(max(n_classes, 1))


def plot_scene_overview(scene: Scene):
    """Mostra a cena, ground truth, desbalanceamento e assinaturas espectrais.

    A distribuição de classes explica por que OA sozinha pode enganar: uma classe
    muito frequente pode dominar a média. As curvas espectrais mostram o quanto as
    classes são separáveis antes da CNN.
    """
    labels = scene.labels
    classes, counts = np.unique(labels[labels > 0], return_counts=True)
    pixels, flat_labels = to_pixel_table(scene)
    cmap = _label_cmap(scene.n_classes)

    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    axes[0, 0].imshow(_robust_rgb(scene.image))
    axes[0, 0].set_title("Composição de falsa cor (bandas espaçadas)")
    axes[0, 0].axis("off")

    ground_truth = np.ma.masked_where(labels == 0, labels - 1)
    im = axes[0, 1].imshow(ground_truth, cmap=cmap, vmin=0, vmax=max(scene.n_classes - 1, 0))
    axes[0, 1].set_title("Ground truth — apenas pixels rotulados")
    axes[0, 1].axis("off")
    fig.colorbar(im, ax=axes[0, 1], shrink=.78, label="Classe (índice)")

    axes[1, 0].bar(classes, counts, color=cmap(np.arange(len(classes))))
    axes[1, 0].set(title="Amostras rotuladas por classe", xlabel="Classe", ylabel="Pixels")
    axes[1, 0].grid(axis="y", alpha=.25)

    for class_id in classes:
        spectra = pixels[flat_labels == class_id]
        axes[1, 1].plot(np.mean(spectra, axis=0), lw=1.2, label=str(class_id))
    axes[1, 1].set(title="Assinatura espectral média por classe", xlabel="Banda", ylabel="Intensidade média")
    axes[1, 1].legend(title="Classe", ncol=2, fontsize=8)
    axes[1, 1].grid(alpha=.25)
    return fig


def pca_diagnostics(scene: Scene, n_components: int = 12):
    """Calcula PCA sem whitening para interpretar variância e mapas de características."""
    pixels, _ = to_pixel_table(scene)
    n_components = min(n_components, pixels.shape[1], pixels.shape[0])
    model = PCA(n_components=n_components, random_state=42)
    components = model.fit_transform(pixels)
    return components, model


def plot_pca_diagnostics(scene: Scene, n_components: int = 12, maps: int = 6):
    """Exibe variância acumulada e mapas das primeiras componentes principais.

    Mapas de PCA são os mapas de características que alimentam o ramo espacial:
    regiões com padrões semelhantes devem ter valores semelhantes nas componentes.
    """
    components, model = pca_diagnostics(scene, n_components)
    maps = min(maps, components.shape[1])
    ncols = 3
    nrows = int(np.ceil(maps / ncols))
    fig = plt.figure(figsize=(15, 4 + 3.5 * nrows), constrained_layout=True)
    grid = fig.add_gridspec(nrows + 1, ncols)
    axis = fig.add_subplot(grid[0, :])
    explained = model.explained_variance_ratio_
    axis.bar(np.arange(1, len(explained) + 1), explained * 100, alpha=.55, label="por componente")
    axis.plot(np.arange(1, len(explained) + 1), np.cumsum(explained) * 100, marker="o", color="crimson", label="acumulada")
    axis.set(title="PCA: variância explicada", xlabel="Componente", ylabel="Variância (%)")
    axis.legend()
    axis.grid(axis="y", alpha=.25)
    for index in range(maps):
        ax = fig.add_subplot(grid[1 + index // ncols, index % ncols])
        image = components[:, index].reshape(scene.shape)
        im = ax.imshow(image, cmap="viridis")
        ax.set_title(f"Mapa de característica: PC {index + 1}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, shrink=.75)
    return fig, components, model


def segmentation_quality(labels: np.ndarray, ground_truth: np.ndarray) -> dict:
    """Mede tamanho, bordas e pureza sem confundir superpixel com classe.

    *Pureza* é a fração da classe dominante dentro de cada superpixel, ponderada
    pelos pixels rotulados. É diagnóstico espacial; não é uma métrica de acurácia
    do classificador e não deve ser usada para escolher modelo isoladamente.
    """
    areas = np.bincount(labels.ravel())
    valid = ground_truth.ravel() > 0
    segment_ids = labels.ravel()[valid]
    true_labels = ground_truth.ravel()[valid]
    weighted_hits = 0
    for segment_id in np.unique(segment_ids):
        segment_truth = true_labels[segment_ids == segment_id]
        weighted_hits += np.bincount(segment_truth).max()
    horizontal = np.count_nonzero(labels[:, 1:] != labels[:, :-1])
    vertical = np.count_nonzero(labels[1:, :] != labels[:-1, :])
    return {
        "superpixels": int(len(areas)),
        "área média": float(areas.mean()),
        "área mediana": float(np.median(areas)),
        "densidade de borda": float((horizontal + vertical) / labels.size),
        "pureza rotulada": float(weighted_hits / max(len(true_labels), 1)),
    }


def plot_segmentation_comparison(scene: Scene, algorithms=("seeds", "slic", "lsc", "watershed")):
    """Compara contornos, regiões e estatísticas dos segmentadores disponíveis.

    Todos recebem a mesma composição RGB criada pelas três primeiras PCs; assim a
    comparação isola o algoritmo e seus hiperparâmetros configurados para a cena.
    """
    pixels, _ = to_pixel_table(scene)
    components, _ = apply_pca(pixels, n_components=3, random_state=42)
    rgb = rgb_composite(components, scene.shape)
    fig, axes = plt.subplots(2, len(algorithms), figsize=(6 * len(algorithms), 9), constrained_layout=True)
    axes = np.asarray(axes).reshape(2, len(algorithms))
    rows = []
    for index, algorithm in enumerate(algorithms):
        local = ExperimentConfig(dataset=scene.code, segmenter=algorithm)
        result = segment_superpixels(rgb, algorithm, _segmenter_params(local))
        outlined = rgb.copy()
        outlined[result.contour_mask > 0] = (255, 255, 255)
        axes[0, index].imshow(outlined)
        axes[0, index].set_title(f"{algorithm.upper()}: contornos")
        axes[1, index].imshow(colorize_segments(result.labels, seed=42))
        axes[1, index].set_title(f"{algorithm.upper()}: regiões")
        for axis in axes[:, index]:
            axis.axis("off")
        rows.append({"segmentador": algorithm.upper(), **segmentation_quality(result.labels, scene.labels)})
    return fig, pd.DataFrame(rows).set_index("segmentador")


def classification_metrics_table(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[pd.DataFrame, dict]:
    """Calcula métricas por classe e os resumos macro para classes desbalanceadas.

    OA, AA (recall macro) e Kappa são o trio consagrado em HSI. Precisão, F1 e IoU
    expõem erros assimétricos por classe; elas complementam, mas não substituem, o
    trio para comparação com trabalhos da literatura.
    """
    labels = np.arange(max(int(y_true.max()), int(y_pred.max())) + 1)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0,
    )
    iou = jaccard_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    per_class = pd.DataFrame({
        "classe": labels + 1,
        "suporte": support,
        "precisão": precision,
        "revocação / AA por classe": recall,
        "F1": f1,
        "IoU": iou,
    })
    summary = {
        "AA / balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "F1 macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "IoU macro": jaccard_score(y_true, y_pred, average="macro", zero_division=0),
    }
    return per_class, summary


def plot_evaluation_dashboard(result: ExperimentResult, scene: Scene):
    """Exibe convergência, métricas, matriz de confusão e mapa de erros do teste."""
    metrics = result.metrics
    history = result.training.history.to_dict()
    y_true = result.data.labels[result.data.splits.test]
    y_pred = result.predicted
    per_class, extra = classification_metrics_table(y_true, y_pred)
    fig, axes = plt.subplots(2, 3, figsize=(20, 11), constrained_layout=True)

    epochs = result.training.history.epochs
    axes[0, 0].plot(epochs, history["loss"], label="treino")
    axes[0, 0].plot(epochs, history["val_loss"], label="validação")
    axes[0, 0].set(title="Convergência: loss", xlabel="Época", ylabel="Cross-entropy")
    axes[0, 0].legend(); axes[0, 0].grid(alpha=.25)
    axes[0, 1].plot(epochs, history["accuracy"], label="treino")
    axes[0, 1].plot(epochs, history["val_accuracy"], label="validação")
    training = result.training
    axes[0, 1].axvline(training.best_epoch, color="#E45756", ls="--", lw=1.5,
                        label=f"melhor época ({training.best_epoch})")
    if training.stopped_early:
        axes[0, 1].axvline(len(training.history), color="#4C78A8", ls=":", lw=2,
                            label=f"parada antecipada ({len(training.history)})")
    target = getattr(training, "early_stopping_target", None)
    if target is not None:
        axes[0, 1].axhline(target, color="#54A24B", ls="-.", lw=1.6,
                            label=f"meta de validação ({target:.3f})")
    axes[0, 1].set(title=(f"Convergência: acurácia | {training.early_stopping_monitor}, "
                           f"patience={training.patience}, Δ≥{training.early_stopping_min_delta:g}"),
                   xlabel="Época", ylabel="Acurácia")
    axes[0, 1].legend(); axes[0, 1].grid(alpha=.25)

    values = [metrics.overall_accuracy, metrics.average_accuracy, metrics.kappa, extra["F1 macro"], extra["IoU macro"]]
    names = ["OA", "AA", "Kappa", "F1 macro", "IoU macro"]
    axes[0, 2].bar(names, values, color=["#4C78A8", "#72B7B2", "#F58518", "#54A24B", "#E45756"])
    axes[0, 2].set(title="Métricas globais", ylabel="Pontuação", ylim=(0, 1))
    axes[0, 2].grid(axis="y", alpha=.25)

    normalized = metrics.confusion / np.maximum(metrics.confusion.sum(axis=1, keepdims=True), 1)
    sns.heatmap(normalized, cmap="Blues", vmin=0, vmax=1, ax=axes[1, 0], cbar_kws={"label": "fração"})
    axes[1, 0].set(title="Matriz de confusão normalizada", xlabel="Predição", ylabel="Classe real")

    axes[1, 1].bar(per_class["classe"], per_class["revocação / AA por classe"], label="recall / AA")
    axes[1, 1].bar(per_class["classe"], per_class["F1"], alpha=.55, label="F1")
    axes[1, 1].set(title="Qualidade por classe", xlabel="Classe", ylabel="Pontuação", ylim=(0, 1))
    axes[1, 1].legend(); axes[1, 1].grid(axis="y", alpha=.25)

    positions = np.flatnonzero(scene.labels.ravel() > 0)[result.data.splits.test]
    errors = np.full(scene.labels.size, np.nan)
    errors[positions] = (y_pred != y_true).astype(float)
    axes[1, 2].imshow(errors.reshape(scene.shape), cmap="Reds", vmin=0, vmax=1)
    axes[1, 2].set_title("Erros no conjunto de teste (vermelho)")
    axes[1, 2].axis("off")
    return fig, per_class, extra


def profile_inference(result: ExperimentResult) -> pd.DataFrame:
    """Mede inferência real e memória de GPU sem misturar esse tempo ao treino."""
    loaders = build_loaders(result.data, result.config, result.device)
    if result.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(result.device)
        torch.cuda.synchronize(result.device)
    started = perf_counter()
    probabilities = predict_proba(result.model, loaders["test"], result.device)
    if result.device.type == "cuda":
        torch.cuda.synchronize(result.device)
        peak_memory = torch.cuda.max_memory_allocated(result.device) / 2**20
    else:
        peak_memory = np.nan
    elapsed = perf_counter() - started
    return pd.DataFrame([{
        "amostras": len(probabilities),
        "inferência (s)": elapsed,
        "ms / pixel": 1000 * elapsed / max(len(probabilities), 1),
        "pico GPU (MiB)": peak_memory,
        "parâmetros": result.model.n_parameters,
    }])


def plot_parallel_profile(timings: StageTimings):
    """Visualiza os ramos independentes e o ganho potencial do paralelismo pré-fusão."""
    durations = dict(timings.stages)
    fig, axis = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
    colors = {"espectral": "#4C78A8", "espacial": "#F58518"}
    for row, (name, duration) in enumerate(durations.items()):
        axis.barh(row, duration, color=colors.get(name, "#72B7B2"), label=name)
        axis.text(duration, row, f" {duration:.3f}s", va="center")
    sequential = sum(durations.values())
    axis.axvline(timings.wall, color="black", ls="--", label=f"wall: {timings.wall:.3f}s")
    axis.axvline(sequential, color="gray", ls=":", label=f"sequencial estimado: {sequential:.3f}s")
    axis.set(yticks=range(len(durations)), yticklabels=list(durations), xlabel="Tempo (s)", title=f"Pré-processamento — backend: {timings.backend}")
    axis.legend(loc="lower right")
    return fig


def plot_runs_dashboard(runs):
    """Compara qualidade e custo dos runs rastreados; o ideal fica no alto à esquerda."""
    rows = [run.row() for run in runs if run.metrics]
    if not rows:
        print("Não há runs concluídos com métricas para visualizar.")
        return None
    frame = pd.DataFrame(rows).sort_values("run_id")
    labels = [run_id.split("_")[-1] for run_id in frame["run_id"]]
    fig, axes = plt.subplots(1, 2, figsize=(17, 5), constrained_layout=True)
    axes[0].plot(labels, frame["oa"], marker="o", label="OA")
    axes[0].plot(labels, frame["aa"], marker="o", label="AA")
    axes[0].plot(labels, frame["kappa"], marker="o", label="Kappa")
    axes[0].set(title="Métricas por run", xlabel="Sufixo do run", ylabel="Pontuação", ylim=(0, 1))
    axes[0].tick_params(axis="x", rotation=45); axes[0].legend(); axes[0].grid(alpha=.25)
    scatter = axes[1].scatter(frame["train_seconds"], frame["oa"], s=90, c=frame["aa"], cmap="viridis", edgecolor="black")
    for _, row in frame.iterrows():
        axes[1].annotate(row["model_type"], (row["train_seconds"], row["oa"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    axes[1].set(title="Trade-off: qualidade × tempo de treino", xlabel="Tempo de treino (s)", ylabel="OA")
    axes[1].grid(alpha=.25)
    fig.colorbar(scatter, ax=axes[1], label="AA")
    return fig


def plot_per_class_metrics(per_class):
    """Todas as métricas de classificação por classe em um único gráfico."""
    figure, axis = plt.subplots(figsize=(14, 5))
    classes = per_class["classe"].to_numpy(); width=.25
    axis.bar(classes-width, per_class["precisão"], width, label="precisão")
    axis.bar(classes, per_class["revocação / AA por classe"], width, label="recall / AA")
    axis.bar(classes+width, per_class["F1"], width, label="F1")
    axis.plot(classes, per_class["IoU"], "o-", color="#E45756", label="IoU")
    axis.set(title="Classificação por classe", xlabel="Classe", ylabel="pontuação", ylim=(0,1))
    axis.legend(ncol=4); axis.grid(axis="y", alpha=.25)
    return figure


def plot_inference_profile_chart(profile):
    """Tempo, latência e pico de memória em gráficos."""
    row=profile.iloc[0]; figure, axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
    for axis,label,value,color,ylabel,title in zip(axes,["inferência","pixel","GPU"],[row["inferência (s)"],row["ms / pixel"],row["pico GPU (MiB)"]],["#4C78A8","#F58518","#54A24B"],["segundos","ms","MiB"],["Tempo de inferência","Latência por pixel","Pico de memória"]):
        axis.bar([label],[value],color=color); axis.set(ylabel=ylabel,title=title); axis.grid(axis="y",alpha=.25)
    return figure


def plot_segment_metrics_visual(segments):
    """Acurácia, confiança, entropia e F1 dos superpixels."""
    frame=segments.copy().sort_values("acurácia",ascending=False)
    figure,axes=plt.subplots(1,2,figsize=(16,5),constrained_layout=True)
    scatter=axes[0].scatter(frame["confiança"],frame["acurácia"],c=frame["entropia"],s=55,cmap="magma_r",edgecolor="black",linewidth=.25)
    axes[0].set(title="Segmentos: confiança × acurácia",xlabel="confiança",ylabel="acurácia",xlim=(0,1),ylim=(0,1)); axes[0].grid(alpha=.25); figure.colorbar(scatter,ax=axes[0],label="entropia")
    order=np.arange(1,len(frame)+1); axes[1].plot(order,frame["F1 da classe"],label="F1 da classe"); axes[1].plot(order,frame["entropia"],label="entropia")
    axes[1].set(title="Segmentos ordenados por acurácia",xlabel="posição do segmento",ylabel="pontuação",ylim=(0,1)); axes[1].legend(); axes[1].grid(alpha=.25)
    return figure


def plot_cnn_summary_visual(summary):
    """Métricas, custo e memória do dashboard individual."""
    figure,axes=plt.subplots(1,3,figsize=(17,4.5),constrained_layout=True)
    metrics=["OA","AA","Kappa","precisão macro","F1 macro","IoU macro"]; axes[0].bar(metrics,[summary.get(x,np.nan) for x in metrics],color=plt.cm.tab10.colors[:len(metrics)]); axes[0].set(title="Qualidade global",ylabel="pontuação",ylim=(0,1)); axes[0].tick_params(axis="x",rotation=35)
    costs=["pré-processamento (s)","treino (s)","inferência (s)"]; axes[1].bar(["pré-processamento","treino","inferência"],[summary.get(x,np.nan) for x in costs],color=["#72B7B2","#F58518","#4C78A8"]); axes[1].set(title="Custo temporal",ylabel="segundos")
    axes[2].bar(["parâmetros","pico GPU (MiB)"],[summary.get("parâmetros",np.nan),summary.get("pico GPU (MiB)",np.nan)],color=["#54A24B","#B279A2"]); axes[2].set(title="Capacidade e memória")
    for axis in axes: axis.grid(axis="y",alpha=.25)
    return figure


def plot_layer_statistics_visual(layers):
    """Média, desvio e esparsidade de todas as camadas monitoradas."""
    figure,axes=plt.subplots(1,2,figsize=(16,5),constrained_layout=True)
    names=layers["camada"].astype(str) if "camada" in layers else np.arange(len(layers)).astype(str)
    axes[0].bar(names,layers["média"],yerr=layers["desvio"],capsize=3,color="#4C78A8"); axes[0].set(title="Ativação média ± desvio",xlabel="Camada",ylabel="ativação"); axes[0].tick_params(axis="x",rotation=35)
    axes[1].bar(names,layers["% zero"],color="#E45756"); axes[1].set(title="Esparsidade por camada",xlabel="Camada",ylabel="% zero",ylim=(0,100)); axes[1].tick_params(axis="x",rotation=35)
    for axis in axes: axis.grid(axis="y",alpha=.25)
    return figure



def plot_segmentation_quality_dashboard(scene, algorithms=("seeds", "slic", "lsc", "watershed")):
    """Converte cada propriedade dos superpixels em gráficos comparáveis.

    O painel mede somente a geometria e a homogeneidade dos segmentos. Pureza não
    é acurácia do classificador: ela responde se uma região tende a conter uma
    única classe rotulada antes da etapa de classificação.
    """
    pixels, _ = to_pixel_table(scene)
    components, _ = apply_pca(pixels, n_components=3, random_state=42)
    rgb = rgb_composite(components, scene.shape)
    rows, area_samples = [], {}
    for algorithm in algorithms:
        local = ExperimentConfig(dataset=scene.code, segmenter=algorithm)
        segmentation = segment_superpixels(rgb, algorithm, _segmenter_params(local))
        rows.append({"segmentador": algorithm.upper(), **segmentation_quality(segmentation.labels, scene.labels)})
        area_samples[algorithm.upper()] = np.bincount(segmentation.labels.ravel())
    frame = pd.DataFrame(rows).set_index("segmentador")
    figure, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
    colors = list(plt.cm.tab10.colors[:len(frame)])
    frame["superpixels"].plot.bar(ax=axes[0, 0], color=colors, legend=False)
    axes[0, 0].set(title="Quantidade de superpixels", xlabel="Segmentador", ylabel="Regiões")
    frame[["área média", "área mediana"]].plot.bar(ax=axes[0, 1], color=["#4C78A8", "#F58518"])
    axes[0, 1].set(title="Tamanho das regiões", xlabel="Segmentador", ylabel="Pixels por região")
    frame["densidade de borda"].plot.bar(ax=axes[0, 2], color=colors, legend=False)
    axes[0, 2].set(title="Densidade de borda", xlabel="Segmentador", ylabel="Fração de vizinhos em borda")
    frame["pureza rotulada"].plot.bar(ax=axes[1, 0], color=colors, legend=False, ylim=(0, 1))
    axes[1, 0].set(title="Pureza rotulada", xlabel="Segmentador", ylabel="Fração da classe dominante")
    for name, areas in area_samples.items():
        axes[1, 1].hist(areas, bins=30, alpha=.45, label=name, density=True)
        axes[1, 2].plot(np.sort(areas), np.linspace(0, 1, len(areas)), label=name)
    axes[1, 1].set(title="Distribuição do tamanho das regiões", xlabel="Pixels por região", ylabel="Densidade")
    axes[1, 2].set(title="Escala acumulada das regiões", xlabel="Pixels por região", ylabel="Fração acumulada", ylim=(0, 1))
    for axis in axes.ravel():
        axis.grid(axis="y", alpha=.25)
    axes[0, 1].legend(title="Medição")
    axes[1, 1].legend(title="Segmentador", fontsize=8)
    axes[1, 2].legend(title="Segmentador", fontsize=8)
    figure.suptitle("Geometria e homogeneidade dos segmentadores", fontsize=15)
    return figure


def plot_segmentation_overlay_suite(scene,algorithms=("seeds","slic","lsc","watershed")):
    """Contornos de todos os segmentadores sobre a mesma composição PCA/RGB."""
    pixels,_=to_pixel_table(scene); components,_=apply_pca(pixels,n_components=3,random_state=42); rgb=rgb_composite(components,scene.shape)
    figure,axes=plt.subplots(2,2,figsize=(15,12),constrained_layout=True); colors=["#00FFFF","#FF00FF","#FFFF00","#FF7F0E"]
    for axis,algorithm,color in zip(axes.ravel(),algorithms,colors):
        local=ExperimentConfig(dataset=scene.code,segmenter=algorithm); segmentation=segment_superpixels(rgb,algorithm,_segmenter_params(local))
        axis.imshow(rgb)
        axis.contour(segmentation.contour_mask > 0, levels=[.5], colors=[color], linewidths=.45)
        axis.plot([], [], color=color, label="contorno do superpixel")
        axis.legend(loc="lower right", fontsize=8, framealpha=.8)
        axis.set(title=f"Sobreposição {algorithm.upper()}")
        axis.axis("off")
    return figure


### 3.4 Glossário de leitura dos gráficos

Os painéis apresentam métricas de qualidade, incerteza, explicabilidade e desempenho. Para evitar que um gráfico seja interpretado apenas pela cor ou pelo título, a célula seguinte cria uma legenda contextual: cada resultado exibido nas seções 3 e 8 mostra os termos pertinentes e como interpretá-los. **Confiança alta não garante acerto**; ela é a probabilidade máxima atribuída pelo modelo. **Entropia alta** indica distribuição de probabilidades mais dividida entre as classes e, portanto, maior incerteza.


In [ ]:
# ============================================================================
# GLOSSÁRIO VISUAL — legendas contextuais dos painéis
# ============================================================================
"""Definições curtas exibidas junto aos gráficos do notebook.

As métricas de teste são sempre de relato; HPO e escolha de receitas usam somente
validação. Isso evita vazamento do conjunto de teste para a seleção do modelo.
"""

from IPython.display import Markdown, display


VISUAL_GLOSSARY = {
    "dados": {
        "título": "Dados, PCA e segmentação",
        "termos": [
            ("RGB de visualização", "Composição de três bandas/componentes apenas para leitura humana; não equivale à imagem hiperespectral inteira."),
            ("PCA / componente principal", "Projeção que concentra variância espectral. Componentes não são classes e não possuem cores físicas fixas."),
            ("Superpixel", "Região de pixels visualmente/espectralmente semelhantes tratada como contexto espacial."),
            ("Densidade de borda", "Proporção de pixels de contorno; valores altos significam segmentação mais fragmentada."),
            ("Pureza rotulada", "Fração da classe dominante em cada superpixel; perto de 1 indica segmentos que respeitam melhor as fronteiras de classe."),
        ],
    },
    "avaliacao": {
        "título": "Qualidade preditiva e convergência",
        "termos": [
            ("Loss / cross-entropy", "Erro de classificação probabilístico; menor é melhor. É diferente de acurácia e penaliza previsões confiantes erradas."),
            ("OA", "Overall Accuracy: proporção total de pixels corretamente classificados; pode favorecer classes muito frequentes."),
            ("AA", "Average Accuracy: média das acurácias por classe. É mais representativa quando as classes são desbalanceadas."),
            ("Kappa", "Concordância além do acaso; 1 é concordância perfeita, 0 é aproximadamente acaso."),
            ("Precisão / recall", "Precisão mede se as predições de uma classe estavam corretas; recall mede quanto da classe real foi recuperado."),
            ("F1 macro", "Média entre classes do equilíbrio precisão–recall; cada classe tem o mesmo peso."),
            ("IoU macro", "Interseção sobre união média por classe; mede sobreposição entre classe prevista e real."),
            ("Melhor época / early stopping", "Linha tracejada: melhor validação. Linha pontilhada: fim antecipado após a janela sem ganho mínimo configurado."),
        ],
    },
    "explicabilidade": {
        "título": "O que a CNN usa para decidir",
        "termos": [
            ("Mapa de características / ativação", "Resposta de um filtro convolucional. Regiões intensas são padrões aos quais aquele filtro respondeu mais."),
            ("MaxPooling", "Mantém a maior ativação de cada vizinhança; realça evidências fortes, mas reduz resolução espacial."),
            ("Grad-CAM", "Mapa de importância por gradiente para a classe prevista. Mostra onde a última camada convolucional sustentou a decisão; não é segmentação real."),
            ("Saliência", "Sensibilidade da saída a cada entrada. Valores altos indicam bandas ou pixels cuja pequena alteração afeta mais a predição."),
            ("Confiança", "Maior probabilidade softmax entre as classes. Alta confiança pode estar errada e não representa calibração estatística por si só."),
            ("Entropia normalizada", "Incerteza da distribuição softmax: 0 significa uma classe dominante; 1 significa probabilidades muito divididas entre as classes."),
        ],
    },
    "comparacao": {
        "título": "Arquitetura da CNN 1D",
        "termos": [
            ("CNN1D", "Opera principalmente sobre a assinatura espectral de cada pixel."),
            ("Parâmetros", "Quantidade de pesos treináveis; mais parâmetros não implica automaticamente melhor generalização."),
            ("Tempo / memória", "Custo medido no ambiente da execução. Compare apenas receitas com mesma cena, split e orçamento de treino."),
        ],
    },
    "hpo": {
        "título": "HPO e seleção de modelos",
        "termos": [
            ("Trial", "Uma receita completa: CNN, segmentador e seus parâmetros, taxa de aprendizado, dropout e demais configurações."),
            ("Val accuracy (objetivo)", "Melhor acurácia no conjunto de validação; é o único critério de escolha do HPO."),
            ("OA/AA/Kappa de teste", "Relato final após o trial. Não participam da escolha, para não vazar informação do teste."),
            ("Vencedor", "Melhor trial de validação por CNN; empates usam menor tempo e número do trial como regra auditável."),
        ],
    },
    "campanha": {
        "título": "Campanha MLOps auditável",
        "termos": [
            ("Seed / repetição", "Inicialização e embaralhamento controlados. Repetições verificam se um resultado não foi apenas sorte."),
            ("Média e desvio", "Resumo da variabilidade entre seeds; desvio menor indica resultado mais estável."),
            ("IC 95%", "Intervalo de confiança aproximado da média; intervalos sobrepostos sugerem cautela ao declarar superioridade."),
            ("Artefatos", "Plano, configuração, ambiente, dados, histórico, métricas e checkpoint preservados para reprodução e auditoria."),
        ],
    },
    "espacial": {
        "título": "Explicação inversa na imagem",
        "termos": [
            ("Classe prevista", "Classe de maior probabilidade para cada pixel rotulado."),
            ("Mapa de erro", "Divergência entre previsão e rótulo de referência; mostra onde o modelo errou no conjunto avaliado."),
            ("Confiança por pixel", "Probabilidade máxima por pixel; leia sempre junto do mapa de erro e da entropia."),
            ("Entropia por pixel", "Incerteza normalizada: tons altos destacam pixels ambíguos, mesmo quando a classe prevista parece confiante."),
            ("Resumo por segmento", "Agrega acurácia, confiança, entropia e F1 dentro de cada superpixel para explicar regiões, não só pixels isolados."),
        ],
    },
    "batch": {
        "título": "Batch, throughput e uso de GPU",
        "termos": [
            ("Batch físico", "Número de amostras enviadas de uma vez à GPU."),
            ("Batch efetivo", "Amostras que contribuem para uma atualização após acumulação de gradientes."),
            ("Throughput", "Amostras de treino por segundo; maior é melhor, desde que a validação permaneça comparável."),
            ("OOM", "Out of Memory: batch não coube na GPU. A falha é mantida no relatório para tornar o limite auditável."),
            ("Batch recomendado", "Maior throughput dentro de 0,5% da melhor validação, sem trocar qualidade por velocidade silenciosamente."),
        ],
    },
}


def glossary_table(*sections):
    """Retorna termos e interpretações apenas dos painéis exibidos na célula."""
    rows = []
    for section in sections:
        block = VISUAL_GLOSSARY[section]
        rows.extend({"painel": block["título"], "termo": term, "como interpretar": meaning}
                    for term, meaning in block["termos"])
    return pd.DataFrame(rows)


def show_glossary(*sections):
    """Exibe legendas contextuais em cartões Markdown após cada grupo de gráficos."""
    labels = " • ".join(VISUAL_GLOSSARY[section]["título"] for section in sections)
    cards = []
    for section in sections:
        for term, meaning in VISUAL_GLOSSARY[section]["termos"]:
            cards.append(f"- **{term}:** {meaning}")
    display(Markdown(f"#### Como ler os gráficos — {labels}\n\n" + "\n".join(cards)))


In [ ]:
RUN_DATA_DIAGNOSTICS = True

if RUN_DATA_DIAGNOSTICS:
    scene_for_visuals = load_scene(config.dataset, download=False)
    display(plot_scene_overview(scene_for_visuals))
    pca_figure, _, _ = plot_pca_diagnostics(scene_for_visuals)
    display(pca_figure)
    segmentation_figure, _ = plot_segmentation_comparison(scene_for_visuals)
    display(segmentation_figure)
    display(plot_segmentation_overlay_suite(scene_for_visuals))
    display(plot_segmentation_quality_dashboard(scene_for_visuals))
    show_glossary('dados')
else:
    print('Diagnósticos de dados desativados.')


### 3.5 Métricas usadas e justificativa

O seu TCC define como núcleo **OA**, **AA**, **Kappa**, tempos, memória, parâmetros e speedup. OA mede o acerto total; AA é a média das revocações por classe e protege as classes raras; Kappa corrige a concordância esperada ao acaso. Este notebook acrescenta precisão, F1 macro e IoU macro por classe como diagnóstico de erros assimétricos. A matriz de confusão e o mapa de erros tornam esses números localizáveis na cena.

Referências: [seção de métricas do TCC](tcc/TCC%20III%20-%20EDUARDO%20SAVIAN/src/2_fundamentacao.tex), [planejamento experimental do TCC](tcc/TCC%20III%20-%20EDUARDO%20SAVIAN/src/3_projeto.tex), [revisão de CNNs em HSI](https://www.sciencedirect.com/org/science/article/pii/S1526149222001825) e [documentação das métricas do scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html).

## 4. Modelos CNN

### 4.1 Entradas, convoluções e fusão

- **CNN1D:** explora o espectro de cada pixel; opcionalmente recebe o ID do superpixel.

O factory `build_model` seleciona o modelo a partir de `config.architecture`.

In [ ]:
# ============================================================================
# MODEL
# ============================================================================
"""CNN 1D sobre a assinatura espectral, com fusão opcional do id do superpixel.

Convolui apenas ao longo das bandas de um pixel: a rede não enxerga vizinhança
alguma por conta própria, e é exatamente por isso que a segunda entrada — a que
diz a qual região homogênea o pixel pertence — tem efeito tão grande aqui.

As arquiteturas 2D e 3D foram implementadas, medidas e retiradas do escopo em
24/09; os resultados estão na entrada 11 do diário de experimentos e o código, no
histórico do git.
"""


import torch
import torch.nn as nn

__all__ = ["CNN1D", "apply_xavier_init", "build_model"]


def apply_xavier_init(module: nn.Module) -> None:
    """Inicializa convoluções e densas com Xavier e zera seus vieses."""
    if isinstance(module, (nn.Conv1d, nn.Linear)):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class _FusionClassifier:
    """Cabeça densa com a fusão da segunda entrada.

    A fusão é **tardia**: as convoluções nunca veem a segmentação, que entra só
    depois do ``Flatten``. Isso mantém os dois ramos de pré-processamento
    independentes — e é o que permite executá-los em paralelo.

    Dois modos, comparáveis porque tudo mais é idêntico:

    - ``scalar``: concatena o id cru como uma única coluna. Reproduz o baseline,
      mas trata um rótulo nominal como número contínuo (o id 78 vira 78,0) e a
      escala bruta acaba dando a essa coluna uma alavancagem desproporcional.
    - ``embedding``: uma ``nn.Embedding`` aprende um vetor por região. Não impõe
      ordem entre ids, não depende da magnitude e a inicialização padrão
      ``N(0, 1)`` deixa a segunda entrada na mesma escala das ativações.
    """

    def __init__(self, features: int, classes: int, two_input: bool, dropout: float,
                 fusion: str = "scalar", n_segments: int = 0, embedding_dim: int = 8) -> None:
        self.two_input = two_input
        self.fusion = fusion if two_input else "scalar"
        if self.two_input and self.fusion == "embedding":
            if n_segments < 1:
                raise ValueError("fusion='embedding' exige n_segments >= 1 (use PreparedData.n_segments).")
            self.segment_embedding = nn.Embedding(n_segments, embedding_dim)
            extra = embedding_dim
        else:
            self.segment_embedding = None
            extra = int(two_input)
        self.fused_features = features + extra
        self.classifier = nn.Sequential(
            nn.Linear(self.fused_features, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, classes),
        )

    def classify(self, features: torch.Tensor, seg: torch.Tensor | None) -> torch.Tensor:
        """Funde embedding e segmentação, validando a segunda entrada quando exigida."""
        if self.two_input:
            if seg is None:
                raise ValueError("Modelo com duas entradas: informe o tensor de segmentação.")
            if self.segment_embedding is not None:
                # O tensor chega como float (N, 1) do DataLoader; a tabela de
                # embeddings indexa por inteiro.
                seg = self.segment_embedding(seg.reshape(-1).long())
            features = torch.cat([features, seg], dim=1)
        return self.classifier(features)

    @property
    def n_parameters(self) -> int:
        """Número de pesos treináveis, útil ao comparar custo das arquiteturas."""
        return sum(parameter.numel() for parameter in self.parameters() if parameter.requires_grad)


#: Kernels das quatro convoluções da CNN1D, na ordem em que são aplicadas.
#: Sem padding, cada uma encurta a sequência em ``kernel - 1``.
CNN1D_KERNELS = (9, 3, 3, 3)

#: Menor número de componentes espectrais que a CNN1D aceita. Com 4 convoluções
#: sem padding consumindo 14 posições, uma entrada menor que isto deixa a saída
#: vazia e o PyTorch falha com "Kernel size can't be greater than actual input
#: size" — erro que aponta para a convolução e não para a causa real, que é a
#: escolha de `pca_components`.
CNN1D_MIN_COMPONENTS = sum(k - 1 for k in CNN1D_KERNELS) + 1


class CNN1D(nn.Module, _FusionClassifier):
    """Baseline: quatro Conv1d sobre a assinatura espectral ``(N, 1, Kb)``."""

    def __init__(self, seq_len: int, classes: int, in_channels: int = 1, two_input: bool = False,
                 dropout: float = 0.5, fusion: str = "scalar",
                 n_segments: int = 0, embedding_dim: int = 8) -> None:
        super().__init__()
        if seq_len < CNN1D_MIN_COMPONENTS:
            raise ValueError(
                f"CNN1D precisa de pelo menos {CNN1D_MIN_COMPONENTS} componentes "
                f"espectrais (kernels {CNN1D_KERNELS} consomem "
                f"{CNN1D_MIN_COMPONENTS - 1}); recebeu {seq_len}. "
                "Aumente `pca_components`."
            )
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=9), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3), nn.ReLU(), nn.Flatten(),
        )
        with torch.no_grad():
            self.n_flat = self.features(torch.zeros(1, in_channels, seq_len)).shape[1]
        _FusionClassifier.__init__(self, self.n_flat, classes, two_input, dropout,
                                   fusion, n_segments, embedding_dim)
        self.apply(apply_xavier_init)

    def forward(self, x: torch.Tensor, seg: torch.Tensor | None = None) -> torch.Tensor:
        return self.classify(self.features(x), seg)


def build_model(config, input_shape: int | tuple[int, ...], classes: int, device,
                n_segments: int = 0) -> tuple[nn.Module, nn.Module, torch.optim.Optimizer]:
    """Constrói a CNN1D, a loss e o Adam a partir da configuração.

    ``input_shape`` vem de ``PreparedData.input_shape``. Aceitar inteiro mantém
    compatibilidade com chamadas antigas, onde ele significava ``Kb`` diretamente.
    ``n_segments`` vem de ``PreparedData.n_segments`` e só é usado quando
    ``config.fusion == "embedding"``.
    """
    seq_len = input_shape if isinstance(input_shape, int) else input_shape[-1]
    model: nn.Module = CNN1D(
        seq_len, classes, two_input=config.two_input, dropout=config.dropout,
        fusion=config.fusion, n_segments=n_segments,
        embedding_dim=config.segment_embedding_dim,
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate, eps=config.adam_eps,
        weight_decay=config.weight_decay,
    )
    return model, criterion, optimizer


## 5. Treinar, avaliar e salvar

### 5.1 Laço de treino, validação e checkpoint

O treino usa batching na GPU, acumulação de gradientes quando necessário, parada antecipada e melhor checkpoint de validação. A avaliação registra OA, AA, kappa, métricas por classe e matriz de confusão.

In [ ]:
# ============================================================================
# TRAIN
# ============================================================================
"""Laço de treino: épocas, histórico e parada antecipada.

O histórico guarda quatro séries por época (`loss`, `accuracy`, `val_loss`,
`val_accuracy`). A parada antecipada monitora uma métrica de validação configurável
e encerra o treino após uma janela (`patience`) sem ganho mínimo significativo.
"""


import copy
import time
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

__all__ = ["EarlyStopping", "History", "TrainResult", "unpack_batch", "run_epoch", "fit"]


def unpack_batch(batch, device):
    """Batch -> `(x, seg, y)` já no device; `seg` é None no modo de uma entrada.

    Centraliza o desempacotamento para que os laços de treino e de predição não
    precisem saber quantas entradas o modelo tem.
    """
    if len(batch) == 3:
        x, seg, y = batch
        seg = seg.to(device, non_blocking=True)
    else:
        (x, y), seg = batch, None
    return x.to(device, non_blocking=True), seg, y.to(device, non_blocking=True)


class EarlyStopping:
    """Encerra após uma janela de épocas sem melhoria relevante na validação.

    Para ``val_accuracy``, uma época só reinicia a janela se aumentar a melhor
    acurácia por pelo menos ``min_delta``. Para ``val_loss``, a redução deve ter
    essa mesma magnitude. Opcionalmente restaura os pesos da melhor época.
    """

    def __init__(self, patience, min_delta, monitor, target, restore_best_weights):
        if monitor not in {"val_accuracy", "val_loss"}:
            raise ValueError("monitor deve ser 'val_accuracy' ou 'val_loss'.")
        self.patience = patience
        self.min_delta = min_delta
        self.monitor = monitor
        self.mode = "max" if monitor == "val_accuracy" else "min"
        self.target = target
        self.target_reached = False
        self.stop_reason = None
        self.restore_best_weights = restore_best_weights
        self.best = -np.inf if self.mode == "max" else np.inf
        self.best_epoch = 0
        self.wait = 0
        self.stopped_epoch = 0
        self._best_state = None

    def _improved(self, value):
        return value > self.best + self.min_delta if self.mode == "max" else value < self.best - self.min_delta

    def _target_reached(self, value):
        if self.target is None:
            return False
        return value >= self.target if self.mode == "max" else value <= self.target

    def _restore(self, model):
        if self.restore_best_weights and self._best_state is not None:
            model.load_state_dict(self._best_state)

    def step(self, value, model, epoch):
        """Registra uma época e para ao atingir meta ou esgotar patience."""
        if self._improved(value):
            self.best, self.best_epoch, self.wait = value, epoch, 0
            if self.restore_best_weights:
                self._best_state = copy.deepcopy(model.state_dict())
        else:
            self.wait += 1
        if self._target_reached(value):
            self.target_reached, self.stopped_epoch = True, epoch
            self.stop_reason = f"meta {self.monitor}={self.target:g} atingida"
            self._restore(model)
            return True
        if self.wait >= self.patience:
            self.stopped_epoch = epoch
            self.stop_reason = f"{self.patience} épocas sem melhoria de pelo menos {self.min_delta:g} em {self.monitor}"
            self._restore(model)
            return True
        return False


@dataclass
class History:
    """Séries por época: perda e acurácia, em treino e validação."""

    loss: list[float] = field(default_factory=list)
    accuracy: list[float] = field(default_factory=list)
    val_loss: list[float] = field(default_factory=list)
    val_accuracy: list[float] = field(default_factory=list)

    def append(self, loss: float, accuracy: float, val_loss: float, val_accuracy: float) -> None:
        self.loss.append(loss)
        self.accuracy.append(accuracy)
        self.val_loss.append(val_loss)
        self.val_accuracy.append(val_accuracy)

    def __len__(self) -> int:
        return len(self.loss)

    @property
    def epochs(self) -> range:
        return range(1, len(self) + 1)

    def to_dict(self) -> dict[str, list[float]]:
        return {
            "loss": self.loss,
            "accuracy": self.accuracy,
            "val_loss": self.val_loss,
            "val_accuracy": self.val_accuracy,
        }


@dataclass
class TrainResult:
    """Histórico e decisão de early stopping de uma execução."""

    history: History
    epoch_times: list[float]
    total_time: float
    stopped_early: bool
    best_val_loss: float
    best_epoch: int
    best_monitor_value: float
    early_stopping_monitor: str
    early_stopping_min_delta: float
    patience: int
    restore_best_weights: bool
    early_stopping_target: float | None
    target_reached: bool
    stop_reason: str

    def summary(self):
        status = (f"Early stopping na época {len(self.history)}: {self.stop_reason}."
                  if self.stopped_early else "Treino encerrou ao atingir o limite de épocas.")
        return (f"Melhor {self.early_stopping_monitor}: {self.best_monitor_value:.4f} "
                f"na época {self.best_epoch} (Δ mínimo: {self.early_stopping_min_delta:g}).\n"
                f"{status} Pesos restaurados: {self.restore_best_weights}.\n"
                f"Tempo total: {self.total_time:.2f}s | {len(self.epoch_times)} épocas")


def run_epoch(
    model,
    loader: DataLoader,
    criterion,
    device,
    optimizer=None,
    gradient_accumulation_steps: int = 1,
) -> tuple[float, float]:
    """Uma passagem completa sobre o loader.

    `optimizer=None` faz a passagem ser só de avaliação: sem backward, sem passo
    do otimizador e com o modelo em modo `eval` (dropout desligado).

    Quando ``gradient_accumulation_steps > 1``, cada batch contribui com sua soma
    de gradientes e a janela é normalizada pelo número **real** de amostras antes
    de ``optimizer.step``. Assim a última janela, que pode ser menor, continua
    equivalente a um batch concatenado. Com valor 1 o caminho numérico original
    é preservado, mantendo a reprodutibilidade do baseline.
    """
    if gradient_accumulation_steps < 1:
        raise ValueError("gradient_accumulation_steps deve ser ao menos 1.")

    training = optimizer is not None
    model.train(training)

    total_loss, total_correct, total = 0.0, 0, 0
    samples_in_window = 0
    if training:
        optimizer.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(training):
        for batch_index, batch in enumerate(loader, start=1):
            x, seg, y = unpack_batch(batch, device)

            logits = model(x, seg)
            loss = criterion(logits, y)

            if training:
                if gradient_accumulation_steps == 1:
                    # Mantém exatamente o caminho numérico do baseline.
                    loss.backward()
                else:
                    # CrossEntropyLoss devolve a média do batch. Acumulamos a
                    # soma e normalizamos uma vez pelo total real da janela; isso
                    # também trata corretamente a última janela incompleta.
                    (loss * y.size(0)).backward()
                    samples_in_window += y.size(0)
                if batch_index % gradient_accumulation_steps == 0 or batch_index == len(loader):
                    if gradient_accumulation_steps > 1:
                        for parameter in model.parameters():
                            if parameter.grad is not None:
                                parameter.grad.div_(samples_in_window)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    samples_in_window = 0

            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)

    return total_loss / total, total_correct / total


def fit(model, loaders, criterion, optimizer, config, device, verbose: bool = True) -> TrainResult:
    """Treina, valida, mede tempo e aplica parada antecipada.

    Cada época faz uma passagem de treino, outra de validação e sincroniza CUDA
    antes de medir duração. A acumulação pertence somente ao treino; validação
    sempre mede os logits do modelo sem gradientes. O retorno preserva todo o
    histórico necessário para curvas e para ``metrics.json``.
    """
    # Repõe a semente do gerador do loader de treino: criar um iterador sobre ele
    # (inspecionar um batch, por exemplo) sorteia uma permutação e adianta o estado.
    # Sem isso, olhar os dados antes de treinar mudaria a ordem dos batches — e o
    # resultado — de forma silenciosa.
    generator = getattr(loaders["train"], "generator", None)
    if generator is not None:
        generator.manual_seed(config.seed)

    stopper = EarlyStopping(
        patience=config.patience,
        min_delta=config.early_stopping_min_delta,
        monitor=config.early_stopping_monitor,
        target=config.early_stopping_target,
        restore_best_weights=config.restore_best_weights,
    )
    history = History()
    epoch_times: list[float] = []
    stopped_early = False

    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        epoch_start = time.perf_counter()
        train_loss, train_acc = run_epoch(
            model, loaders["train"], criterion, device, optimizer,
            gradient_accumulation_steps=config.gradient_accumulation_steps,
        )
        val_loss, val_acc = run_epoch(model, loaders["val"], criterion, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - epoch_start)

        history.append(train_loss, train_acc, val_loss, val_acc)

        if verbose:
            print(
                f"Epoch {epoch}/{config.epochs} - {epoch_times[-1]:.2f}s"
                f" - loss: {train_loss:.4f} - accuracy: {train_acc:.4f}"
                f" - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}"
            )

        monitored_value = val_acc if config.early_stopping_monitor == "val_accuracy" else val_loss
        if stopper.step(monitored_value, model, epoch):
            stopped_early = True
            break

    return TrainResult(
        history=history,
        epoch_times=epoch_times,
        total_time=time.perf_counter() - started,
        stopped_early=stopped_early,
        best_val_loss=float(min(history.val_loss)),
        best_epoch=stopper.best_epoch,
        best_monitor_value=float(stopper.best),
        early_stopping_monitor=config.early_stopping_monitor,
        early_stopping_min_delta=config.early_stopping_min_delta,
        patience=config.patience,
        restore_best_weights=config.restore_best_weights,
        early_stopping_target=config.early_stopping_target,
        target_reached=stopper.target_reached,
        stop_reason=stopper.stop_reason if stopped_early else "limite de épocas atingido",
    )

# ============================================================================
# EVALUATE
# ============================================================================
"""Predição e métricas de classificação (OA, AA, kappa, matriz de confusão)."""


from dataclasses import dataclass

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
)


__all__ = ["Metrics", "predict_proba", "evaluate", "binary_readout"]


@dataclass(frozen=True)
class Metrics:
    """Métricas padrão da literatura de classificação hiperespectral.

    OA (overall accuracy) pesa cada amostra igualmente; AA (average accuracy) pesa
    cada *classe* igualmente, e por isso cai quando as classes minoritárias vão
    mal. Kappa desconta o acerto esperado por acaso.
    """

    overall_accuracy: float
    average_accuracy: float
    kappa: float
    per_class_accuracy: np.ndarray
    confusion: np.ndarray
    test_loss: float
    test_accuracy: float
    report: str

    def summary(self) -> str:
        return (
            f"Test loss:     {self.test_loss}\n"
            f"Test accuracy: {self.test_accuracy}\n"
            f"OA:    {self.overall_accuracy}\n"
            f"AA:    {self.average_accuracy}\n"
            f"kappa: {self.kappa}"
        )


@torch.no_grad()
def predict_proba(model, loader, device) -> np.ndarray:
    """Probabilidades por classe para todas as amostras do loader.

    O softmax é explícito porque a última camada do modelo devolve logits: no
    treino quem o aplica é a `CrossEntropyLoss`.
    """
    model.eval()
    batches = []
    for batch in loader:
        x, seg, _ = unpack_batch(batch, device)
        batches.append(torch.softmax(model(x, seg), dim=1).cpu())
    return torch.cat(batches).numpy()


def _average_accuracy(confusion: np.ndarray) -> tuple[np.ndarray, float]:
    """Acurácia por classe (diagonal / suporte) e sua média.

    Uma classe sem nenhuma amostra no teste conta como 0 — mesmo resultado do
    `np.nan_to_num` do original, mas sem provocar divisão por zero.
    """
    support = confusion.sum(axis=1)
    per_class = np.divide(
        np.diag(confusion), support,
        out=np.zeros(len(support), dtype=float), where=support > 0,
    )
    return per_class, float(per_class.mean())


def evaluate(model, loader, criterion, device, y_true: np.ndarray) -> tuple[Metrics, np.ndarray]:
    """Roda a predição no loader e calcula todas as métricas.

    Retorna `(Metrics, classes_previstas)`.
    """
    probabilities = predict_proba(model, loader, device)
    predicted = np.argmax(probabilities, axis=1)

    confusion = confusion_matrix(y_true, predicted)
    per_class, average_accuracy = _average_accuracy(confusion)
    test_loss, test_accuracy = run_epoch(model, loader, criterion, device)

    metrics = Metrics(
        overall_accuracy=float(accuracy_score(y_true, predicted)),
        average_accuracy=average_accuracy,
        kappa=float(cohen_kappa_score(y_true, predicted)),
        per_class_accuracy=per_class,
        confusion=confusion,
        test_loss=test_loss,
        test_accuracy=test_accuracy,
        report=classification_report(y_true, predicted, zero_division=1),
    )
    return metrics, predicted


def binary_readout(confusion: np.ndarray) -> dict[str, float]:
    """Leitura binária das duas primeiras classes da matriz de confusão.

    Reproduz o cálculo da seção "Evaluating (Livro)" do notebook original, que
    aplica a fórmula de TP/TN/FP/FN de um problema **binário** sobre uma matriz
    multiclasse: só olha o bloco 2x2 do canto superior esquerdo e ignora todas as
    outras classes. Mantido para comparação com o trabalho original — não é uma
    métrica válida deste problema; use `Metrics` (OA/AA/kappa).
    """
    tn, fp = float(confusion[0][0]), float(confusion[0][1])
    fn, tp = float(confusion[1][0]), float(confusion[1][1])

    accuracy = (tp + tn) / (tp + tn + fn + fp)
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f_score = (2 * recall * precision) / (recall + precision) if (recall + precision) else float("nan")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f_score": f_score}

# ============================================================================
# PLOTS
# ============================================================================
"""Gráficos: curvas de treino, matriz de confusão e etapas da segmentação.

Todas as funções recebem os dados prontos e devolvem a figura — nada de ler
variável global nem de chamar `plt.show()` escondido dentro do cálculo, como no
notebook original.
"""


from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

__all__ = [
    "plot_history",
    "plot_confusion",
    "plot_pca_components",
    "plot_rgb_channels",
    "plot_segmentation_steps",
]


def _grid(images: dict[str, np.ndarray], cmaps=None, figsize=(20, 10), columns: int = 4):
    cmaps = cmaps or {}
    fig = plt.figure(figsize=figsize)
    for i, (title, image) in enumerate(images.items(), start=1):
        axis = fig.add_subplot(2, columns, i)
        axis.imshow(image, cmap=cmaps.get(title))
        axis.axis("off")
        axis.set_title(title)
    return fig


def plot_history(history, metric: str, save_to: Path | None = None):
    """Curva de treino x validação para 'Loss' ou 'Accuracy'."""
    keys = {"Loss": ("loss", "val_loss"), "Accuracy": ("accuracy", "val_accuracy")}
    if metric not in keys:
        raise ValueError(f"metric deve ser um de {sorted(keys)}, recebido {metric!r}")
    train_key, val_key = keys[metric]
    values = history.to_dict()

    fig = plt.figure(figsize=(10, 7))
    plt.plot(history.epochs, values[train_key], "g", label="Training")
    plt.plot(history.epochs, values[val_key], "b", label="Validation")
    plt.title(f"Training and Validation {metric}")
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend(loc="upper center")
    plt.grid()
    if save_to is not None:
        plt.savefig(save_to)
    return fig


def plot_confusion(confusion: np.ndarray, normalize: bool = False, save_to: Path | None = None):
    """Matriz de confusão; `normalize=True` mostra o percentual por linha."""
    import pandas as pd
    import seaborn as sns

    matrix, fmt = confusion, "d"
    if normalize:
        matrix = 100 * confusion.astype(float) / confusion.sum(axis=1)[:, np.newaxis]
        fmt = ".2f"

    fig = plt.figure(figsize=(10, 7))
    sns.heatmap(pd.DataFrame(matrix), annot=True, fmt=fmt, cmap="Blues")
    plt.xlabel("Previsão")
    plt.ylabel("Ground Truth")
    if save_to is not None:
        plt.savefig(save_to)
    return fig


def plot_pca_components(components: np.ndarray, shape: tuple[int, int], count: int = 3):
    """As primeiras componentes principais como imagens."""
    height, width = shape
    return _grid({
        f"Band - {i}": components[:, i].reshape(height, width) for i in range(count)
    }, cmaps={f"Band - {i}": "nipy_spectral" for i in range(count)})


def plot_rgb_channels(rgb: np.ndarray):
    """Os três canais da composição RGB, cada um em seu próprio degradê."""
    def ramp(color):
        return plt.cm.colors.ListedColormap(np.linspace([0, 0, 0], color, 256))

    channels = {"Red": (rgb[:, :, 0], [1, 0, 0]), "Green": (rgb[:, :, 1], [0, 1, 0]), "Blue": (rgb[:, :, 2], [0, 0, 1])}
    return _grid(
        {name: image for name, (image, _) in channels.items()},
        cmaps={name: ramp(color) for name, (_, color) in channels.items()},
    )


def plot_segmentation_steps(steps: dict[str, np.ndarray]):
    """Máscara de contorno, fundo, frente e composição final do SEEDS."""
    return _grid(steps)

# ============================================================================
# CHECKPOINT
# ============================================================================
"""Salvar e recarregar a CNN1D treinada.

O padrão do PyTorch é salvar apenas o `state_dict` (os tensores de peso) e
reconstruir a arquitetura antes de carregá-lo. Por isso o checkpoint guarda
também `model_type` e o layout da entrada que `load_checkpoint` precisa para
remontar a rede correta.
"""


from pathlib import Path

import torch


__all__ = ["save_checkpoint", "load_checkpoint"]


def save_checkpoint(model, config: ExperimentConfig, input_shape: int | tuple[int, ...], classes: int, path: Path) -> Path:
    """Grava pesos e a arquitetura necessária para reconstruí-los.

    ``input_shape`` preserva o layout específico: ``(1, Kb)`` para 1D,
    ``(3, P, P)`` para 2D e ``(1, B, P, P)`` para 3D.
    """
    torch.save(
        {
            "state_dict": model.state_dict(),
            "input_shape": list(input_shape) if not isinstance(input_shape, int) else input_shape,
            "classes": classes,
            "model_type": config.model_type,
            "two_input": config.two_input,
            "dropout": config.dropout,
            "fusion": config.fusion,
            "n_segments": getattr(model, "segment_embedding", None).num_embeddings
            if getattr(model, "segment_embedding", None) is not None else 0,
            "embedding_dim": config.segment_embedding_dim,
            "dataset": config.dataset,
            "config": config.to_dict(),
        },
        path,
    )
    return path


def load_checkpoint(path: Path, device=None) -> tuple[object, dict]:
    """Reconstrói a arquitetura, carrega os pesos e devolve o modelo em `eval()`."""
    device = device or torch.device("cpu")
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model_type = checkpoint.get("model_type", "cnn1d")
    input_shape = checkpoint.get("input_shape", checkpoint.get("seq_len"))
    # Checkpoints anteriores à fusão por embedding não têm estas chaves; os
    # defaults reproduzem exatamente o comportamento escalar antigo.
    fusion_kwargs = {
        "fusion": checkpoint.get("fusion", "scalar"),
        "n_segments": checkpoint.get("n_segments", 0),
        "embedding_dim": checkpoint.get("embedding_dim", 8),
    }
    if model_type != "cnn1d":
        raise ValueError(f"Checkpoint de arquitetura fora do escopo: {model_type!r}.")
    seq_len = input_shape if isinstance(input_shape, int) else input_shape[-1]
    model = CNN1D(seq_len, checkpoint["classes"], two_input=checkpoint["two_input"],
                  dropout=checkpoint.get("dropout", 0.5), **fusion_kwargs)
    model = model.to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint


### 5.2 Interpretabilidade visual das CNNs

Esta seção responde **o que a rede ativou** e **onde/por qual dimensão a decisão foi mais sensível**. Ela separa três ideias: ativações por filtro, redução feita por MaxPooling e atribuição à classe prevista via Grad-CAM/saliência. Nenhuma delas substitui OA, AA, Kappa ou validação experimental.

In [ ]:
# ============================================================================
# EXPLICABILIDADE — ativações, MaxPooling, saliência, Grad-CAM e comparação
# ============================================================================
"""Ferramentas de explicabilidade visual para as CNNs 1D, 2D e 3D.

Os hooks observam saídas intermediárias durante uma inferência e são removidos no
fim. Portanto, não alteram pesos, gradientes de treino nem a predição registrada.
Ativação não é uma explicação causal completa: ela indica quais padrões passaram
pelas camadas; Grad-CAM e saliência complementam mostrando a influência local na
classe prevista.
"""

from __future__ import annotations

from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F


def explanation_sample(result: ExperimentResult, split: str = "test", batch_index: int = 0):
    """Obtém uma única amostra já no device, estável e sem alterar o DataLoader.

    `batch_index` escolhe a posição dentro do primeiro batch do split. Para explicar
    outro pixel, aumente-o até `batch_size - 1` ou construa um DataLoader dedicado.
    """
    loaders = build_loaders(result.data, result.config, result.device)
    x, seg, y = unpack_batch(next(iter(loaders[split])), result.device)
    if not 0 <= batch_index < len(x):
        raise IndexError(f"batch_index deve estar entre 0 e {len(x) - 1}.")
    return x[batch_index:batch_index + 1], None if seg is None else seg[batch_index:batch_index + 1], y[batch_index].item()


def capture_activations(model, x: torch.Tensor, seg: torch.Tensor | None = None) -> list[dict]:
    """Captura Conv, MaxPool e pooling adaptativo da passagem direta de uma amostra."""
    watched = (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d,
               torch.nn.MaxPool1d, torch.nn.MaxPool2d, torch.nn.MaxPool3d,
               torch.nn.AdaptiveAvgPool1d, torch.nn.AdaptiveAvgPool2d, torch.nn.AdaptiveAvgPool3d)
    records, handles = [], []
    for name, module in model.named_modules():
        if not isinstance(module, watched):
            continue
        def hook(_module, _inputs, output, module_name=name, module_type=type(module).__name__):
            tensor = output[0] if isinstance(output, tuple) else output
            records.append({"nome": module_name, "tipo": module_type, "ativação": tensor.detach()[0].float().cpu().numpy()})
        handles.append(module.register_forward_hook(hook))
    previous_mode = model.training
    model.eval()
    try:
        with torch.no_grad():
            logits = model(x, seg)
    finally:
        for handle in handles:
            handle.remove()
        model.train(previous_mode)
    predicted = int(logits.argmax(dim=1).item())
    return records, predicted


def _activation_image(activation: np.ndarray, channel: int) -> np.ndarray:
    """Reduz uma ativação de CNN1D para uma imagem inspecionável."""
    raise ValueError("Ativação não possui dimensão espacial para imagem.")


def plot_activation_atlases(records: list[dict], max_channels: int = 12) -> dict[str, object]:
    """Gera uma figura por camada, com os mapas de características mais ativos.

    Cada linha é um filtro e o eixo horizontal representa as posições espectrais:
    mostra em que faixas do espectro cada filtro reage.
    """
    figures = {}
    for record in records:
        activation = record["ativação"]
        name = f"{record['nome']} ({record['tipo']})"
        if activation.ndim == 2:  # C, L — CNN1D
            channels = min(max_channels, activation.shape[0])
            fig, axis = plt.subplots(figsize=(14, max(3, channels * .35)), constrained_layout=True)
            im = axis.imshow(activation[:channels], cmap="magma", aspect="auto")
            axis.set(title=f"Ativações espectrais — {name}", xlabel="Posição espectral", ylabel="Filtro")
            fig.colorbar(im, ax=axis, label="Ativação")
        else:  # C,H,W ou C,D,H,W
            channels = min(max_channels, activation.shape[0])
            columns = min(4, channels)
            rows = int(np.ceil(channels / columns))
            fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.5 * rows), constrained_layout=True)
            axes = np.atleast_1d(axes).ravel()
            scores = np.mean(np.abs(activation), axis=tuple(range(1, activation.ndim)))
            selected = np.argsort(scores)[-channels:][::-1]
            for axis, channel in zip(axes, selected):
                image = _activation_image(activation, int(channel))
                im = axis.imshow(image, cmap="magma")
                axis.set_title(f"filtro {channel}")
                axis.axis("off")
                fig.colorbar(im, ax=axis, shrink=.72)
            for axis in axes[len(selected):]:
                axis.axis("off")
            dimension = "projeção máxima nas bandas" if activation.ndim == 4 else "mapa espacial"
            fig.suptitle(f"Ativações — {name} ({dimension})", fontsize=14)
        figures[name] = fig
    return figures


def plot_maxpool_effect(records: list[dict]) -> dict[str, object]:
    """Compara entrada e saída de cada MaxPooling disponível na arquitetura.

    O painel mostra o filtro mais ativo imediatamente antes e depois do pool. O
    MaxPooling não 'entende' uma classe sozinho: ele retém a maior resposta local,
    reduzindo resolução e tornando a representação mais tolerante a pequenos
    deslocamentos. CNN1D atual não possui MaxPooling; ela preserva a resolução
    espectral até o flatten, o que também fica explícito no aviso retornado.
    """
    figures = {}
    pool_indices = [i for i, item in enumerate(records) if "MaxPool" in item["tipo"]]
    if not pool_indices:
        print("Esta CNN1D não usa MaxPooling: as convoluções preservam as posições espectrais até o Flatten.")
        return figures
    for index in pool_indices:
        before, after = records[index - 1], records[index]
        input_activation, output_activation = before["ativação"], after["ativação"]
        scores = np.mean(np.abs(input_activation), axis=tuple(range(1, input_activation.ndim)))
        channel = int(np.argmax(scores))
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
        if input_activation.ndim == 2:
            axes[0].plot(input_activation[channel]); axes[1].plot(output_activation[channel])
            for axis in axes:
                axis.set(xlabel="Posição espectral", ylabel="Ativação")
        else:
            left = _activation_image(input_activation, channel)
            right = _activation_image(output_activation, channel)
            axes[0].imshow(left, cmap="magma"); axes[1].imshow(right, cmap="magma")
            for axis in axes:
                axis.axis("off")
        axes[0].set_title(f"Antes: {before['nome']}\n{input_activation.shape}")
        axes[1].set_title(f"Depois: {after['nome']}\n{output_activation.shape}")
        fig.suptitle(f"Efeito do MaxPooling — filtro {channel}: mantém a resposta local máxima")
        figures[after["nome"]] = fig
    return figures


def grad_cam(model, x: torch.Tensor, seg: torch.Tensor | None = None) -> tuple[np.ndarray, int, str]:
    """Calcula Grad-CAM no último Conv, condicionado à classe prevista.

    O gradiente da pontuação da classe pesa os filtros da última convolução. O
    resultado destaca regiões/posições cuja ativação mais elevou essa previsão;
    não é uma prova de causalidade nem substitui a avaliação quantitativa.
    """
    conv_types = (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d)
    candidates = [(name, module) for name, module in model.named_modules() if isinstance(module, conv_types)]
    if not candidates:
        raise ValueError("Modelo sem camada convolucional para Grad-CAM.")
    name, target = candidates[-1]
    cache = {}
    forward = target.register_forward_hook(lambda _m, _i, out: cache.__setitem__("activation", out))
    backward = target.register_full_backward_hook(lambda _m, _gi, go: cache.__setitem__("gradient", go[0]))
    previous_mode = model.training
    model.eval()
    try:
        model.zero_grad(set_to_none=True)
        logits = model(x, seg)
        predicted = int(logits.argmax(dim=1).item())
        logits[0, predicted].backward()
        activation, gradient = cache["activation"], cache["gradient"]
        axes = tuple(range(2, activation.ndim))
        weights = gradient.mean(dim=axes, keepdim=True)
        cam = torch.relu((weights * activation).sum(dim=1))[0]
        cam = cam.detach().float().cpu().numpy()
        cam = (cam - cam.min()) / max(cam.max() - cam.min(), 1e-12)
    finally:
        forward.remove(); backward.remove(); model.train(previous_mode)
    return cam, predicted, name


def plot_grad_cam(cam: np.ndarray, model_type: str, predicted: int, layer_name: str):
    """Renderiza Grad-CAM como perfil espectral, mapa espacial ou projeção 3D."""
    if model_type == "cnn1d":
        fig, axis = plt.subplots(figsize=(14, 4), constrained_layout=True)
        axis.plot(cam, color="crimson", lw=2)
        axis.fill_between(np.arange(len(cam)), cam, alpha=.25, color="crimson")
        axis.set(title=f"Grad-CAM espectral — classe prevista {predicted + 1} ({layer_name})", xlabel="Posição após convoluções", ylabel="Importância normalizada")
        axis.grid(alpha=.25)
        return fig
        image = axis.imshow(cam, cmap="jet", vmin=0, vmax=1)
        axis.set(title=f"Grad-CAM espacial — classe prevista {predicted + 1}", xlabel="Patch X", ylabel="Patch Y")
        fig.colorbar(image, ax=axis, label="Importância")
        return fig
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    spatial = cam.max(axis=0)
    spectral = cam.mean(axis=(1, 2))
    image = axes[0].imshow(spatial, cmap="jet", vmin=0, vmax=1)
    axes[0].set(title=f"Grad-CAM 3D: projeção espacial\nclasse {predicted + 1}", xlabel="Patch X", ylabel="Patch Y")
    fig.colorbar(image, ax=axes[0], label="Importância")
    axes[1].plot(spectral, color="crimson", lw=2)
    axes[1].set(title="Grad-CAM 3D: importância espectral", xlabel="Posição espectral", ylabel="Importância média")
    axes[1].grid(alpha=.25)
    return fig


def plot_input_saliency(model, x: torch.Tensor, seg: torch.Tensor | None = None, model_type: str = "cnn1d"):
    """Mostra sensibilidade local da classe prevista à entrada (|d logit / d entrada|)."""
    input_tensor = x.detach().clone().requires_grad_(True)
    previous_mode = model.training
    model.eval(); model.zero_grad(set_to_none=True)
    logits = model(input_tensor, seg)
    predicted = int(logits.argmax(dim=1).item())
    logits[0, predicted].backward()
    saliency = input_tensor.grad.detach().abs()[0].float().cpu().numpy()
    model.train(previous_mode)
    if model_type == "cnn1d":
        fig, axis = plt.subplots(figsize=(14, 4), constrained_layout=True)
        axis.plot(saliency[0], color="#4C78A8")
        axis.set(title=f"Saliência da entrada espectral — classe prevista {predicted + 1}", xlabel="Componente PCA", ylabel="|gradiente|")
        axis.grid(alpha=.25)
        image = axis.imshow(saliency.sum(axis=0), cmap="inferno")
        axis.set(title=f"Saliência espacial — classe prevista {predicted + 1}", xlabel="Patch X", ylabel="Patch Y")
        fig.colorbar(image, ax=axis, label="|gradiente| somado")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
        image = axes[0].imshow(saliency[0].sum(axis=0), cmap="inferno")
        axes[0].set(title="Saliência 3D: espacial", xlabel="Patch X", ylabel="Patch Y")
        fig.colorbar(image, ax=axes[0], label="|gradiente|")
        axes[1].plot(saliency[0].sum(axis=(1, 2)), color="#4C78A8")
        axes[1].set(title="Saliência 3D: espectral", xlabel="Banda PCA", ylabel="|gradiente|")
        axes[1].grid(alpha=.25)
    return fig


def plot_prediction_uncertainty(result: ExperimentResult, scene: Scene):
    """Plota confiança e entropia da previsão nos pixels de teste.

    Confiança baixa ou entropia alta sinalizam regiões candidatas a inspeção ou
    aquisição de novos rótulos; não significam automaticamente que a previsão é
    errada. A normalização de entropia mantém a escala em [0, 1].
    """
    loaders = build_loaders(result.data, result.config, result.device)
    probabilities = predict_proba(result.model, loaders["test"], result.device)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(np.maximum(probabilities, 1e-12))).sum(axis=1) / np.log(probabilities.shape[1])
    positions = np.flatnonzero(scene.labels.ravel() > 0)[result.data.splits.test]
    confidence_map = np.full(scene.labels.size, np.nan); confidence_map[positions] = confidence
    entropy_map = np.full(scene.labels.size, np.nan); entropy_map[positions] = entropy
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
    for axis, values, title, cmap in (
        (axes[0], confidence_map.reshape(scene.shape), "Confiança máxima", "viridis"),
        (axes[1], entropy_map.reshape(scene.shape), "Entropia preditiva", "magma"),
    ):
        image = axis.imshow(values, cmap=cmap, vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
        fig.colorbar(image, ax=axis, label="0 → baixo | 1 → alto")
    axes[2].hist(confidence, bins=20, alpha=.7, label="confiança")
    axes[2].hist(entropy, bins=20, alpha=.7, label="entropia")
    axes[2].set(title="Distribuição de incerteza no teste", xlabel="Pontuação", ylabel="Pixels")
    axes[2].legend(); axes[2].grid(alpha=.25)
    return fig


In [ ]:
# ============================================================================
# DASHBOARDS PADRONIZADOS POR CNN — todos os layers e compilado final
# ============================================================================
"""Dashboard da CNN1D.

Cada arquitetura recebe exatamente o mesmo protocolo de avaliação: métricas
globais, métricas por classe, custo, incerteza, Grad-CAM, saliência e estatísticas
de todas as camadas-folha. A visualização de um layer muda apenas para respeitar a
geometria do tensor: espectro (1D), mapa (2D) ou projeção de volume (3D).
"""

import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def capture_all_layer_outputs(model, x, seg=None):
    """Observa cada camada-folha executada numa inferência da amostra escolhida.

    Inclui convoluções, ReLUs, MaxPooling, pooling adaptativo, Flatten, camadas
    lineares e Dropout. A fusão de segmento é uma operação funcional (`torch.cat`)
    e, por não ser um módulo PyTorch, aparece indiretamente na primeira Linear da
    cabeça classificadora.
    """
    records, handles = [], []
    for name, module in model.named_modules():
        if not name or any(module.children()):
            continue
        def hook(_module, _inputs, output, layer_name=name, layer_type=type(module).__name__):
            value = output[0] if isinstance(output, tuple) else output
            if torch.is_tensor(value):
                records.append({
                    "nome": layer_name,
                    "tipo": layer_type,
                    "ativação": value.detach()[0].float().cpu().numpy(),
                })
        handles.append(module.register_forward_hook(hook))
    previous_mode = model.training
    model.eval()
    try:
        with torch.no_grad():
            logits = model(x, seg)
    finally:
        for handle in handles:
            handle.remove()
        model.train(previous_mode)
    return records, int(logits.argmax(dim=1).item())


def layer_statistics(records):
    """Resume shape, dispersão e esparsidade de todas as ativações capturadas."""
    rows = []
    for index, record in enumerate(records, start=1):
        values = record["ativação"]
        rows.append({
            "ordem": index,
            "layer": record["nome"],
            "tipo": record["tipo"],
            "shape": " × ".join(map(str, values.shape)),
            "média": float(values.mean()),
            "desvio": float(values.std()),
            "mínimo": float(values.min()),
            "máximo": float(values.max()),
            "% zero": float(100 * np.mean(np.isclose(values, 0))),
        })
    return pd.DataFrame(rows)


def _layer_visual(axis, values, layer_type):
    """Escolhe a representação mais legível para a geometria de cada layer."""
    if values.ndim == 1:  # Flatten e Linear
        visible = min(96, len(values))
        axis.bar(np.arange(visible), values[:visible], width=1, color="#4C78A8")
        axis.set(xlabel="Neurônio", ylabel="Ativação")
    elif values.ndim == 2:  # filtros × posição: CNN1D
        image = axis.imshow(values, aspect="auto", cmap="magma")
        axis.set(xlabel="Posição espectral", ylabel="Filtro")
        plt.colorbar(image, ax=axis, fraction=.046)
        axis.set(xlabel="X", ylabel="Y")
        plt.colorbar(image, ax=axis, fraction=.046)
        axis.set(xlabel="X", ylabel="Y")
        plt.colorbar(image, ax=axis, fraction=.046)
    else:
        axis.text(.5, .5, f"Tensor {values.shape}\nsem projeção visual", ha="center", va="center")
    axis.set_title(f"{layer_type}: {values.shape}", fontsize=9)
    axis.grid(False)


def plot_all_layers(records, layers_per_page=6):
    """Cria páginas de dashboard para *todos* os layers, inclusive ReLU e Linear.

    À esquerda está a representação da ativação; à direita, o histograma dela. As
    páginas evitam miniaturas ilegíveis em redes com muitas camadas.
    """
    pages = {}
    for page_start in range(0, len(records), layers_per_page):
        subset = records[page_start:page_start + layers_per_page]
        fig, axes = plt.subplots(len(subset), 2, figsize=(15, 4.2 * len(subset)), constrained_layout=True)
        axes = np.atleast_2d(axes)
        for row, record in enumerate(subset):
            values = record["ativação"]
            _layer_visual(axes[row, 0], values, record["tipo"])
            axes[row, 0].set_title(f"{record['nome']} — {record['tipo']} | {values.shape}", loc="left", fontsize=10, fontweight="bold")
            axes[row, 1].hist(values.ravel(), bins=40, color="#F58518", alpha=.85)
            axes[row, 1].set(title="Distribuição das ativações", xlabel="Valor", ylabel="Contagem")
            axes[row, 1].grid(alpha=.2)
        page = page_start // layers_per_page + 1
        fig.suptitle(f"Todas as camadas — página {page}", fontsize=15)
        pages[f"camadas {page}"] = fig
    return pages


def dashboard_summary(result):
    """Calcula o mesmo vetor de métricas para qualquer uma das três arquiteturas."""
    y_true = result.data.labels[result.data.splits.test]
    y_pred = result.predicted
    per_class, extra = classification_metrics_table(y_true, y_pred)
    inference = profile_inference(result).iloc[0].to_dict()
    return {
        "CNN": result.config.model_type.upper(),
        "OA": result.metrics.overall_accuracy,
        "AA": result.metrics.average_accuracy,
        "Kappa": result.metrics.kappa,
        "precisão macro": float(per_class["precisão"].mean()),
        "F1 macro": extra["F1 macro"],
        "IoU macro": extra["IoU macro"],
        "loss teste": result.metrics.test_loss,
        "pré-processamento (s)": result.prep_timings.wall,
        "treino (s)": result.training.total_time,
        "inferência (s)": inference["inferência (s)"],
        "ms / pixel": inference["ms / pixel"],
        "pico GPU (MiB)": inference["pico GPU (MiB)"],
        "parâmetros": result.model.n_parameters,
    }, per_class


def _render_cnn_dashboard_core(result, scene, max_channels=8):
    """Núcleo do dashboard explicável de uma CNN já treinada.

    Retorna os artefatos em vez de chamar `display`, permitindo que a mesma função
    seja usada no dashboard individual e no compilado final. A versão
    pública `render_cnn_dashboard`, na seção 5.6, acrescenta a explicação espacial.
    """
    x, seg, true_class = explanation_sample(result, split="test", batch_index=0)
    records, predicted_class = capture_all_layer_outputs(result.model, x, seg)
    summary, per_class = dashboard_summary(result)
    cam, cam_class, cam_layer = grad_cam(result.model, x, seg)
    figures = {
        "métricas e erros": plot_evaluation_dashboard(result, scene)[0],
        "saliência": plot_input_saliency(result.model, x, seg, result.config.model_type),
        "Grad-CAM": plot_grad_cam(cam, result.config.model_type, cam_class, cam_layer),
        "incerteza": plot_prediction_uncertainty(result, scene),
    }
    figures.update({f"{key}": value for key, value in plot_all_layers(records).items()})
    figures.update({f"MaxPool — {key}": value for key, value in plot_maxpool_effect(records).items()})
    return {
        "summary": summary,
        "per_class": per_class,
        "layers": layer_statistics(records),
        "figures": figures,
        "classe real": true_class + 1,
        "classe prevista": predicted_class + 1,
    }


def _display_cnn_dashboard_core(name, dashboard):
    """Núcleo da exibição de um dashboard individual: gráficos e legendas."""
    from IPython.display import Markdown, display
    display(Markdown(f"## Dashboard — {name.upper()}"))
    display(plot_cnn_summary_visual(dashboard["summary"]))
    display(plot_per_class_metrics(dashboard["per_class"]))
    display(plot_layer_statistics_visual(dashboard["layers"]))
    print(f"Pixel explicado — classe real: {dashboard['classe real']} | prevista: {dashboard['classe prevista']}")
    for title, figure in dashboard["figures"].items():
        display(Markdown(f"#### {title}"))
        display(figure)
        plt.close(figure)
def plot_final_cnn_compilation(dashboards):
    """Compila as mesmas métricas, classes e custo de todas as CNNs em uma visão final."""
    summary = pd.DataFrame([dashboard["summary"] for dashboard in dashboards.values()]).set_index("CNN")
    per_class_rows = []
    for name, dashboard in dashboards.items():
        table = dashboard["per_class"].copy()
        table["CNN"] = name.upper()
        per_class_rows.append(table)
    per_class = pd.concat(per_class_rows, ignore_index=True)
    ordered = summary.index.tolist()
    fig, axes = plt.subplots(2, 3, figsize=(21, 12), constrained_layout=True)
    summary[["OA", "AA", "Kappa", "precisão macro", "F1 macro", "IoU macro"]].T.plot.bar(ax=axes[0, 0], rot=0)
    axes[0, 0].set(title="Métricas globais iguais para todas as CNNs", ylabel="Pontuação", ylim=(0, 1))
    summary[["pré-processamento (s)", "treino (s)", "inferência (s)"]].plot.bar(ax=axes[0, 1], rot=0)
    axes[0, 1].set(title="Custo temporal", ylabel="Segundos")
    summary[["parâmetros", "pico GPU (MiB)"]].plot.bar(ax=axes[0, 2], rot=0)
    axes[0, 2].set(title="Capacidade e memória")
    f1_matrix = per_class.pivot(index="CNN", columns="classe", values="F1").reindex(ordered)
    im = axes[1, 0].imshow(f1_matrix, aspect="auto", vmin=0, vmax=1, cmap="YlGnBu")
    axes[1, 0].set(title="F1 por classe", xlabel="Classe", ylabel="CNN", xticks=range(f1_matrix.shape[1]), xticklabels=f1_matrix.columns)
    fig.colorbar(im, ax=axes[1, 0], label="F1")
    recall_matrix = per_class.pivot(index="CNN", columns="classe", values="revocação / AA por classe").reindex(ordered)
    im = axes[1, 1].imshow(recall_matrix, aspect="auto", vmin=0, vmax=1, cmap="magma")
    axes[1, 1].set(title="Revocação / AA por classe", xlabel="Classe", ylabel="CNN", xticks=range(recall_matrix.shape[1]), xticklabels=recall_matrix.columns)
    fig.colorbar(im, ax=axes[1, 1], label="Recall")
    axes[1, 2].scatter(summary["parâmetros"], summary["OA"], s=120, c=summary["treino (s)"], cmap="viridis", edgecolor="black")
    for name, row in summary.iterrows():
        axes[1, 2].annotate(name, (row["parâmetros"], row["OA"]), xytext=(5, 5), textcoords="offset points")
    axes[1, 2].set(title="Trade-off: capacidade × OA", xlabel="Parâmetros", ylabel="OA", ylim=(0, 1))
    axes[1, 2].grid(alpha=.25)
    for axis in axes.ravel()[:3]:
        axis.grid(axis="y", alpha=.25)
    return fig, summary, per_class


### 5.4 Otimização automática de hiperparâmetros

Três estratégias percorrem **o mesmo espaço**, com **o mesmo orçamento de trials** e **a mesma cobertura inicial** dos quatro segmentadores. A única diferença é como cada candidato é escolhido — é o que torna a comparação entre elas honesta.

| Estratégia | Como escolhe o próximo candidato |
| --- | --- |
| `random` | Sorteio uniforme, sem memória. É a linha de base. |
| `tpe` | Hipertunador bayesiano (Optuna). Modela a densidade dos trials bons e a dos ruins e propõe pontos que maximizam a razão entre elas. |
| `genetic` | Algoritmo genético elitista: seleção por torneio, crossover uniforme, mutação por gene e elite que passa sem retreinar. |

O espaço é totalmente discreto: `HPO_SPACE` define learning rate, dropout, weight decay, batch e componentes de PCA, e mais dois genes cobrem o segmentador e qual das três receitas internas dele usar. São 2.916 combinações para a CNN1D e 8.748 para as demais — grande demais para grade exaustiva, que é justamente o motivo de existirem estratégias de busca.

A representação comum é uma **tupla de índices**, um por gene (`hpo_genes` / `hpo_decode`). É ela que permite ao algoritmo genético fazer crossover sem codificação binária e ao TPE tratar o segmentador como categórico e os demais eixos como ordinais.

A seleção usa exclusivamente a validação; métricas de teste são relato, nunca critério. `compare_hpo_strategies` roda as três e `plot_hpo_strategies` mostra a curva de convergência — compare pelo **trial em que cada uma atinge um patamar**, não só pelo valor final: chegar ao mesmo lugar com metade dos treinos é o ganho real de um hipertunador.

In [ ]:
# ============================================================================
# HPO AUTOMATIZADO — random search por CNN, segmentador e receita de treino
# ============================================================================
"""Busca de hiperparâmetros reprodutível, sem vazamento do conjunto de teste.

Esta é a **única** definição de `hpo_candidates` e `automated_hpo` do notebook:
o espaço de busca, a geração de candidatos e a avaliação ficam juntos para que
não exista uma segunda versão sombreando esta em função da ordem de execução.

Estratégia: *random search* com cobertura garantida de segmentadores. Enumerar o
produto cartesiano completo do espaço abaixo daria milhares de combinações — a
amostragem aleatória com semente fixa é reprodutível e cobre bem mais eixos pelo
mesmo orçamento de treinos. Os primeiros `len(segmenters)` trials de cada CNN são
forçados a segmentadores distintos, de modo que nenhum algoritmo fique de fora por
azar de sorteio.

A seleção é feita por melhor acurácia de validação; OA/AA/Kappa de teste aparecem
nos relatórios, mas não entram no objetivo de escolha. Para uma publicação final,
reserve o teste para a configuração vencedora e repita a campanha com várias seeds.
"""

from dataclasses import replace

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#: Eixos sorteados a cada trial, além do segmentador e de seus parâmetros internos.
#: `pca_components` não aparece aqui porque depende da cena; é construído por
#: `_pca_options`, que aplica o piso imposto pelos kernels da CNN1D.
HPO_SPACE = {
    "learning_rate": (3e-4, 1e-3, 3e-3),
    "dropout": (0.2, 0.35, 0.5),
    "weight_decay": (0.0, 1e-5, 1e-4),
    "batch_size": (32, 64, 128),
}

#: Tentativas de resorteio antes de aceitar uma configuração repetida.
_HPO_MAX_DRAWS = 32


def _pca_options(dataset, model_type="cnn1d", scales=(0.5, 1.0, 1.5)):
    """Número de componentes de PCA em torno da receita padrão da cena.

    A escala é relativa porque o valor de referência já difere entre cenas
    (Indian Pines usa 30; PaviaU e Salinas, 15).

    O piso vem da **arquitetura**, não da segmentação. A composição RGB usa um PCA
    próprio de 3 componentes; este aqui alimenta as convoluções. A CNN1D encurta a
    sequência em 14 posições, então abaixo de `CNN1D_MIN_COMPONENTS` a saída fica
    vazia. Sem esse piso, cenas com receita 15 (PaviaU, Salinas) geram a opção 8 e
    o treino quebra dentro da primeira convolução.
    """
    base = PCA_COMPONENTS[dataset]
    piso = CNN1D_MIN_COMPONENTS if model_type == "cnn1d" else 3
    return sorted({max(piso, int(round(base * scale))) for scale in scales})


def segmenter_hpo_overrides(segmenter, dataset):
    """Retorna opções controladas em torno da receita padrão de cada segmentador.

    SEEDS varia número de regiões, iterações e prior; SLIC varia region_size/ruler;
    LSC varia region_size/ratio; Watershed varia marcadores. O valor efetivamente
    usado vai para `config.segmenter_overrides` e é gravado no `config.json` do run.
    """
    base = {
        "seeds": SEEDS_PARAMS,
        "slic": SLIC_PARAMS,
        "lsc": LSC_PARAMS,
        "watershed": WATERSHED_PARAMS,
    }[segmenter][dataset]
    if segmenter == "seeds":
        return [
            {"num_superpixels": max(40, int(base.num_superpixels * factor)), "num_iterations": iterations, "prior": prior}
            for factor, iterations, prior in ((.70, 120, 1), (1.0, base.num_iterations, base.prior), (1.35, 250, 3))
        ]
    if segmenter == "slic":
        return [
            {"region_size": max(3, base.region_size - 3), "ruler": 5.0},
            {"region_size": base.region_size, "ruler": base.ruler},
            {"region_size": base.region_size + 3, "ruler": 20.0},
        ]
    if segmenter == "lsc":
        return [
            {"region_size": max(3, base.region_size - 3), "ratio": .04},
            {"region_size": base.region_size, "ratio": base.ratio},
            {"region_size": base.region_size + 3, "ratio": .12},
        ]
    return [{"markers": max(20, int(base.markers * factor))} for factor in (.70, 1.0, 1.35)]


def hpo_search_space(base_config, model_type, space=None):
    """Eixos efetivamente sorteados para esta CNN e esta cena.

    Devolver o espaço explicitamente permite gravá-lo junto da campanha: o leitor
    do TCC vê o que foi buscado, e não apenas o que venceu.
    """
    axes = {name: tuple(values) for name, values in (space or HPO_SPACE).items()}
    axes["pca_components"] = tuple(_pca_options(base_config.dataset, model_type))
    return axes


def hpo_genes(base_config, model_type, space=None, segmenters=None):
    """O espaço de busca como uma lista ordenada de genes discretos.

    Cada gene é `(nome, valores)` e um candidato é uma tupla de índices — um por
    gene. Essa representação é o denominador comum das três estratégias: o random
    search sorteia índices, o TPE os sugere e o algoritmo genético os cruza e muta.
    Todos os eixos são discretos, o que dispensa qualquer codificação real/binária.
    """
    axes = hpo_search_space(base_config, model_type, space)
    segmenters = tuple(segmenters or SEGMENTERS)
    # `segmenter_hpo_overrides` devolve sempre três receitas por segmentador, então
    # o índice do override é um gene retangular, independente do segmentador sorteado.
    genes = [("segmenter", segmenters), ("override_index", (0, 1, 2))]
    genes += [(name, values) for name, values in axes.items()]
    return genes


def hpo_decode(genes, individual, dataset):
    """Traduz uma tupla de índices nos campos que `ExperimentConfig` entende."""
    chosen = {name: values[index % len(values)] for (name, values), index in zip(genes, individual)}
    segmenter = chosen.pop("segmenter")
    overrides = segmenter_hpo_overrides(segmenter, dataset)
    position = chosen.pop("override_index") % len(overrides)
    return {"segmenter": segmenter, "segmenter_overrides": dict(overrides[position]), **chosen}


class _RandomSearcher:
    """Sorteio uniforme com cobertura garantida de segmentadores e sem repetição.

    Não usa o resultado dos trials anteriores — é a linha de base contra a qual as
    duas estratégias guiadas são comparadas.
    """

    name = "random"

    def __init__(self, genes, rng, coverage):
        self.genes, self.rng, self.coverage = genes, rng, list(coverage)
        self.seen = set()

    def _draw(self):
        return tuple(int(self.rng.integers(len(values))) for _, values in self.genes)

    def ask(self):
        forced = self.coverage.pop(0) if self.coverage else None
        individual = self._draw()
        for _ in range(_HPO_MAX_DRAWS):
            candidate = individual if forced is None else (forced,) + individual[1:]
            if candidate not in self.seen:
                break
            individual = self._draw()
        self.seen.add(candidate)
        return candidate

    def tell(self, individual, score):
        """Random search é memoryless: o resultado não altera os sorteios seguintes."""


class _TPESearcher:
    """Hipertunador bayesiano (Tree-structured Parzen Estimator) via Optuna.

    O TPE modela duas densidades sobre o espaço — a dos trials bons e a dos ruins —
    e propõe pontos que maximizam a razão entre elas. Diferente do random search,
    cada sugestão depende de tudo o que já foi observado.

    `segmenter` e `override_index` são declarados como categóricos (não têm ordem
    natural); os demais eixos como inteiros ordenados, porque neles "o próximo
    valor" realmente significa "um pouco maior".
    """

    name = "tpe"

    def __init__(self, genes, seed, coverage):
        import optuna

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        self.genes = genes
        self.study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=int(seed), n_startup_trials=max(4, len(coverage))),
        )
        # Os mesmos trials de cobertura do random search, para que as estratégias
        # partam da mesma informação inicial sobre os segmentadores.
        for index in coverage:
            self.study.enqueue_trial({"segmenter": int(index)}, skip_if_exists=False)
        self._current = None

    def ask(self):
        self._current = self.study.ask()
        indices = []
        for name, values in self.genes:
            if name in ("segmenter", "override_index"):
                indices.append(int(self._current.suggest_categorical(name, list(range(len(values))))))
            else:
                indices.append(int(self._current.suggest_int(name, 0, len(values) - 1)))
        return tuple(indices)

    def tell(self, individual, score):
        self.study.tell(self._current, float(score))


class _GeneticSearcher:
    """Algoritmo genético elitista sobre o espaço discreto de índices.

    Uma geração inteira é avaliada antes de produzir a seguinte. Operadores:

    - **seleção por torneio**: `tournament` indivíduos sorteados, vence o de maior
      validação. Mantém pressão seletiva sem descartar a diversidade de imediato.
    - **crossover uniforme**: cada gene do filho vem de um dos dois pais com
      probabilidade 1/2. Adequado aqui porque os genes são independentes — não há
      vizinhança a preservar, como haveria numa codificação binária de um número.
    - **mutação por gene**: com probabilidade `mutation_rate`, um gene é resorteado.
      É o que impede a convergência prematura para um único segmentador.
    - **elitismo**: os `elite` melhores passam à geração seguinte **sem retreinar**,
      o que economiza orçamento e garante que o melhor nunca piore.
    """

    name = "genetic"

    def __init__(self, genes, rng, coverage, population=8, elite=2, mutation_rate=.25, tournament=3):
        self.genes, self.rng = genes, rng
        self.population_size = max(4, int(population))
        self.elite = max(1, min(int(elite), self.population_size - 1))
        self.mutation_rate, self.tournament = float(mutation_rate), max(2, int(tournament))
        self.generation, self.previous = [], []
        self.queue = [(index,) + self._draw()[1:] for index in coverage]
        while len(self.queue) < self.population_size:
            self.queue.append(self._draw())

    def _draw(self):
        return tuple(int(self.rng.integers(len(values))) for _, values in self.genes)

    def _tournament(self):
        pool = [self.previous[int(self.rng.integers(len(self.previous)))] for _ in range(self.tournament)]
        return max(pool, key=lambda item: item[0])[1]

    def _breed(self):
        ranked = sorted(self.generation, key=lambda item: item[0], reverse=True)
        self.previous = ranked
        elites = ranked[:self.elite]
        # Elites entram na próxima geração já avaliados: não voltam para a fila.
        self.generation = list(elites)
        children = []
        while len(children) < self.population_size - len(elites):
            first, second = self._tournament(), self._tournament()
            child = tuple(
                first[position] if self.rng.random() < .5 else second[position]
                for position in range(len(self.genes))
            )
            child = tuple(
                int(self.rng.integers(len(values))) if self.rng.random() < self.mutation_rate else gene
                for gene, (_, values) in zip(child, self.genes)
            )
            children.append(child)
        return children

    def ask(self):
        if not self.queue:
            self.queue = self._breed()
        return self.queue.pop(0)

    def tell(self, individual, score):
        self.generation.append((float(score), individual))


#: Estratégias disponíveis. "random" é a linha de base; "tpe" é o hipertunador
#: bayesiano; "genetic" é o algoritmo evolutivo.
HPO_STRATEGIES = ("random", "tpe", "genetic")


def make_hpo_searcher(strategy, genes, segmenters, seed, model_type, population=8):
    """Instancia a estratégia de busca, dando a todas a mesma cobertura inicial."""
    coverage = list(range(len(tuple(segmenters))))
    offset = MODEL_TYPES.index(model_type)
    rng = np.random.default_rng(seed + offset)
    if strategy == "random":
        return _RandomSearcher(genes, rng, coverage)
    if strategy == "tpe":
        return _TPESearcher(genes, seed + offset, coverage)
    if strategy == "genetic":
        return _GeneticSearcher(genes, rng, coverage, population=population)
    raise ValueError(f"estratégia inválida: {strategy!r}. Use uma de {HPO_STRATEGIES}.")


def hpo_candidates(base_config, model_type, segmenters, trials, seed=42, space=None):
    """Pré-gera os candidatos do random search, para inspecionar o plano da busca.

    Só existe para a estratégia aleatória: TPE e algoritmo genético são adaptativos,
    e o candidato `n+1` depende do resultado do candidato `n`.
    """
    genes = hpo_genes(base_config, model_type, space, segmenters)
    searcher = make_hpo_searcher("random", genes, segmenters, seed, model_type)
    return [
        replace(base_config, model_type=model_type,
                **hpo_decode(genes, searcher.ask(), base_config.dataset))
        for _ in range(max(trials, len(tuple(segmenters))))
    ]


def automated_hpo(base_config, trials_per_cnn=8, search_epochs=10, backend="thread",
                  model_types=MODEL_TYPES, segmenters=None, space=None,
                  strategy="random", population=8, retain="winner"):
    """Executa HPO por arquitetura e devolve candidatos, vencedores e registro auditável.

    `strategy` escolhe entre as três buscas — todas percorrem o mesmo espaço, com o
    mesmo orçamento de trials e a mesma cobertura inicial de segmentadores, o que
    torna a comparação entre elas honesta.

    `search_epochs` limita o custo da busca. Os vencedores podem depois ser
    retreinados pelo orçamento completo de `base_config.epochs` antes da comparação
    final. O score de seleção é o maior `val_accuracy` observado no histórico; o
    teste nunca participa da escolha.

    `retain` controla a memória, e não é um detalhe: cada `ExperimentResult` segura
    sua própria cópia dos dados preparados.

    - ``"winner"`` (padrão): mantém só o melhor de cada arquitetura e libera os
      demais assim que a métrica é extraída. É o que torna a campanha viável em
      cenas maiores.
    - ``"all"``: mantém tudo. Só use com CNN1D em cenas pequenas, ou para inspeção.

    A tabela devolvida carrega sempre a `config` de cada trial — que é barata e
    reconstrói qualquer execução —, então liberar o objeto não perde rastreabilidade.
    """
    if retain not in {"winner", "all"}:
        raise ValueError("retain deve ser 'winner' ou 'all'.")
    segmenters = tuple(segmenters or SEGMENTERS)
    trials_per_cnn = max(trials_per_cnn, len(segmenters))
    rows, winners, winner_trials = [], {}, {}
    for model_type in model_types:
        genes = hpo_genes(base_config, model_type, space, segmenters)
        searcher = make_hpo_searcher(strategy, genes, segmenters, base_config.seed, model_type, population)
        best = None
        for trial in range(1, trials_per_cnn + 1):
            individual = searcher.ask()
            candidate = replace(base_config, model_type=model_type,
                                **hpo_decode(genes, individual, base_config.dataset))
            trial_config = replace(
                candidate,
                epochs=search_epochs,
                patience=min(candidate.patience, max(2, search_epochs // 3)),
            )
            result = run_experiment(trial_config, backend=backend, verbose=False)
            validation_score = float(max(result.training.history.val_accuracy))
            searcher.tell(individual, validation_score)
            # As métricas são extraídas ANTES de qualquer liberação: são escalares
            # e arrays pequenos, e é tudo o que os relatórios consomem.
            rows.append({
                "CNN": model_type.upper(), "estratégia": strategy, "trial": trial,
                "segmentador": trial_config.segmenter,
                "parâmetros segmentador": json.dumps(trial_config.segmenter_overrides, sort_keys=True),
                "learning rate": trial_config.learning_rate, "dropout": trial_config.dropout,
                "weight decay": trial_config.weight_decay, "batch": trial_config.batch_size,
                "PCA": trial_config.pca_components,
                "val_accuracy (objetivo)": validation_score,
                "OA teste (relato)": result.metrics.overall_accuracy,
                "AA teste (relato)": result.metrics.average_accuracy,
                "Kappa teste (relato)": result.metrics.kappa,
                "tempo treino (s)": result.training.total_time,
                "épocas executadas": len(result.training.history),
                "parâmetros do modelo": int(result.model.n_parameters),
                "config": trial_config,
            })
            if best is None or validation_score > best[0]:
                if best is not None and retain != "all":
                    release_result(best[1])          # o campeão anterior perdeu o posto
                best = (validation_score, result, trial)
            elif retain != "all":
                release_result(result)               # não é o melhor: libera agora
            rows[-1]["melhor até agora"] = best[0]
            if retain == "all":
                rows[-1]["resultado"] = result
        winners[model_type] = best[1]
        winner_trials[model_type.upper()] = best[2]
    table = pd.DataFrame(rows)
    # O vencedor é marcado por (CNN, trial), e não pela identidade do objeto: com
    # `retain="winner"` os demais resultados já foram liberados.
    table["vencedor"] = [
        winner_trials.get(cnn) == trial for cnn, trial in zip(table["CNN"], table["trial"])
    ]
    return table, winners


def compare_hpo_strategies(base_config, trials_per_cnn=12, search_epochs=10, backend="thread",
                           model_types=("cnn1d",), segmenters=None, space=None,
                           strategies=HPO_STRATEGIES, population=6):
    """Roda as três estratégias com orçamento idêntico e devolve uma tabela única.

    É o experimento que responde "vale a pena um hipertunador?". Todas partem da
    mesma semente, do mesmo espaço e do mesmo número de treinos; a única diferença
    é como cada trial é escolhido. Compare pela curva de melhor-até-agora, e não só
    pelo valor final: uma busca que chega ao mesmo lugar em metade dos trials é
    melhor mesmo empatando no fim.
    """
    frames = []
    for strategy in strategies:
        table, _ = automated_hpo(
            base_config, trials_per_cnn=trials_per_cnn, search_epochs=search_epochs,
            backend=backend, model_types=model_types, segmenters=segmenters,
            space=space, strategy=strategy, population=population,
        )
        frames.append(table)
    return pd.concat(frames, ignore_index=True)


def plot_hpo_strategies(comparison):
    """Convergência, custo e escolhas de cada estratégia de busca."""
    import seaborn as sns

    fig, axes = plt.subplots(1, 3, figsize=(19, 5.2), constrained_layout=True)
    colors = dict(zip(HPO_STRATEGIES, ("#4C78A8", "#F58518", "#54A24B")))
    for (strategy, cnn), group in comparison.groupby(["estratégia", "CNN"]):
        ordered = group.sort_values("trial")
        axes[0].plot(ordered["trial"], ordered["melhor até agora"], marker="o", ms=4,
                     color=colors.get(strategy), label=f"{strategy} · {cnn}")
        axes[1].scatter(ordered["trial"], ordered["val_accuracy (objetivo)"],
                        color=colors.get(strategy), s=45, alpha=.75, label=f"{strategy} · {cnn}")
    axes[0].set(title="Convergência: melhor validação até o trial n", xlabel="Trial",
                ylabel="melhor val_accuracy")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=.25)
    axes[1].set(title="Validação de cada trial individual", xlabel="Trial", ylabel="val_accuracy")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=.25)
    usage = comparison.groupby(["estratégia", "segmentador"]).size().reset_index(name="trials")
    sns.barplot(data=usage, x="estratégia", y="trials", hue="segmentador", ax=axes[2])
    axes[2].set(title="Onde cada estratégia gastou o orçamento", ylabel="trials")
    axes[2].grid(axis="y", alpha=.25)
    fig.suptitle("Random search × TPE (bayesiano) × algoritmo genético — mesmo espaço, mesmo orçamento",
                 fontsize=13)
    return fig


def hpo_strategy_summary(comparison):
    """Melhor validação, trial em que ela apareceu e custo total, por estratégia."""
    rows = []
    for (strategy, cnn), group in comparison.groupby(["estratégia", "CNN"]):
        best = group.loc[group["val_accuracy (objetivo)"].idxmax()]
        rows.append({
            "estratégia": strategy, "CNN": cnn,
            "melhor val_accuracy": float(best["val_accuracy (objetivo)"]),
            "encontrado no trial": int(best["trial"]),
            "trials": int(len(group)),
            "tempo total (s)": float(group["tempo treino (s)"].sum()),
            "segmentador vencedor": best["segmentador"],
            "OA teste (relato)": float(best["OA teste (relato)"]),
        })
    return pd.DataFrame(rows).sort_values(["CNN", "melhor val_accuracy"], ascending=[True, False])


#: Eixos mostrados por `plot_hpo_space`, com o rótulo usado no gráfico.
HPO_PLOT_AXES = (
    ("learning rate", "log"), ("dropout", "linear"), ("weight decay", "symlog"),
    ("batch", "linear"), ("PCA", "linear"),
)


def plot_hpo_space(hpo_table):
    """Mostra a validação obtida em cada valor de cada eixo do espaço de busca.

    É o gráfico que responde "o que a busca de fato variou e o que importou":
    um eixo cujo valor não muda a validação é candidato a sair do espaço na
    próxima campanha, liberando orçamento para os eixos que importam.
    """
    import seaborn as sns

    fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
    for axis, (column, scale) in zip(axes.ravel(), HPO_PLOT_AXES):
        if column not in hpo_table or hpo_table[column].nunique() < 1:
            axis.set_axis_off()
            continue
        sns.stripplot(data=hpo_table, x=column, y="val_accuracy (objetivo)", hue="CNN",
                      dodge=True, jitter=.18, s=8, ax=axis)
        axis.set(title=f"Validação × {column}", ylim=(0, 1), ylabel="melhor val_accuracy")
        axis.grid(axis="y", alpha=.25)
        axis.legend_.remove() if axis.legend_ else None
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, title="CNN", loc="outside right upper")
    fig.suptitle("Espaço de busca do random search — um painel por eixo", fontsize=13)
    return fig


def plot_hpo_by_cnn(hpo_table, model_type):
    """Mostra candidatos, segmentadores e score de validação de uma única CNN."""
    subset = hpo_table[hpo_table["CNN"] == model_type.upper()].copy().sort_values("trial")
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
    colors = {name: color for name, color in zip(SEGMENTERS, plt.cm.tab10.colors)}
    axes[0].bar(subset["trial"], subset["val_accuracy (objetivo)"], color=[colors[name] for name in subset["segmentador"]])
    for _, row in subset.iterrows():
        marker = "★" if row["vencedor"] else ""
        axes[0].text(row["trial"], row["val_accuracy (objetivo)"], marker, ha="center", va="bottom", fontsize=14)
    axes[0].set(title=f"HPO — {model_type.upper()} (seleção por validação)", xlabel="Trial", ylabel="Melhor val_accuracy", ylim=(0, 1))
    axes[0].grid(axis="y", alpha=.25)
    for segmenter, group in subset.groupby("segmentador"):
        axes[1].scatter(group["tempo treino (s)"], group["val_accuracy (objetivo)"],
                        label=segmenter, s=100, color=colors[segmenter], edgecolor="black")
    axes[1].set(title="Qualidade de validação × custo", xlabel="Tempo de treino (s)", ylabel="Melhor val_accuracy", ylim=(0, 1))
    axes[1].legend(title="Segmentador"); axes[1].grid(alpha=.25)
    return fig, subset.drop(columns="resultado", errors="ignore")


def plot_hpo_compilation(hpo_table):
    """Compila HPO das três CNNs, segmentadores e vencedores em uma única figura."""
    summary = hpo_table.groupby("CNN", as_index=False).apply(
        lambda group: group.loc[group["val_accuracy (objetivo)"].idxmax()], include_groups=False
    ).reset_index(drop=True)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)
    for cnn, group in hpo_table.groupby("CNN"):
        axes[0].scatter(group["trial"], group["val_accuracy (objetivo)"], label=cnn, s=85)
    axes[0].set(title="Busca por arquitetura", xlabel="Trial", ylabel="val_accuracy", ylim=(0, 1))
    axes[0].legend(); axes[0].grid(alpha=.25)
    pivot = hpo_table.pivot_table(index="CNN", columns="segmentador", values="val_accuracy (objetivo)", aggfunc="max")
    pivot.plot.bar(ax=axes[1], ylim=(0, 1), rot=0, title="Melhor validação por segmentador")
    axes[1].set_ylabel("val_accuracy"); axes[1].grid(axis="y", alpha=.25)
    axes[2].bar(summary["CNN"], summary["OA teste (relato)"], label="OA teste")
    axes[2].bar(summary["CNN"], summary["AA teste (relato)"], alpha=.55, label="AA teste")
    axes[2].set(title="Vencedores HPO — relatório de teste", ylabel="Pontuação", ylim=(0, 1))
    axes[2].legend(); axes[2].grid(axis="y", alpha=.25)
    return fig, summary


### 5.5 Campanha MLOps auditável

O HPO exploratório sugere candidatos; esta campanha os repete com várias seeds e registra cada tentativa. O ranking usa validação média e IC95%; OA/AA/Kappa de teste permanecem no relatório para evitar a escolha indireta pelo teste.

In [ ]:
# ============================================================================
# CAMPANHA MLOPS AUDITÁVEL — plano, repetições, artefatos e análise automática
# ============================================================================
"""Campanhas reprodutíveis para comparar modelos em vez de olhar uma única execução.

Cada tentativa é um run normal do tracker e grava configuração, ambiente,
fingerprint dos dados, métricas, histórico, checkpoint e status. A campanha agrega
esses runs por uma receita explícita, preserva o plano original, atualiza resultados
parciais após cada tentativa e calcula estatísticas somente a partir dos artefatos.
"""

from dataclasses import replace
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from uuid import uuid4

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _campaign_json(path: Path, value):
    """Grava JSON ordenado e legível, usando escrita atômica por substituição."""
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary.replace(path)


def build_campaign_plan(base_config, trials_per_cnn=4, seeds=(11, 29, 42),
                        search_epochs=10, model_types=MODEL_TYPES,
                        segmenters=None):
    """Cria a matriz auditável: candidatos por CNN × repetições por seed.

    A função reutiliza o espaço de busca do HPO e garante cobertura de cada
    segmentador. A seed é parte da receita e, portanto, aparece tanto no plano
    quanto em `config.json` de todo run. A ordem é determinística para permitir
    retomada ou comparação entre campanhas.
    """
    segmenters = tuple(segmenters or SEGMENTERS)
    trials_per_cnn = max(trials_per_cnn, len(segmenters))
    recipes = []
    for model_type in model_types:
        candidates = hpo_candidates(base_config, model_type, segmenters, trials_per_cnn, base_config.seed)
        for candidate_index, candidate in enumerate(candidates, start=1):
            for repeat, seed in enumerate(seeds, start=1):
                config = replace(
                    candidate, seed=int(seed), epochs=search_epochs,
                    patience=min(candidate.patience, max(2, search_epochs // 3)),
                )
                recipes.append({
                    "trial_key": f"{model_type}_c{candidate_index:02d}_s{seed}",
                    "model_type": model_type,
                    "candidate_index": candidate_index,
                    "repeat": repeat,
                    "config": config.to_dict(),
                })
    return recipes


def run_audited_experiment(config, campaign_dir: Path, campaign_id: str, trial_key: str, backend="thread"):
    """Executa uma receita e torna sucesso ou falha permanentemente auditável.

    Mesmo uma exceção gera `failure.json`, é vinculada ao `trial_key` e permanece
    no resultado parcial da campanha. Assim, remover uma falha para melhorar um
    resultado não é possível sem deixar alteração explícita no plano/manifesto.
    """
    context = create_run(config)
    run_link = {
        "campaign_id": campaign_id,
        "campaign_directory": str(campaign_dir.resolve()),
        "trial_key": trial_key,
    }
    _campaign_json(context.directory / "campaign.json", run_link)
    try:
        result = run_experiment(config, backend=backend, verbose=False)
        checkpoint = context.directory / "model.pt"
        save_checkpoint(result.model, config, result.data.input_shape(config.model_type), result.data.n_classes, checkpoint)
        record_run(context, result, artifacts={"checkpoint": checkpoint, "campaign": context.directory / "campaign.json"})
        pdf_report = generate_pdf_report(context.directory, f"Run {context.run_id}", result=result)
        manifest_path = context.directory / "manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest["artifacts"]["pdf_report"] = pdf_report.name
        _campaign_json(manifest_path, manifest)
        return {
            "trial_key": trial_key, "run_id": context.run_id, "status": "completed",
            "run_directory": str(context.directory.resolve()),
            "model_type": config.model_type, "segmenter": config.segmenter,
            "learning_rate": config.learning_rate, "dropout": config.dropout, "seed": config.seed,
            # Critério de seleção: validação. Métricas de teste são exclusivamente relatadas.
            "validation_best": float(max(result.training.history.val_accuracy)),
            "oa_test": result.metrics.overall_accuracy, "aa_test": result.metrics.average_accuracy,
            "kappa_test": result.metrics.kappa, "test_loss": result.metrics.test_loss,
            "train_seconds": result.training.total_time, "prep_seconds": result.prep_timings.wall,
            "checkpoint": str(checkpoint.resolve()), "pdf_report": str(pdf_report.resolve()), "result": result,
        }
    except BaseException as error:
        fail_run(context, error)
        return {
            "trial_key": trial_key, "run_id": context.run_id, "status": "failed",
            "run_directory": str(context.directory.resolve()),
            "model_type": config.model_type, "segmenter": config.segmenter,
            "learning_rate": config.learning_rate, "dropout": config.dropout, "seed": config.seed,
            "error_type": type(error).__name__, "error": str(error), "result": None,
        }


def _serializable_records(records):
    """Remove objetos PyTorch do relatório de campanha sem perder links aos artefatos."""
    return [{key: value for key, value in record.items() if key != "result"} for record in records]


def run_audited_campaign(base_config, trials_per_cnn=4, seeds=(11, 29, 42),
                         search_epochs=10, backend="thread", campaign_root=None):
    """Executa uma campanha completa e persiste o estado após *cada* tentativa.

    Arquivos produzidos em `outputs/campaigns/<id>/`:
    `campaign.json` (intenção e status), `plan.json` (matriz imutável),
    `results.json` (resultado parcial/final), `summary.csv`, `analysis.json` e
    `report.md`. Cada linha aponta para um run individual em `outputs/runs/`.
    """
    started = datetime.now(timezone.utc)
    campaign_id = f"campaign_{started.strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"
    directory = Path(campaign_root or output_dir() / "campaigns") / campaign_id
    directory.mkdir(parents=True, exist_ok=False)
    plan = build_campaign_plan(base_config, trials_per_cnn, seeds, search_epochs)
    plan_hash = sha256(json.dumps(plan, sort_keys=True).encode("utf-8")).hexdigest()
    campaign = {
        "schema_version": 1, "campaign_id": campaign_id, "status": "running",
        "started_at_utc": started.isoformat(), "backend": backend,
        "selection_metric": "validation_best", "test_policy": "report_only",
        "base_config": base_config.to_dict(), "plan_sha256": plan_hash,
        "planned_trials": len(plan), "seeds": list(seeds),
    }
    _campaign_json(directory / "campaign.json", campaign)
    _campaign_json(directory / "plan.json", plan)
    records = []
    for recipe in plan:
        config = ExperimentConfig(**recipe["config"])
        record = run_audited_experiment(config, directory, campaign_id, recipe["trial_key"], backend)
        records.append(record)
        _campaign_json(directory / "results.json", _serializable_records(records))
    frame = pd.DataFrame(_serializable_records(records))
    analysis = analyze_campaign(frame)
    frame.to_csv(directory / "summary.csv", index=False)
    _campaign_json(directory / "analysis.json", analysis["json"])
    (directory / "report.md").write_text(campaign_report(campaign, frame, analysis), encoding="utf-8")
    campaign.update({
        "status": "completed", "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "completed_trials": int((frame["status"] == "completed").sum()),
        "failed_trials": int((frame["status"] == "failed").sum()),
        "artifacts": ["plan.json", "results.json", "summary.csv", "analysis.json", "report.md", "report.pdf"],
    })
    _campaign_json(directory / "campaign.json", campaign)
    pdf_report = generate_pdf_report(directory, f"Campanha {campaign_id}")
    global_pdf, _, _ = generate_all_runs_pdf()
    return {"campaign": campaign, "directory": directory, "records": frame, "objects": {r["run_id"]: r["result"] for r in records if r["result"] is not None}, "analysis": analysis, "pdf_report": pdf_report, "global_pdf_report": global_pdf}


def _bootstrap_interval(values, seed=42, samples=2000):
    """Intervalo bootstrap de 95% da média, explícito mesmo com poucas seeds."""
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        return float(values[0]), float(values[0])
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(samples, len(values)), replace=True).mean(axis=1)
    return tuple(map(float, np.quantile(means, (.025, .975))))


def analyze_campaign(records: pd.DataFrame):
    """Agrega repetições por receita e produz ranking apenas pela validação.

    O dataframe por receita contém média, desvio e IC95% de validação, além das
    métricas de teste somente para contextualização. `run_ids` preserva exatamente
    quais execuções contribuíram para cada número agregado.
    """
    completed = records[records["status"] == "completed"].copy()
    if completed.empty:
        return {"recipes": pd.DataFrame(), "models": pd.DataFrame(), "json": {"completed": 0}}
    groups = ["model_type", "segmenter", "learning_rate", "dropout"]
    rows = []
    for key, group in completed.groupby(groups, sort=True):
        ci_low, ci_high = _bootstrap_interval(group["validation_best"], seed=42)
        rows.append({
            **dict(zip(groups, key)), "n_execuções": len(group),
            "validação média": group["validation_best"].mean(), "validação desvio": group["validation_best"].std(ddof=0),
            "validação ic95 baixo": ci_low, "validação ic95 alto": ci_high,
            "OA teste média": group["oa_test"].mean(), "AA teste média": group["aa_test"].mean(),
            "Kappa teste média": group["kappa_test"].mean(), "treino médio (s)": group["train_seconds"].mean(),
            "run_ids": list(group["run_id"]),
        })
    recipes = pd.DataFrame(rows).sort_values(["validação média", "validação desvio"], ascending=[False, True]).reset_index(drop=True)
    recipes.index += 1
    recipes.index.name = "ranking por validação"
    model_rows = []
    for model_type, group in completed.groupby("model_type", sort=True):
        low, high = _bootstrap_interval(group["validation_best"], seed=42)
        model_rows.append({
            "CNN": model_type.upper(), "n_execuções": len(group),
            "validação média": group["validation_best"].mean(), "validação desvio": group["validation_best"].std(ddof=0),
            "validação ic95 baixo": low, "validação ic95 alto": high,
            "OA teste média": group["oa_test"].mean(), "AA teste média": group["aa_test"].mean(),
            "Kappa teste média": group["kappa_test"].mean(), "treino médio (s)": group["train_seconds"].mean(),
        })
    models = pd.DataFrame(model_rows).sort_values("validação média", ascending=False).reset_index(drop=True)
    return {
        "recipes": recipes, "models": models,
        "json": {
            "completed": int(len(completed)), "failed": int((records["status"] == "failed").sum()),
            "selection_metric": "validation_best", "bootstrap": {"seed": 42, "samples": 2000, "confidence": .95},
            "ranking": json.loads(recipes.to_json(orient="records")),
            "models": json.loads(models.to_json(orient="records")),
        },
    }


def plot_campaign_analysis(records: pd.DataFrame, analysis):
    """Mostra repetição por seed, intervalos, segmentadores e custo da campanha."""
    completed = records[records["status"] == "completed"].copy()
    if completed.empty:
        print("Campanha sem runs concluídos para analisar.")
        return None
    recipes, models = analysis["recipes"], analysis["models"]
    fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
    for model_type, group in completed.groupby("model_type"):
        axes[0, 0].scatter(group["seed"], group["validation_best"], s=75, label=model_type.upper())
    axes[0, 0].set(title="Repetições: validação por seed", xlabel="Seed", ylabel="Melhor val_accuracy", ylim=(0, 1))
    axes[0, 0].legend(); axes[0, 0].grid(alpha=.25)
    positions = np.arange(len(models))
    means = models["validação média"].to_numpy()
    errors = np.vstack((means - models["validação ic95 baixo"], models["validação ic95 alto"] - means))
    axes[0, 1].bar(positions, means, yerr=errors, capsize=5, color="#4C78A8")
    axes[0, 1].set(title="Média por CNN com IC95% bootstrap", xticks=positions, xticklabels=models["CNN"], ylabel="val_accuracy", ylim=(0, 1))
    axes[0, 1].grid(axis="y", alpha=.25)
    pivot = completed.pivot_table(index="model_type", columns="segmenter", values="validation_best", aggfunc="mean")
    image = axes[1, 0].imshow(pivot, aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
    axes[1, 0].set(title="Validação média: CNN × segmentador", yticks=range(len(pivot.index)), yticklabels=[item.upper() for item in pivot.index], xticks=range(len(pivot.columns)), xticklabels=pivot.columns)
    fig.colorbar(image, ax=axes[1, 0], label="val_accuracy")
    scatter = axes[1, 1].scatter(recipes["treino médio (s)"], recipes["validação média"], c=recipes["OA teste média"], cmap="viridis", s=100, edgecolor="black")
    for rank, row in recipes.head(min(8, len(recipes))).iterrows():
        axes[1, 1].annotate(f"#{rank} {row['model_type'].upper()}/{row['segmenter']}", (row["treino médio (s)"], row["validação média"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    axes[1, 1].set(title="Receitas: qualidade de validação × custo", xlabel="Treino médio (s)", ylabel="val_accuracy", ylim=(0, 1))
    axes[1, 1].grid(alpha=.25); fig.colorbar(scatter, ax=axes[1, 1], label="OA teste (relato)")
    return fig


def campaign_report(campaign, records: pd.DataFrame, analysis) -> str:
    """Gera relatório Markdown que aponta para os run_ids e documenta a política."""
    lines = [
        "# Relatório de campanha MLOps", "",
        f"- Campanha: `{campaign['campaign_id']}`", f"- Início: {campaign['started_at_utc']}",
        f"- Plano SHA-256: `{campaign['plan_sha256']}`", "- Seleção: maior `validation_best`; teste é somente relatório.",
        f"- Runs concluídos: {(records['status'] == 'completed').sum()} | falhos: {(records['status'] == 'failed').sum()}", "",
        "## Ranking de receitas", "",
    ]
    recipes = analysis["recipes"]
    if recipes.empty:
        return "\n".join(lines + ["Nenhum run concluído.", ""])
    lines.extend(["| rank | CNN | segmentador | lr | dropout | n | val média | IC95% | OA teste | runs |", "|---:|---|---|---:|---:|---:|---:|---|---:|---|"])
    for rank, row in recipes.iterrows():
        lines.append(
            f"| {rank} | {row['model_type'].upper()} | {row['segmenter']} | {row['learning_rate']:.0e} | {row['dropout']:.2f} | {row['n_execuções']} | "
            f"{row['validação média']:.4f} | [{row['validação ic95 baixo']:.4f}, {row['validação ic95 alto']:.4f}] | {row['OA teste média']:.4f} | {', '.join(row['run_ids'])} |"
        )
    return "\n".join(lines) + "\n"


In [ ]:
# ============================================================================
# RELATÓRIOS PDF — documentação automática de runs e campanhas
# ============================================================================
"""Geração portátil de PDFs somente com Matplotlib, já usado pelo notebook.

O PDF reúne as curvas/métricas disponíveis e todos os documentos textuais ou
estruturados do diretório auditado. Checkpoints e outras saídas binárias entram no
inventário, preservando a ligação entre o relatório e cada artefato gerado.
"""

from matplotlib.backends.backend_pdf import PdfPages
from textwrap import wrap


def _pdf_text(pdf, title, content, lines_per_page=52):
    """Acrescenta texto longo ao PDF em páginas legíveis, sem truncar documentos."""
    lines = []
    for raw_line in str(content).splitlines() or [""]:
        lines.extend(wrap(raw_line, width=115, replace_whitespace=False) or [""])
    for offset in range(0, len(lines), lines_per_page):
        figure = plt.figure(figsize=(11.69, 8.27))  # A4 horizontal
        figure.text(.04, .95, title, fontsize=14, fontweight="bold", va="top")
        figure.text(.04, .90, "\n".join(lines[offset:offset + lines_per_page]), family="monospace", fontsize=7.5, va="top")
        pdf.savefig(figure, bbox_inches="tight")
        plt.close(figure)


def _pdf_inventory(directory):
    """Lista todos os artefatos e separa documentos transcritíveis de binários."""
    root = Path(directory)
    documents, binaries = [], []
    for item in sorted(root.rglob("*")):
        if not item.is_file() or item.name == "report.pdf":
            continue
        relative = item.relative_to(root)
        entry = f"{relative} ({item.stat().st_size:,} bytes)"
        if item.suffix.lower() in {".json", ".csv", ".md", ".txt"}:
            documents.append((item, entry))
        else:
            binaries.append(entry)
    return documents, binaries


def generate_pdf_report(directory, title, result=None):
    """Cria ``report.pdf`` a partir de todos os documentos de uma execução/campanha.

    ``result`` é opcional. Quando existe, o relatório ganha curvas de loss e
    acurácia, além de OA, AA e Kappa. Sem resultado (caso de campanha), os
    documentos agregados continuam sendo incluídos integralmente.
    """
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / "report.pdf"
    documents, binaries = _pdf_inventory(directory)
    with PdfPages(destination) as pdf:
        cover = plt.figure(figsize=(11.69, 8.27))
        cover.text(.06, .88, title, fontsize=22, fontweight="bold")
        cover.text(.06, .78, "Relatório MLOps gerado automaticamente\n"
                   f"Diretório: {directory.resolve()}\n"
                   f"Documentos transcritos: {len(documents)}\n"
                   f"Artefatos binários inventariados: {len(binaries)}", fontsize=12)
        pdf.savefig(cover, bbox_inches="tight")
        plt.close(cover)

        if result is not None:
            history = result.training.history
            figure, axes = plt.subplots(1, 2, figsize=(11.69, 5.2), constrained_layout=True)
            axes[0].plot(history.epochs, history.loss, label="treino")
            axes[0].plot(history.epochs, history.val_loss, label="validação")
            axes[0].set(title="Loss", xlabel="Época", ylabel="Cross-entropy")
            axes[1].plot(history.epochs, history.accuracy, label="treino")
            axes[1].plot(history.epochs, history.val_accuracy, label="validação")
            axes[1].set(title="Acurácia", xlabel="Época", ylabel="Pontuação", ylim=(0, 1))
            for axis in axes:
                axis.legend(); axis.grid(alpha=.25)
            pdf.savefig(figure)
            plt.close(figure)
            _pdf_text(pdf, "Métricas e decisão de treino", json.dumps(result_metrics(result), ensure_ascii=False, indent=2))

        _pdf_text(pdf, "Inventário de artefatos", "Documentos:\n" + "\n".join(entry for _, entry in documents) + "\n\nBinários:\n" + "\n".join(binaries))
        for document, entry in documents:
            try:
                _pdf_text(pdf, f"Documento: {entry}", document.read_text(encoding="utf-8"))
            except UnicodeDecodeError:
                _pdf_text(pdf, f"Documento não textual: {entry}", "Não foi possível decodificar como UTF-8.")
    return destination


def generate_standalone_pdf_report(result):
    """Audita a execução simples da seção 8 em um diretório de relatório próprio."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    directory = output_dir() / "reports" / f"run_{stamp}_{uuid4().hex[:8]}"
    directory.mkdir(parents=True, exist_ok=False)
    (directory / "config.json").write_text(json.dumps(result.config.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
    (directory / "metrics.json").write_text(json.dumps(result_metrics(result), ensure_ascii=False, indent=2), encoding="utf-8")
    (directory / "report.md").write_text("# Execução individual\n\nRelatório automático da seção 8.\n", encoding="utf-8")
    return generate_pdf_report(directory, f"Execução individual {stamp}", result=result)


In [ ]:
# ============================================================================
# OBSERVABILIDADE GLOBAL — todos os runs auditados
# ============================================================================
"""Painéis e PDF consolidados derivados somente dos artefatos persistidos.

Não há recomputação de métricas nem leitura dos tensores do modelo: cada ponto dos
gráficos vem de ``config.json``, ``metrics.json`` e ``manifest.json`` de um run.
Isso mantém o relatório global reproduzível e adequado para auditoria MLOps.
"""

ALL_RUN_PLOT_NOTES = {
    "quality": ("Qualidade por arquitetura", "OA, AA e Kappa de todos os metrics.json. Compare a distribuição: caixas altas e estreitas indicam qualidade alta e estável; AA é mais segura em classes desbalanceadas."),
    "segmenter": ("AA por CNN e segmentador", "Média de AA agrupada por model_type e segmenter. A cor representa desempenho médio, não causalidade; grupos com poucos runs exigem cautela."),
    "pareto": ("Fronteira qualidade × custo", "AA e tempo de treino de cada run. Pontos no alto à esquerda são preferíveis: maior qualidade com menor custo. Cor identifica a CNN e tamanho representa batch físico."),
    "times": ("Distribuição de custos", "Tempos de treino e pré-processamento gravados em metrics.json. Use para identificar variabilidade entre seeds e custo relativo das arquiteturas."),
    "hyperparameters": ("Efeito de hiperparâmetros", "Melhor val_accuracy do histórico versus learning rate; tamanho do ponto é dropout. A métrica de validação é usada porque teste não participa de seleção."),
    "early": ("Convergência e early stopping", "Épocas realmente executadas versus melhor validação. Cor informa se houve parada antecipada; a linha/meta vem da configuração de cada run."),
    "batch": ("Batch × qualidade", "Batch físico de config.json versus validação e AA. A dispersão revela se batches maiores alteraram a qualidade; throughput deve ser lido no sweep específico."),
    "timeline": ("Evolução temporal", "Runs ordenados pelo timestamp do ID. Serve para detectar regressões após mudanças de receita; não substitui comparação controlada."),
    "classes": ("Acurácia por classe", "Média de per_class_accuracy por CNN, apenas para runs com a mesma quantidade de classes. Tons altos indicam classes mais bem reconhecidas."),
    "gaps": ("Consistência entre métricas", "OA e AA do mesmo run. A diagonal representa igualdade; OA muito maior que AA pode indicar classes frequentes mascarando classes raras."),
    "coverage": ("Cobertura experimental", "Cada barra é a quantidade de runs de uma combinação CNN/segmentador; repetição melhora a evidência."),
    "training": ("Evolução de todos os treinos", "Linhas claras são runs individuais; linha forte é média por CNN e faixa é desvio padrão. Validação deve subir enquanto a loss cai."),
    "validation_test": ("Validação × teste", "Cada ponto compara a melhor validação da receita com AA do teste. A diagonal é referência, não objetivo de seleção: o teste continua reservado para relato."),
    "stopping": ("Meta e early stopping", "Barras mostram a proporção de runs que atingiu a meta de validação e que encerrou antecipadamente. Elas revelam a eficiência do orçamento de épocas."),
    "seeds": ("Repetibilidade por seed", "Pontos mostram AA por seed; caixas mostram dispersão por arquitetura. Menor dispersão sugere resultado mais estável, desde que haja repetição suficiente."),
}


def load_all_audited_runs(runs_root=None):
    """Lê todos os runs concluídos diretamente de seus documentos auditáveis."""
    rows = []
    root = Path(runs_root or output_dir() / "runs")
    for directory in sorted(root.glob("*")):
        if not directory.is_dir():
            continue
        config_path, metrics_path, manifest_path = directory / "config.json", directory / "metrics.json", directory / "manifest.json"
        if not config_path.exists() or not metrics_path.exists():
            continue
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
            payload = json.loads(metrics_path.read_text(encoding="utf-8"))
            manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
        except (json.JSONDecodeError, OSError):
            continue
        metrics, history = payload.get("metrics", {}), payload.get("history", {})
        validation = history.get("val_accuracy", [])
        stopping = payload.get("early_stopping", {})
        rows.append({
            "run_id": directory.name, "status": manifest.get("status", "unknown"),
            "dataset": config.get("dataset"), "CNN": str(config.get("model_type", "unknown")).upper(),
            "segmentador": config.get("segmenter", "nenhum"), "batch": config.get("batch_size"),
            "batch efetivo": config.get("batch_size", 0) * config.get("gradient_accumulation_steps", 1),
            "learning rate": config.get("learning_rate"), "dropout": config.get("dropout"), "seed": config.get("seed"),
            "OA": metrics.get("overall_accuracy"), "AA": metrics.get("average_accuracy"), "Kappa": metrics.get("kappa"),
            "melhor validação": max(validation) if validation else float("nan"),
            "épocas executadas": payload.get("epochs_run", len(validation)),
            "parou cedo": payload.get("stopped_early", False),
            "meta validação": stopping.get("target", config.get("early_stopping_target")),
            "meta atingida": stopping.get("target_reached", False),
            "treino (s)": payload.get("train_time"), "pré-processamento (s)": payload.get("prep_timings", {}).get("wall"),
            "acurácia por classe": metrics.get("per_class_accuracy", []), "diretório": str(directory.resolve()),
        })
    return pd.DataFrame(rows)


def show_all_run_note(key):
    """Mostra como o gráfico foi obtido e a regra correta de leitura."""
    title, note = ALL_RUN_PLOT_NOTES[key]
    display(Markdown(f"> **Como ler — {title}:** {note}"))


In [ ]:
# ============================================================================
# FIGURAS E PDF GLOBAIS — definição única de all_run_figures e do relatório
# ============================================================================
"""Curvas de evolução, cobertura, repetibilidade e o PDF consolidado de todos os runs.

Esta célula depende de `_all_run_core_figures`, `ALL_RUN_PLOT_NOTES` e
`load_all_audited_runs`, definidos na célula anterior, e é a única a definir
`all_run_figures` e `generate_all_runs_pdf`.
"""

def load_all_training_histories(runs_root=None):
    """Curvas persistidas de loss e acurácia de todos os runs auditados."""
    traces=[]
    for directory in Path(runs_root or output_dir()/"runs").glob("*"):
        mp,cp=directory/"metrics.json",directory/"config.json"
        if not mp.exists() or not cp.exists(): continue
        payload=json.loads(mp.read_text(encoding="utf-8")); history=payload.get("history",{})
        if history.get("val_accuracy"):
            config=json.loads(cp.read_text(encoding="utf-8")); traces.append({"CNN":str(config.get("model_type","unknown")).upper(),"history":history})
    return traces


def plot_training_evolution_all_runs():
    """Linhas individuais, média e desvio por CNN para quatro sinais de treino."""
    traces=load_all_training_histories()
    if not traces: return None
    figure,axes=plt.subplots(2,2,figsize=(16,10),constrained_layout=True)
    # A mesma cor identifica a mesma arquitetura nos quatro painéis. Linhas
    # translúcidas são runs individuais; a linha espessa e a faixa são média ±
    # desvio. Isso evita que a paleta automática misture CNNs diferentes.
    palette = {"CNN1D": "#0072B2"}
    for axis,(key,title) in zip(axes.ravel(),[("accuracy","Acurácia de treino"),("val_accuracy","Acurácia de validação"),("loss","Loss de treino"),("val_loss","Loss de validação")]):
        for cnn in sorted({x["CNN"] for x in traces}):
            color = palette.get(cnn, "#CC79A7")
            series=[np.asarray(x["history"].get(key,[]),float) for x in traces if x["CNN"]==cnn and x["history"].get(key)]
            if not series: continue
            for values in series:
                axis.plot(np.arange(1,len(values)+1), values, color=color, alpha=.10, lw=.9)
            length=max(map(len,series)); stack=np.full((len(series),length),np.nan)
            for row,values in enumerate(series): stack[row,:len(values)]=values
            mean,std=np.nanmean(stack,axis=0),np.nanstd(stack,axis=0); epochs=np.arange(1,length+1)
            axis.plot(epochs, mean, color=color, lw=2.8, label=f"{cnn} — média")
            axis.fill_between(epochs, mean-std, mean+std, color=color, alpha=.18, label=f"{cnn} — ±1 desvio")
        axis.set(title=title,xlabel="Época",ylabel="pontuação" if "Acurácia" in title else "cross-entropy")
        axis.legend(fontsize=8, ncol=2); axis.grid(alpha=.25)
    return figure


SEGMENTATION_REPORT_NOTES = {
    "segmenter_overlays": (
        "Saída visual dos segmentadores",
        "Cada painel usa a mesma composição PCA/RGB; a linha colorida é o contorno de um superpixel. Compare se os contornos acompanham limites visuais, sem confundir superpixel com classe prevista.",
    ),
    "segmenter_geometry": (
        "Escala e homogeneidade dos segmentadores",
        "O painel compara número e tamanho das regiões, densidade de borda, pureza rotulada e distribuições de área. Pureza mede homogeneidade da região e não é acurácia da CNN.",
    ),
}


def segmentation_report_figures(frame):
    """Gera a evidência visual dos quatro segmentadores para o PDF consolidado.

    A cena é escolhida a partir dos `config.json` já auditados. O cálculo usa as
    receitas padrão de visualização, não altera qualquer run e falha de maneira
    informativa caso os arquivos da cena não estejam disponíveis localmente.
    """
    datasets = [] if frame.empty else frame["dataset"].dropna().unique().tolist()
    dataset = datasets[0] if datasets else globals().get("config", ExperimentConfig()).dataset
    try:
        scene = load_scene(dataset, download=False)
        return {
            "segmenter_overlays": plot_segmentation_overlay_suite(scene),
            "segmenter_geometry": plot_segmentation_quality_dashboard(scene),
        }, None
    except Exception as error:
        return {}, f"Não foi possível gerar a saída dos segmentadores para {dataset}: {type(error).__name__}: {error}"


def _all_run_core_figures(frame):
    """Gráficos-base do conjunto de runs; `all_run_figures` os complementa."""
    import seaborn as sns
    palette = dict(zip(sorted(frame["CNN"].dropna().unique()), sns.color_palette("tab10")))
    figures = {}
    melted = frame.melt(id_vars=["CNN"], value_vars=["OA", "AA", "Kappa"], var_name="métrica", value_name="valor").dropna()
    figure, axis = plt.subplots(figsize=(13, 6)); sns.boxplot(data=melted, x="métrica", y="valor", hue="CNN", ax=axis, palette=palette); sns.stripplot(data=melted, x="métrica", y="valor", hue="CNN", dodge=True, ax=axis, palette=palette, alpha=.45, legend=False); axis.set(title="Qualidade de todos os runs", ylim=(0, 1)); axis.grid(axis="y", alpha=.25); figures["quality"] = figure
    heat = frame.pivot_table(index="CNN", columns="segmentador", values="AA", aggfunc="mean")
    if not heat.empty:
        figure, axis = plt.subplots(figsize=(10, 5)); sns.heatmap(heat, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="YlGnBu", ax=axis); axis.set_title("AA média por CNN e segmentador"); figures["segmenter"] = figure
    figure, axis = plt.subplots(figsize=(11, 6)); sns.scatterplot(data=frame, x="treino (s)", y="AA", hue="CNN", size="batch", sizes=(35, 240), palette=palette, ax=axis); axis.set(title="Pareto: qualidade × custo", ylim=(0, 1)); axis.grid(alpha=.25); figures["pareto"] = figure
    times = frame.melt(id_vars=["CNN"], value_vars=["treino (s)", "pré-processamento (s)"], var_name="etapa", value_name="segundos").dropna()
    figure, axis = plt.subplots(figsize=(12, 6)); sns.boxplot(data=times, x="etapa", y="segundos", hue="CNN", palette=palette, ax=axis); axis.set(title="Distribuição de tempo por etapa"); axis.grid(axis="y", alpha=.25); figures["times"] = figure
    figure, axis = plt.subplots(figsize=(11, 6)); sns.scatterplot(data=frame, x="learning rate", y="melhor validação", hue="CNN", size="dropout", sizes=(40, 220), palette=palette, ax=axis); axis.set_xscale("log"); axis.set(title="Hiperparâmetros: learning rate × validação", ylim=(0, 1)); axis.grid(alpha=.25); figures["hyperparameters"] = figure
    figure, axis = plt.subplots(figsize=(11, 6)); sns.scatterplot(data=frame, x="épocas executadas", y="melhor validação", hue="parou cedo", style="CNN", palette={True: "#E45756", False: "#54A24B"}, ax=axis); axis.set(title="Convergência e parada antecipada", ylim=(0, 1)); axis.grid(alpha=.25); figures["early"] = figure
    figure, axis = plt.subplots(figsize=(11, 6)); sns.scatterplot(data=frame, x="batch", y="melhor validação", hue="CNN", style="segmentador", palette=palette, ax=axis); axis.set(title="Batch físico × melhor validação", ylim=(0, 1)); axis.grid(alpha=.25); figures["batch"] = figure
    ordered = frame.sort_values("run_id").reset_index(drop=True)
    figure, axis = plt.subplots(figsize=(14, 6)); axis.plot(ordered.index + 1, ordered["AA"], marker="o", label="AA"); axis.plot(ordered.index + 1, ordered["melhor validação"], marker=".", label="melhor validação"); axis.set(title="Linha do tempo dos runs", xlabel="Ordem temporal do run", ylabel="Pontuação", ylim=(0, 1)); axis.legend(); axis.grid(alpha=.25); figures["timeline"] = figure
    compatible = frame[frame["acurácia por classe"].map(len) == frame["acurácia por classe"].map(len).max()]
    if not compatible.empty and compatible["acurácia por classe"].map(len).iloc[0] > 0:
        values = compatible.groupby("CNN")["acurácia por classe"].apply(lambda series: np.mean(np.vstack(series), axis=0))
        matrix = np.vstack(values.to_list())
        figure, axis = plt.subplots(figsize=(14, max(3, len(values) * .8))); sns.heatmap(matrix, vmin=0, vmax=1, cmap="viridis", annot=True, fmt=".2f", yticklabels=values.index, ax=axis); axis.set(title="Acurácia média por classe", xlabel="Classe", ylabel="CNN"); figures["classes"] = figure
    return figures



def all_run_figures(frame):
    """Todos os gráficos inferíveis de config.json, metrics.json e manifest.json.

    Definição única: os painéis-base vêm de `_all_run_core_figures` e os painéis de
    evolução, cobertura e repetibilidade são acrescentados aqui. Não há redefinição
    posterior, então rodar esta célula duas vezes produz exatamente o mesmo objeto.
    """
    if frame.empty:
        return {}
    figures = _all_run_core_figures(frame)
    import seaborn as sns
    figure,axis=plt.subplots(figsize=(8,7)); sns.scatterplot(data=frame,x="OA",y="AA",hue="CNN",style="segmentador",s=85,ax=axis); axis.plot([0,1],[0,1],"--",color="black",lw=1); axis.set(title="OA × AA: efeito do desbalanceamento",xlim=(0,1),ylim=(0,1)); axis.grid(alpha=.25); figures["gaps"]=figure
    coverage=frame.groupby(["CNN","segmentador"]).size().reset_index(name="runs"); figure,axis=plt.subplots(figsize=(12,5)); sns.barplot(data=coverage,x="CNN",y="runs",hue="segmentador",ax=axis); axis.set(title="Cobertura de runs por CNN e segmentador",ylabel="quantidade de runs"); axis.grid(axis="y",alpha=.25); figures["coverage"]=figure
    training=plot_training_evolution_all_runs()
    if training is not None: figures["training"]=training

    figure, axis = plt.subplots(figsize=(9, 7))
    sns.scatterplot(data=frame, x="melhor validação", y="AA", hue="CNN", style="segmentador", s=90, ax=axis)
    axis.plot([0, 1], [0, 1], "--", color="black", lw=1, label="referência 1:1")
    axis.set(title="Melhor validação × AA de teste", xlim=(0, 1), ylim=(0, 1), xlabel="melhor validação", ylabel="AA de teste")
    axis.grid(alpha=.25); axis.legend()
    figures["validation_test"] = figure
    stopping = frame.groupby("CNN")[["meta atingida", "parou cedo"]].mean().reset_index().melt(id_vars="CNN", var_name="evento", value_name="proporção")
    figure, axis = plt.subplots(figsize=(10, 5))
    sns.barplot(data=stopping, x="CNN", y="proporção", hue="evento", ax=axis)
    axis.set(title="Meta de validação e early stopping", ylabel="proporção de runs", ylim=(0, 1))
    axis.grid(axis="y", alpha=.25)
    figures["stopping"] = figure
    if "seed" in frame and frame["seed"].notna().any():
        figure, axis = plt.subplots(figsize=(11, 6))
        sns.boxplot(data=frame, x="CNN", y="AA", hue="CNN", legend=False, ax=axis)
        sns.stripplot(data=frame, x="CNN", y="AA", hue="seed", dodge=False, palette="tab10", ax=axis)
        axis.set(title="AA por seed: repetibilidade", ylabel="AA de teste", ylim=(0, 1))
        axis.grid(axis="y", alpha=.25); axis.legend(title="seed", bbox_to_anchor=(1.02, 1), loc="upper left")
        figures["seeds"] = figure
    return figures


def generate_all_runs_pdf(runs_root=None, reports_root=None):
    """PDF visual de todos os runs, mais o CSV bruto para conferência externa.

    O PDF é inteiramente visual (sem páginas de tabela); quem precisar dos números
    exatos usa o CSV, que carrega as mesmas linhas lidas dos artefatos auditados.
    """
    frame = load_all_audited_runs(runs_root)
    reports = Path(reports_root or output_dir() / "reports")
    reports.mkdir(parents=True, exist_ok=True)
    csv_path, pdf_path = reports / "all_runs.csv", reports / "all_runs_report.pdf"
    frame.drop(columns=["acurácia por classe"], errors="ignore").to_csv(csv_path, index=False)
    with PdfPages(pdf_path) as pdf:
        _pdf_text(pdf,"Relatório visual consolidado",f"Runs incluídos: {len(frame)}\nDatasets: {sorted(frame['dataset'].dropna().unique().tolist()) if not frame.empty else []}\nOrigem: config.json, metrics.json e manifest.json de cada run.")
        for key,figure in all_run_figures(frame).items():
            pdf.savefig(figure,bbox_inches="tight"); _pdf_text(pdf,f"Como ler: {ALL_RUN_PLOT_NOTES[key][0]}",ALL_RUN_PLOT_NOTES[key][1]); plt.close(figure)
        segment_figures, segment_error = segmentation_report_figures(frame)
        if segment_error:
            _pdf_text(pdf, "Saídas dos segmentadores", segment_error)
        for key, figure in segment_figures.items():
            pdf.savefig(figure, bbox_inches="tight")
            _pdf_text(pdf, f"Como ler: {SEGMENTATION_REPORT_NOTES[key][0]}", SEGMENTATION_REPORT_NOTES[key][1])
            plt.close(figure)
    return pdf_path, csv_path, frame


### 5.6 Explicação inversa: da decisão para a imagem

Esta camada reconstrói previsões, confiança, entropia, acertos, erros e F1 no local onde ocorreram. Em seguida agrega essas evidências por superpixel e escolhe automaticamente detecções representativas para Grad-CAM.

In [ ]:
# ============================================================================
# EXPLICAÇÃO INVERSA NA CENA — saída do modelo -> pixels -> segmentos -> imagem
# ============================================================================
"""Reconstrói decisões e métricas nas coordenadas originais da imagem HSI.

O modelo é treinado apenas nos pixels rotulados; por isso mapas de predição,
confiança e erro são explicitamente mascarados fora do teste. Essa escolha evita
inventar uma métrica onde não há ground truth. A segmentação é reconstruída de
forma determinística com a receita registrada, e cada superpixel recebe estatísticas
agregadas das previsões que o atravessam.
"""

from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap


_SPATIAL_CONTEXT_CACHE = {}


def rebuild_spatial_context(result, scene):
    """Reproduz RGB, superpixels e contornos da receita do run, com cache seguro."""
    key = (scene.code, scene.shape, result.config.segmenter, result.config.seed)
    if key not in _SPATIAL_CONTEXT_CACHE:
        pixels, raw_labels = to_pixel_table(scene)
        spatial = spatial_branch(
            pixels, raw_labels, scene.shape, _segmenter_params(result.config),
            result.config.seed, result.config.segmenter,
        )
        _, components, rgb, segmentation = spatial
        _SPATIAL_CONTEXT_CACHE[key] = SegmentationArtifacts(components=components, rgb=rgb, result=segmentation)
    return _SPATIAL_CONTEXT_CACHE[key]


def inverse_prediction_maps(result, scene):
    """Projeta probabilidades e métricas do teste de volta à matriz H×W da cena."""
    loaders = build_loaders(result.data, result.config, result.device)
    probabilities = predict_proba(result.model, loaders["test"], result.device)
    predicted = probabilities.argmax(axis=1)
    true = result.data.labels[result.data.splits.test]
    positions = np.flatnonzero(scene.labels.ravel() > 0)[result.data.splits.test]
    if not np.array_equal(predicted, result.predicted):
        raise RuntimeError("Ordem de probabilidades não coincide com as previsões registradas do run.")
    per_class, _ = classification_metrics_table(true, predicted)
    f1_lookup = per_class.set_index("classe")["F1"].to_dict()
    total = scene.labels.size
    def canvas(values):
        image = np.full(total, np.nan, dtype=float)
        image[positions] = values
        return image.reshape(scene.shape)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(np.maximum(probabilities, 1e-12))).sum(axis=1) / np.log(probabilities.shape[1])
    return {
        "positions": positions, "probabilities": probabilities, "true": true, "predicted": predicted,
        "confidence": confidence, "entropy": entropy, "per_class": per_class,
        "ground_truth": canvas(true + 1), "prediction": canvas(predicted + 1),
        "correct": canvas((true == predicted).astype(float)), "confidence_map": canvas(confidence),
        "entropy_map": canvas(entropy), "f1_map": canvas(np.array([f1_lookup[int(label + 1)] for label in true])),
    }


def segment_decision_metrics(result, scene, inverse, artifacts):
    """Agrega acerto, confiança, entropia e F1 por superpixel na imagem original."""
    segment_ids = artifacts.labels.ravel()[inverse["positions"]]
    rows, maps = [], {name: np.full(scene.labels.size, np.nan) for name in ("acurácia", "confiança", "entropia", "F1 da classe", "amostras teste")}
    for segment_id in np.unique(segment_ids):
        mask = segment_ids == segment_id
        spatial_mask = artifacts.labels.ravel() == segment_id
        true = inverse["true"][mask]
        predicted = inverse["predicted"][mask]
        row = {
            "segmento": int(segment_id), "amostras teste": int(mask.sum()),
            "acurácia": float(np.mean(true == predicted)),
            "confiança": float(np.mean(inverse["confidence"][mask])),
            "entropia": float(np.mean(inverse["entropy"][mask])),
            "F1 da classe": float(np.mean(inverse["f1_map"].ravel()[inverse["positions"]][mask])),
            "classe real dominante": int(np.bincount(true).argmax() + 1),
            "classe prevista dominante": int(np.bincount(predicted).argmax() + 1),
        }
        rows.append(row)
        for name in maps:
            maps[name][spatial_mask] = row[name]
    return pd.DataFrame(rows).sort_values(["acurácia", "amostras teste"]), {name: value.reshape(scene.shape) for name, value in maps.items()}


def _contoured(axis, base, contours, title, cmap=None, vmin=None, vmax=None, discrete=False):
    """Exibe um mapa e coloca contornos de superpixels para ligar métrica e região."""
    if base.ndim == 3:
        axis.imshow(base)
    else:
        image = axis.imshow(base, cmap=cmap, vmin=vmin, vmax=vmax)
        if not discrete:
            plt.colorbar(image, ax=axis, shrink=.72)
    axis.contour(contours > 0, levels=[.5], colors="white", linewidths=.35, alpha=.8)
    axis.set_title(title); axis.axis("off")


def plot_inverse_prediction_dashboard(result, scene, inverse, artifacts):
    """Mostra exatamente onde o modelo acertou, errou, ficou incerto e teve F1 baixo."""
    fig, axes = plt.subplots(2, 3, figsize=(19, 12), constrained_layout=True)
    _contoured(axes[0, 0], artifacts.rgb, artifacts.contour_mask, f"Entrada PCA/RGB + {result.config.segmenter.upper()}")
    _contoured(axes[0, 1], inverse["ground_truth"], artifacts.contour_mask, "Ground truth — pixels de teste", cmap=_label_cmap(result.data.n_classes), vmin=1, vmax=result.data.n_classes, discrete=True)
    _contoured(axes[0, 2], inverse["prediction"], artifacts.contour_mask, "Classe prevista — pixels de teste", cmap=_label_cmap(result.data.n_classes), vmin=1, vmax=result.data.n_classes, discrete=True)
    _contoured(axes[1, 0], inverse["correct"], artifacts.contour_mask, "Acerto por pixel (verde) / erro (vermelho)", cmap=ListedColormap(["#d73027", "#1a9850"]), vmin=0, vmax=1)
    _contoured(axes[1, 1], inverse["confidence_map"], artifacts.contour_mask, "Confiança máxima da previsão", cmap="viridis", vmin=0, vmax=1)
    _contoured(axes[1, 2], inverse["f1_map"], artifacts.contour_mask, "F1 da classe real aplicado à imagem", cmap="magma", vmin=0, vmax=1)
    fig.suptitle(f"Processo inverso — {result.config.model_type.upper()}: modelo → métrica → localização", fontsize=15)
    return fig


def plot_segment_metric_dashboard(result, artifacts, segment_maps):
    """Aplica métricas agregadas a cada região produzida pelo segmentador."""
    fig, axes = plt.subplots(2, 3, figsize=(19, 12), constrained_layout=True)
    _contoured(axes[0, 0], colorize_segments(artifacts.labels), artifacts.contour_mask, "IDs dos segmentos")
    _contoured(axes[0, 1], segment_maps["amostras teste"], artifacts.contour_mask, "Cobertura: pixels de teste por segmento", cmap="Blues")
    _contoured(axes[0, 2], segment_maps["acurácia"], artifacts.contour_mask, "Acurácia por segmento", cmap="RdYlGn", vmin=0, vmax=1)
    _contoured(axes[1, 0], segment_maps["confiança"], artifacts.contour_mask, "Confiança média por segmento", cmap="viridis", vmin=0, vmax=1)
    _contoured(axes[1, 1], segment_maps["entropia"], artifacts.contour_mask, "Entropia média por segmento", cmap="magma", vmin=0, vmax=1)
    _contoured(axes[1, 2], segment_maps["F1 da classe"], artifacts.contour_mask, "F1 médio da classe real por segmento", cmap="magma", vmin=0, vmax=1)
    fig.suptitle(f"Segmentador + decisões da {result.config.model_type.upper()}", fontsize=15)
    return fig


def _single_input_from_index(result, index):
    """Reconstrói exatamente o tensor de uma amostra do teste para Grad-CAM local."""
    tensors = _tensors(result.data, np.asarray([index]), result.config.model_type)
    return unpack_batch(tensors, result.device)


def plot_representative_detections(result, inverse, limit=4):
    """Explica automaticamente casos fáceis, incertos e erros confiantes com Grad-CAM.

    Os exemplos são escolhidos por regras auditáveis: maior confiança correta,
    maior entropia e maior confiança incorreta. Dessa forma o painel não escolhe
    manualmente casos visualmente convenientes.
    """
    true, predicted = inverse["true"], inverse["predicted"]
    correct = true == predicted
    candidates = OrderedDict()
    if correct.any():
        candidates["acerto mais confiante"] = int(np.flatnonzero(correct)[np.argmax(inverse["confidence"][correct])])
    candidates["maior incerteza"] = int(np.argmax(inverse["entropy"]))
    if (~correct).any():
        candidates["erro mais confiante"] = int(np.flatnonzero(~correct)[np.argmax(inverse["confidence"][~correct])])
    candidates["menor confiança"] = int(np.argmin(inverse["confidence"]))
    unique = list(OrderedDict((index, label) for label, index in candidates.items()).items())[:limit]
    fig, axes = plt.subplots(len(unique), 2, figsize=(13, 4 * len(unique)), constrained_layout=True)
    axes = np.atleast_2d(axes)
    test_indices = result.data.splits.test
    for row, (local_index, label) in enumerate(unique):
        x, seg, _ = _single_input_from_index(result, int(test_indices[local_index]))
        cam, predicted_class, layer = grad_cam(result.model, x, seg)
        if result.config.model_type == "cnn1d":
            axes[row, 0].plot(x[0, 0].detach().cpu(), color="#4C78A8")
            axes[row, 0].set(xlabel="Componente PCA", ylabel="Valor")
            axes[row, 1].plot(cam, color="crimson")
            axes[row, 1].set(xlabel="Posição após convoluções", ylabel="Grad-CAM")
            low, high = patch.min(axis=(0, 1), keepdims=True), patch.max(axis=(0, 1), keepdims=True)
            axes[row, 0].imshow((patch - low) / np.maximum(high - low, 1e-12))
            axes[row, 1].imshow(cam, cmap="jet", vmin=0, vmax=1)
        else:
            axes[row, 0].imshow(x[0, 0].detach().cpu().max(dim=0).values, cmap="gray")
            axes[row, 1].imshow(cam.max(axis=0), cmap="jet", vmin=0, vmax=1)
        axes[row, 0].set_title(f"{label}: entrada local")
        axes[row, 1].set_title(f"Grad-CAM: real {true[local_index] + 1}, prevista {predicted_class + 1}\n{layer}")
        for axis in axes[row]: axis.grid(False)
    fig.suptitle("Detecções representativas escolhidas automaticamente", fontsize=15)
    return fig


def render_spatial_explanation(result, scene):
    """Executa a explicação inversa completa e retorna figuras e métricas para qualquer CNN."""
    artifacts = rebuild_spatial_context(result, scene)
    inverse = inverse_prediction_maps(result, scene)
    segment_table, segment_maps = segment_decision_metrics(result, scene, inverse, artifacts)
    return {
        "segmentos": segment_table,
        "figures": {
            "decisões remontadas na cena": plot_inverse_prediction_dashboard(result, scene, inverse, artifacts),
            "métricas aplicadas aos segmentos": plot_segment_metric_dashboard(result, artifacts, segment_maps),
            "detecções locais explicadas": plot_representative_detections(result, inverse),
        },
    }


# Integra a explicação espacial à rotina de dashboard da CNN1D.
def render_cnn_dashboard(result, scene, max_channels=8):
    """Dashboard completo: núcleo da seção 5.3 + decisões reconstruídas na cena.

    Definição única e não reentrante — chama `_render_cnn_dashboard_core`, um nome
    fixo, em vez de guardar um alias da versão anterior de si mesma.
    """
    dashboard = _render_cnn_dashboard_core(result, scene, max_channels=max_channels)
    spatial = render_spatial_explanation(result, scene)
    dashboard["segmentos"] = spatial["segmentos"]
    dashboard["figures"].update({f"imagem — {name}": figure for name, figure in spatial["figures"].items()})
    return dashboard


def display_cnn_dashboard(name, dashboard):
    """Exibe o dashboard padrão e os indicadores de superpixels como gráfico."""
    _display_cnn_dashboard_core(name, dashboard)
    if "segmentos" in dashboard:
        from IPython.display import Markdown, display
        display(Markdown("#### Métricas agregadas por segmento"))
        display(plot_segment_metrics_visual(dashboard["segmentos"]))


In [ ]:
# ============================================================================
# POLÍTICA DE SELEÇÃO — repetir somente os melhores casos encontrados pelo HPO
# ============================================================================
"""Ponte entre screening de HPO e campanha final de resultados científicos.

O HPO é uma fase de triagem: avalia candidatos com orçamento reduzido e seleciona
por validação. A campanha final não mistura candidatos inferiores no comparativo;
ela repete exclusivamente o vencedor de cada CNN, com várias seeds e orçamento de
épocas completo. A decisão e todos os candidatos ficam congelados em `selection.json`.
"""


def select_hpo_winners(hpo_table):
    """Seleciona o melhor trial por CNN usando apenas `val_accuracy (objetivo)`.

    Critérios de desempate, nesta ordem: maior validação, menor tempo de treino e
    menor número do trial. OA/AA/Kappa de teste não participam da escolha.
    """
    winners = {}
    for cnn, group in hpo_table.groupby("CNN", sort=True):
        ranked = group.sort_values(
            ["val_accuracy (objetivo)", "tempo treino (s)", "trial"],
            ascending=[False, True, True],
        )
        winners[cnn.lower()] = ranked.iloc[0]
    return winners


def run_best_hpo_campaign(hpo_table, final_epochs, seeds=(11, 29, 42), backend="thread", campaign_root=None):
    """Repete e audita os melhores casos HPO com múltiplas seeds.

    O diretório da campanha contém o snapshot de todos os candidatos HPO, a regra
    de seleção, um hash do plano final e os links para cada run final. Assim é
    possível responder exatamente *por que* uma receita foi promovida ao estudo
    comparativo e quais execuções formaram suas médias/intervalos.
    """
    winners = select_hpo_winners(hpo_table)
    started = datetime.now(timezone.utc)
    campaign_id = f"best_hpo_{started.strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"
    directory = Path(campaign_root or output_dir() / "campaigns") / campaign_id
    directory.mkdir(parents=True, exist_ok=False)
    screening = hpo_table.drop(columns=["resultado", "config"], errors="ignore").to_dict(orient="records")
    selection = []
    final_plan = []
    for cnn, row in winners.items():
        winner_config = replace(row["config"], epochs=final_epochs)
        selection.append({
            "CNN": cnn.upper(), "selected_trial": int(row["trial"]),
            "selection_metric": "val_accuracy (objetivo)", "selection_value": float(row["val_accuracy (objetivo)"]),
            "config": winner_config.to_dict(),
        })
        for repeat, seed in enumerate(seeds, start=1):
            final_config = replace(winner_config, seed=int(seed))
            final_plan.append({
                "trial_key": f"final_{cnn}_s{seed}", "phase": "final_best_hpo",
                "repeat": repeat, "config": final_config.to_dict(),
            })
    final_hash = sha256(json.dumps(final_plan, sort_keys=True).encode("utf-8")).hexdigest()
    campaign = {
        "schema_version": 1, "campaign_id": campaign_id, "status": "running",
        "started_at_utc": started.isoformat(), "backend": backend,
        "selection_policy": "best validation HPO per CNN; test report-only",
        "selection_metric": "validation_best", "test_policy": "report_only",
        "screening_trials": len(screening), "planned_final_trials": len(final_plan),
        "final_plan_sha256": final_hash, "seeds": list(seeds),
    }
    _campaign_json(directory / "campaign.json", campaign)
    _campaign_json(directory / "hpo_screening.json", screening)
    _campaign_json(directory / "selection.json", selection)
    _campaign_json(directory / "final_plan.json", final_plan)
    records = []
    for recipe in final_plan:
        config = ExperimentConfig(**recipe["config"])
        record = run_audited_experiment(config, directory, campaign_id, recipe["trial_key"], backend)
        record["phase"] = recipe["phase"]
        records.append(record)
        _campaign_json(directory / "results.json", _serializable_records(records))
    frame = pd.DataFrame(_serializable_records(records))
    analysis = analyze_campaign(frame)
    frame.to_csv(directory / "summary.csv", index=False)
    _campaign_json(directory / "analysis.json", analysis["json"])
    report_context = {**campaign, "plan_sha256": final_hash}
    (directory / "report.md").write_text(campaign_report(report_context, frame, analysis), encoding="utf-8")
    campaign.update({
        "status": "completed", "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "completed_trials": int((frame["status"] == "completed").sum()),
        "failed_trials": int((frame["status"] == "failed").sum()),
        "artifacts": ["hpo_screening.json", "selection.json", "final_plan.json", "results.json", "summary.csv", "analysis.json", "report.md", "report.pdf"],
    })
    _campaign_json(directory / "campaign.json", campaign)
    pdf_report = generate_pdf_report(directory, f"Campanha HPO {campaign_id}")
    global_pdf, _, _ = generate_all_runs_pdf()
    objects = {record["run_id"]: record["result"] for record in records if record["result"] is not None}
    return {"campaign": campaign, "directory": directory, "records": frame, "objects": objects, "analysis": analysis, "selection": pd.DataFrame(selection), "pdf_report": pdf_report, "global_pdf_report": global_pdf}


def best_hpo_results_by_cnn(campaign):
    """Retorna um run representativo por CNN para os dashboards finais.

    Com a mesma receita vencedora repetida por seed, escolhe-se o run de maior
    validação apenas para visualização; o ranking científico continua sendo a média
    e o intervalo da campanha, nunca esse run individual.
    """
    results = {}
    completed = campaign["records"].query("status == 'completed'")
    for model_type, group in completed.groupby("model_type"):
        selected = group.sort_values("validation_best", ascending=False).iloc[0]
        results[model_type] = campaign["objects"][selected["run_id"]]
    return results


In [ ]:
# ============================================================================
# BATCH SWEEP — escada uniforme de batch nos vencedores do HPO, com auditoria
# ============================================================================
"""Escada de batch físico da GPU, medida sobre os vencedores do HPO.

O HPO da seção 5.4 já varia `batch_size` como eixo de **qualidade**, misturado aos
demais. Este sweep é complementar e responde outra pergunta: fixada a receita
vencedora, qual batch entrega mais **throughput** sem perder validação. Por isso ele
varre uma escada uniforme de 50 em 50 em vez de sortear, continua mesmo se um valor
falhar por falta de memória, e escolhe pelo maior throughput dentro da tolerância.
"""

from dataclasses import replace
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from uuid import uuid4

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def run_audited_batch_sweep(config, batch_start=50, batch_max=300, step=50, epochs=10, quality_tolerance=.005, backend="thread", campaign_root=None):
    """Compara uma escada uniforme e escolhe velocidade sem sacrificar validação."""
    if any(not isinstance(value, int) or value <= 0 for value in (batch_start, batch_max, step)):
        raise ValueError("batch_start, batch_max e step devem ser inteiros positivos.")
    if batch_start > batch_max or (batch_max - batch_start) % step:
        raise ValueError("A escada deve terminar exatamente em batch_max.")
    if not 0 <= quality_tolerance < 1:
        raise ValueError("quality_tolerance deve estar no intervalo [0, 1).")
    batch_sizes = list(range(batch_start, batch_max + 1, step))
    started = datetime.now(timezone.utc)
    campaign_id = f"batch_sweep_{started.strftime('%Y%m%dT%H%M%SZ')}_{config.model_type}_{uuid4().hex[:8]}"
    directory = Path(campaign_root or output_dir() / "campaigns") / campaign_id
    directory.mkdir(parents=True, exist_ok=False)
    plan = [
        {"trial_key": f"{config.model_type}_batch_{batch}", "batch_size": batch,
         "config": replace(config, batch_size=batch, epochs=epochs, patience=min(config.patience, max(2, epochs // 3))).to_dict()}
        for batch in batch_sizes
    ]
    manifest = {
        "schema_version": 1, "campaign_id": campaign_id, "status": "running", "kind": "batch_sweep",
        "started_at_utc": started.isoformat(), "selection_metric": "validation_best then throughput",
        "plan_sha256": sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(), "batch_sizes": batch_sizes,
        "batch_ladder": {"start": batch_start, "step": step, "max": batch_max, "rationale": "incrementos fixos medem ganho marginal e limite de memória"},
        "quality_tolerance": quality_tolerance,
    }
    _campaign_json(directory / "campaign.json", manifest)
    _campaign_json(directory / "plan.json", plan)
    records = []
    for recipe in plan:
        candidate = ExperimentConfig(**recipe["config"])
        record = run_audited_experiment(candidate, directory, campaign_id, recipe["trial_key"], backend)
        record["batch_size"] = candidate.batch_size
        record["effective_batch"] = candidate.effective_batch_size
        if record["status"] == "completed":
            result = record["result"]
            record["validation_best"] = float(max(result.training.history.val_accuracy))
            record["throughput_samples_s"] = len(result.data.splits.train) * len(result.training.history) / max(result.training.total_time, 1e-12)
        records.append(record)
        _campaign_json(directory / "results.json", _serializable_records(records))
    frame = pd.DataFrame(_serializable_records(records))
    completed = frame.query("status == 'completed'").copy()
    if not completed.empty:
        best_validation = completed["validation_best"].max()
        viable = completed[completed["validation_best"] >= best_validation - quality_tolerance]
        winner = viable.sort_values(["throughput_samples_s", "batch_size"], ascending=[False, True]).iloc[0].to_dict()
    else:
        winner = None
    frame.to_csv(directory / "summary.csv", index=False)
    analysis = {"selection": f"maior throughput com validação até {quality_tolerance:.3f} do melhor valor", "batch_ladder_rationale": "incrementos fixos isolam o efeito de +50 amostras e revelam saturação/OOM", "quality_tolerance": quality_tolerance, "winner": winner}
    _campaign_json(directory / "analysis.json", analysis)
    manifest.update({"status": "completed", "finished_at_utc": datetime.now(timezone.utc).isoformat(), "winner": winner, "artifacts": ["plan.json", "results.json", "summary.csv", "analysis.json", "report.pdf"]})
    _campaign_json(directory / "campaign.json", manifest)
    pdf_report = generate_pdf_report(directory, f"Sweep de batches {campaign_id}")
    global_pdf, _, _ = generate_all_runs_pdf()
    return {"directory": directory, "records": frame, "winner": winner, "batch_sizes": batch_sizes, "quality_tolerance": quality_tolerance, "pdf_report": pdf_report, "global_pdf_report": global_pdf, "objects": {r["run_id"]: r["result"] for r in records if r["result"] is not None}}


def plot_batch_sweep(sweep):
    """Visualiza qualidade, custo e throughput em função do batch físico da GPU."""
    frame = sweep["records"].query("status == 'completed'").sort_values("batch_size")
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
    axes[0, 0].plot(frame["batch_size"], frame["validation_best"], marker="o", label="validação")
    axes[0, 0].plot(frame["batch_size"], frame["oa_test"], marker="o", label="OA teste (relato)")
    axes[0, 0].set(title="Qualidade × batch", xlabel="Batch físico", ylabel="Pontuação", ylim=(0, 1)); axes[0, 0].legend(); axes[0, 0].grid(alpha=.25)
    axes[0, 1].plot(frame["batch_size"], frame["train_seconds"], marker="o", color="#F58518")
    axes[0, 1].set(title="Tempo de treino × batch", xlabel="Batch físico", ylabel="Segundos"); axes[0, 1].grid(alpha=.25)
    axes[1, 0].plot(frame["batch_size"], frame["throughput_samples_s"], marker="o", color="#54A24B")
    axes[1, 0].set(title="Throughput × batch", xlabel="Batch físico", ylabel="Amostras/s"); axes[1, 0].grid(alpha=.25)
    axes[1, 1].bar(frame["batch_size"].astype(str), frame["aa_test"], label="AA")
    axes[1, 1].bar(frame["batch_size"].astype(str), frame["kappa_test"], alpha=.55, label="Kappa")
    axes[1, 1].set(title="Robustez por batch", xlabel="Batch físico", ylabel="Pontuação", ylim=(0, 1)); axes[1, 1].legend(); axes[1, 1].grid(axis="y", alpha=.25)
    return fig


def run_batch_sweeps_for_hpo_winners(hpo_table, batch_start=50, batch_max=300, step=50, epochs=10, quality_tolerance=.005, backend="thread"):
    """Executa a mesma escada justificável para o melhor HPO de cada CNN."""
    winners = select_hpo_winners(hpo_table)
    return {
        name: run_audited_batch_sweep(row["config"], batch_start=batch_start, batch_max=batch_max, step=step, epochs=epochs, quality_tolerance=quality_tolerance, backend=backend)
        for name, row in winners.items()
    }


## 6. MLOps: rastreabilidade, registro e observabilidade

### 6.1 Identidade da execução, artefatos e métricas

Cada execução recebe um identificador, salva configuração, versões, fingerprints dos dados, métricas, artefatos e manifesto. O registry promove somente runs concluídos; as funções de observabilidade consolidam comparações e alertas locais.

In [ ]:
# ============================================================================
# TRACKING
# ============================================================================
"""Rastreabilidade local de experimentos, sem depender de um servidor externo.

Cada execução ganha um diretório imutável em ``outputs/runs/`` com a configuração,
as impressões digitais dos dados, ambiente, métricas e artefatos. Isso responde às
perguntas básicas de MLOps para o TCC: qual configuração produziu um modelo, com
qual versão de código/dados e qual foi seu resultado.
"""


import hashlib
import json
import platform
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import TYPE_CHECKING, Any
from uuid import uuid4

import numpy as np
import torch


if TYPE_CHECKING:
    pass  # Tipos já estão definidos em células anteriores.

__all__ = [
    "RunContext", "array_fingerprint", "data_fingerprint", "create_run", "record_run",
    "fail_run", "result_metrics",
]


def _write_json(path: Path, value: dict[str, Any]) -> Path:
    """Persiste um dicionário em JSON UTF-8 legível.

    Args:
        path: Arquivo de destino dentro (ou abaixo) do diretório da execução.
        value: Estrutura já serializável por ``json``.

    Returns:
        O próprio caminho, para facilitar a composição com outras funções.

    Os campos são ordenados para tornar ``diff`` entre execuções mais útil. Esta
    é uma função interna: o chamador público é ``create_run`` ou ``record_run``.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path


def array_fingerprint(array: np.ndarray | None) -> str | None:
    """Gera a identidade SHA-256 de um array que chega ao modelo.

    O hash inclui shape, dtype e bytes em ordem contígua. Portanto, a mesma
    matriz com dtype diferente, shape diferente ou qualquer valor modificado
    produz uma identidade diferente. ``None`` é preservado para representar a
    ausência intencional da segunda entrada no modo espectral.
    """
    if array is None:
        return None
    contiguous = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(contiguous.shape).encode())
    digest.update(contiguous.dtype.str.encode())
    digest.update(memoryview(contiguous).cast("B"))
    return digest.hexdigest()


def data_fingerprint(data: "PreparedData") -> dict[str, Any]:
    """Cria o contrato dos dados já processados usados nesta execução.

    Não faz hash dos `.mat` crus: o objetivo é registrar exatamente o que a CNN
    recebeu depois de PCA, remoção de background e segmentação. Os hashes dos
    índices garantem também que uma mudança de seed ou split seja detectável.
    O retorno contém apenas tipos serializáveis em JSON.
    """
    return {
        "spectra": {"shape": list(data.spectra.shape), "sha256": array_fingerprint(data.spectra)},
        "labels": {"shape": list(data.labels.shape), "sha256": array_fingerprint(data.labels)},
        "segments": None if data.segments is None else {
            "shape": list(data.segments.shape), "sha256": array_fingerprint(data.segments),
        },
        # Uma execução 2D/3D não consome só ``spectra``. Registrar estes arrays
        # impede que dois patches/modelos espaciais diferentes pareçam o mesmo
        # dataset no tracker local.
        "splits": {
            "train": array_fingerprint(data.splits.train),
            "val": array_fingerprint(data.splits.val),
            "test": array_fingerprint(data.splits.test),
        },
        "scene_shape": list(data.scene_shape),
        "n_classes": data.n_classes,
    }


def _git_revision() -> str | None:
    """Obtém o commit completo sem falhar fora de um repositório Git."""
    try:
        return subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=repo_root(), text=True,
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=True,
        ).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def _environment() -> dict[str, Any]:
    """Coleta somente dados de ambiente que influenciam reprodução/performance."""
    cuda_available = torch.cuda.is_available()
    environment: dict[str, Any] = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": cuda_available,
        "git_revision": _git_revision(),
    }
    if cuda_available:
        environment["gpu"] = torch.cuda.get_device_name(0)
        environment["gpu_capability"] = list(torch.cuda.get_device_capability(0))
    return environment


@dataclass(frozen=True)
class RunContext:
    """Identidade e diretório de uma execução rastreada."""

    run_id: str
    directory: Path


def create_run(config: "ExperimentConfig", base_dir: Path | None = None) -> RunContext:
    """Abre uma execução rastreada antes que dados ou pesos sejam criados.

    Args:
        config: Receita imutável que será usada pelo pipeline.
        base_dir: Raiz de runs alternativa. Sem valor, usa ``outputs/runs``.

    Returns:
        ``RunContext`` com identificador único UTC e diretório já criado.

    A função cria ``config.json``, ``environment.json`` e um manifesto com
    estado ``running``. Se o pipeline falhar depois, ``fail_run`` deve fechar
    esse estado; se terminar, ``record_run`` o substitui por ``completed``.
    """
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{timestamp}_{config.dataset}_{uuid4().hex[:8]}"
    directory = (base_dir or output_dir() / "runs") / run_id
    directory.mkdir(parents=True, exist_ok=False)
    context = RunContext(run_id=run_id, directory=directory)
    _write_json(directory / "config.json", config.to_dict())
    _write_json(directory / "environment.json", _environment())
    _write_json(directory / "manifest.json", {
        "schema_version": 1,
        "run_id": run_id,
        "status": "running",
        "started_at_utc": timestamp,
        "artifacts": ["config.json", "environment.json"],
    })
    return context


def result_metrics(result: "ExperimentResult") -> dict[str, Any]:
    """Converte ``ExperimentResult`` em um payload JSON sem tensores/arrays NumPy.

    O mesmo formato é usado em ``metrics.json`` e na saída opcional ``--json``.
    Centralizá-lo evita que uma métrica apareça em um formato no tracker e em
    outro no relatório solicitado pela CLI.
    """
    return {
        "metrics": {
            "overall_accuracy": result.metrics.overall_accuracy,
            "average_accuracy": result.metrics.average_accuracy,
            "kappa": result.metrics.kappa,
            "test_loss": result.metrics.test_loss,
            "test_accuracy": result.metrics.test_accuracy,
            "per_class_accuracy": result.metrics.per_class_accuracy.tolist(),
        },
        "history": result.training.history.to_dict(),
        "epochs_run": len(result.training.history),
        "prep_timings": {
            "backend": result.prep_timings.backend,
            "wall": result.prep_timings.wall,
            "stages": result.prep_timings.stages,
        },
        "train_time": result.training.total_time,
        "stopped_early": result.training.stopped_early,
        "best_val_loss": result.training.best_val_loss,
        "early_stopping": {
            "monitor": result.training.early_stopping_monitor,
            "min_delta": result.training.early_stopping_min_delta,
            "patience": result.training.patience,
            "restore_best_weights": result.training.restore_best_weights,
            "target": result.training.early_stopping_target,
            "target_reached": result.training.target_reached,
            "best_epoch": result.training.best_epoch,
            "best_monitor_value": result.training.best_monitor_value,
            "stop_reason": result.training.stop_reason,
        },
    }


def fail_run(context: RunContext, error: BaseException) -> None:
    """Marca uma execução como falha e persiste uma causa humana legível.

    Não relança nem tenta recuperar a exceção: essa decisão pertence à CLI. A
    função é chamada no ``except`` para que um run interrompido não pareça ter
    sido concluído com sucesso ao ser analisado posteriormente.
    """
    _write_json(context.directory / "failure.json", {
        "error_type": type(error).__name__,
        "message": str(error),
    })
    _write_json(context.directory / "manifest.json", {
        "schema_version": 1,
        "run_id": context.run_id,
        "status": "failed",
        "config": "config.json",
        "environment": "environment.json",
        "failure": "failure.json",
    })


def record_run(
    context: RunContext,
    result: "ExperimentResult",
    artifacts: dict[str, Path] | None = None,
) -> None:
    """Fecha uma execução bem-sucedida e vincula todos os seus artefatos.

    Args:
        context: Identidade retornada por ``create_run``.
        result: Objetos finais de dados, treino, modelo e avaliação.
        artifacts: Mapa lógico (por exemplo, ``checkpoint``) para o arquivo
            gerado. Caminhos dentro do run ficam relativos; externos ficam
            absolutos para não perder a referência.

    ``data.json`` e ``metrics.json`` são gravados antes do manifesto final. Isso
    garante que ``status=completed`` sempre represente uma execução auditável.
    """
    artifacts = artifacts or {}
    metrics = result_metrics(result)
    data_record = data_fingerprint(result.data)
    data_record["contract"] = result.data_contract
    _write_json(context.directory / "data.json", data_record)
    _write_json(context.directory / "metrics.json", metrics)
    def artifact_path(path: Path) -> str:
        try:
            return str(path.relative_to(context.directory))
        except ValueError:
            return str(path.resolve())

    _write_json(context.directory / "manifest.json", {
        "schema_version": 1,
        "run_id": context.run_id,
        "status": "completed",
        "config": "config.json",
        "environment": "environment.json",
        "data": "data.json",
        "metrics": "metrics.json",
        "artifacts": {name: artifact_path(path) for name, path in artifacts.items()},
    })

# ============================================================================
# REGISTRY
# ============================================================================
"""Registry local de modelos aprovados, baseado nos runs rastreados.

O tracker guarda *todas* as execuções; o registry guarda apenas o campeão de uma
família comparável. A promoção é deliberadamente explícita (``--promote``): uma
execução não vira referência só porque terminou sem erro.
"""


import json
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import TYPE_CHECKING, Any


if TYPE_CHECKING:
    pass  # Tipos já estão definidos em células anteriores.

__all__ = ["PromotionDecision", "comparison_key", "champion_path", "promote_candidate"]


@dataclass(frozen=True)
class PromotionDecision:
    """Resultado explicável de uma tentativa de promover um modelo."""

    promoted: bool
    reason: str
    registry_path: Path


def comparison_key(result: "ExperimentResult") -> dict[str, Any]:
    """Define quais condições devem permanecer fixas numa disputa por campeão.

    Segmentador, backend e batch ficam fora de propósito: são justamente opções
    que o experimento pode querer comparar. Cena, arquitetura, patch, split e
    seed permanecem fixos para evitar promover uma métrica não comparável.
    """
    config = result.config
    return {
        "dataset": config.dataset,
        "model_type": config.model_type,
        "two_input": config.two_input,
        "pca_components": config.pca_components,
        "test_ratio": config.test_ratio,
        "valid_ratio": config.valid_ratio,
        "seed": config.seed,
    }


def champion_path(result: "ExperimentResult", root: Path | None = None) -> Path:
    """Local determinístico do campeão da cena e arquitetura da execução."""
    base = root or output_dir() / "registry"
    return base / result.config.dataset / result.config.model_type / "champion.json"


def _score(candidate: dict[str, Any]) -> tuple[float, float, float]:
    """Ordena candidatos por AA, kappa e OA, nessa ordem para classes desbalanceadas."""
    metrics = candidate["metrics"]
    return metrics["average_accuracy"], metrics["kappa"], metrics["overall_accuracy"]


def _load(path: Path) -> dict[str, Any] | None:
    """Lê um campeão existente; ausência é normal no primeiro modelo da família."""
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def promote_candidate(
    context: "RunContext",
    result: "ExperimentResult",
    checkpoint: Path,
    root: Path | None = None,
) -> PromotionDecision:
    """Promove candidato se ele for comparável e superior ao campeão atual.

    Args:
        context: Run rastreado que produziu a execução candidata.
        result: Métricas e configuração já concluídas.
        checkpoint: Peso salvo que será apontado pelo registry.
        root: Raiz alternativa, útil em testes; padrão ``outputs/registry``.

    A regra lexicográfica é AA → kappa → OA. Se não existir campeão, o primeiro
    candidato válido entra. Se a chave de comparação for diferente, não há
    sobrescrita: o usuário deve criar uma família/critério conscientemente.
    """
    path = champion_path(result, root)
    candidate = {
        "run_id": context.run_id,
        "run_directory": str(context.directory.resolve()),
        "checkpoint": str(checkpoint.resolve()),
        "comparison_key": comparison_key(result),
        "config": result.config.to_dict(),
        "metrics": {
            "overall_accuracy": result.metrics.overall_accuracy,
            "average_accuracy": result.metrics.average_accuracy,
            "kappa": result.metrics.kappa,
        },
        "promoted_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    champion = _load(path)
    if champion is not None and champion["comparison_key"] != candidate["comparison_key"]:
        return PromotionDecision(False, "chave de comparação diferente do campeão atual", path)
    if champion is not None and _score(candidate) <= _score(champion):
        return PromotionDecision(False, "AA/kappa/OA não superou o campeão atual", path)

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(candidate, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    reason = "primeiro campeão da família" if champion is None else "superou campeão em AA/kappa/OA"
    return PromotionDecision(True, reason, path)

# ============================================================================
# OBSERVABILITY
# ============================================================================
"""Leitura e sumarização dos artefatos de MLOps já produzidos pelos runs.

Não há serviço externo: esta camada transforma os JSONs imutáveis de
``outputs/runs`` em registros comparáveis, adequados para terminal, Markdown e
integração posterior com dashboards/MLflow.
"""


import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any


__all__ = ["RunObservation", "discover_runs", "filter_runs", "runs_summary", "markdown_report"]


@dataclass(frozen=True)
class RunObservation:
    """Visão plana dos campos que importam para observar uma execução."""

    run_id: str
    directory: Path
    status: str
    config: dict[str, Any]
    metrics: dict[str, Any] | None
    train_time: float | None
    prep_time: float | None
    error: str | None = None

    def row(self) -> dict[str, Any]:
        """Converte o run para uma linha JSON/tabular sem objetos Path ou NumPy."""
        metrics = self.metrics or {}
        return {
            "run_id": self.run_id,
            "status": self.status,
            "dataset": self.config.get("dataset"),
            "model_type": self.config.get("model_type", "cnn1d"),
            "segmenter": self.config.get("segmenter", "seeds"),
            "two_input": self.config.get("two_input"),
            "epochs": self.config.get("epochs"),
            "oa": metrics.get("overall_accuracy"),
            "aa": metrics.get("average_accuracy"),
            "kappa": metrics.get("kappa"),
            "train_seconds": self.train_time,
            "prep_seconds": self.prep_time,
            "error": self.error,
            "directory": str(self.directory),
        }


def _read(path: Path) -> dict[str, Any]:
    """Lê um JSON de run; arquivos malformados são erro visível ao operador."""
    return json.loads(path.read_text(encoding="utf-8"))


def discover_runs(runs_dir: Path | None = None) -> list[RunObservation]:
    """Descobre runs concluídos, falhos ou em andamento sem modificar artefatos.

    Diretórios sem manifesto são ignorados: podem ser cópias parciais durante um
    upload. O resultado é ordenado do mais recente para o mais antigo, pois o
    identificador começa por data UTC.
    """
    base = runs_dir or output_dir(create=False) / "runs"
    if not base.exists():
        return []
    observations = []
    for directory in sorted((path for path in base.iterdir() if path.is_dir()), reverse=True):
        manifest_path = directory / "manifest.json"
        if not manifest_path.exists():
            continue
        manifest = _read(manifest_path)
        config = _read(directory / "config.json") if (directory / "config.json").exists() else {}
        metrics_file = directory / "metrics.json"
        metrics_record = _read(metrics_file) if metrics_file.exists() else {}
        failure_file = directory / "failure.json"
        failure = _read(failure_file) if failure_file.exists() else {}
        observations.append(RunObservation(
            run_id=manifest.get("run_id", directory.name), directory=directory,
            status=manifest.get("status", "unknown"), config=config,
            metrics=metrics_record.get("metrics"), train_time=metrics_record.get("train_time"),
            prep_time=metrics_record.get("prep_timings", {}).get("wall"),
            error=failure.get("message"),
        ))
    return observations


def filter_runs(
    runs: list[RunObservation], dataset: str | None = None, model_type: str | None = None,
    segmenter: str | None = None, include_incomplete: bool = False,
) -> list[RunObservation]:
    """Filtra runs sem misturar candidatos falhos nas métricas comparáveis."""
    return [
        run for run in runs
        if (include_incomplete or run.status == "completed")
        and (dataset is None or run.config.get("dataset") == dataset)
        and (model_type is None or run.config.get("model_type", "cnn1d") == model_type)
        and (segmenter is None or run.config.get("segmenter", "seeds") == segmenter)
    ]


def runs_summary(runs: list[RunObservation]) -> dict[str, Any]:
    """Calcula contagens e melhor run por AA, kappa e OA para observabilidade."""
    completed = [run for run in runs if run.status == "completed" and run.metrics]
    best = max(
        completed,
        key=lambda run: (run.metrics["average_accuracy"], run.metrics["kappa"], run.metrics["overall_accuracy"]),
        default=None,
    )
    return {
        "total": len(runs),
        "completed": len(completed),
        "failed": sum(run.status == "failed" for run in runs),
        "best_by_aa": None if best is None else best.row(),
    }


def markdown_report(runs: list[RunObservation]) -> str:
    """Renderiza um resumo textual compacto pronto para relatório, issue ou TCC."""
    summary = runs_summary(runs)
    lines = [
        "# Observabilidade de experimentos", "",
        f"Runs: {summary['total']} | concluídos: {summary['completed']} | falhos: {summary['failed']}", "",
        "| run | status | cena | modelo | segmentador | OA | AA | kappa | treino (s) |",
        "|---|---|---|---|---|---:|---:|---:|---:|",
    ]
    for run in runs:
        row = run.row()
        def metric(value):
            return "-" if value is None else f"{value:.4f}"
        lines.append(
            f"| {row['run_id']} | {row['status']} | {row['dataset'] or '-'} | "
            f"{row['model_type']} | {row['segmenter']} | {metric(row['oa'])} | "
            f"{metric(row['aa'])} | {metric(row['kappa'])} | {metric(row['train_seconds'])} |"
        )
    if summary["best_by_aa"]:
        best = summary["best_by_aa"]
        lines.extend(["", f"Melhor por AA: `{best['run_id']}` ({best['aa']:.4f})."])
    return "\n".join(lines) + "\n"


## 7. Orquestrar as variantes

### 7.1 Receitas comparáveis e paralelismo

`run_experiment` liga todas as etapas: dispositivo, dados, loaders, modelo, treino e avaliação.

In [ ]:
# ============================================================================
# PIPELINE
# ============================================================================
"""Pipeline completo: cena crua -> dados prontos -> modelo treinado -> métricas.

Este módulo é a única peça que conhece a ordem das etapas. O notebook e o CLI
`scripts/run_experiment.py` são só dois frontends para as mesmas funções, o que
garante que os dois produzam exatamente o mesmo resultado.

A preparação dos dados vive em `hsi.parallel`, que a executa com os dois ramos
(espectral e espacial) em sequência ou concorrentemente, conforme o backend.
"""


from dataclasses import dataclass
from pathlib import Path

import numpy as np


__all__ = ["prepare_data", "run_experiment", "ExperimentResult"]


def prepare_data(
    config: ExperimentConfig,
    data_dir: Path | None = None,
    scene: Scene | None = None,
    backend: str = "sequential",
) -> tuple[PreparedData, SegmentationArtifacts | None]:
    """Da cena em disco até os arrays prontos para virar tensores.

    Etapas: carga -> tabela de pixels -> remoção do background -> PCA ->
    (opcional) segmentação SEEDS -> split estratificado.

    `backend` seleciona como os dois ramos independentes são executados; veja
    `hsi.parallel`. O resultado é o mesmo em qualquer um deles — use
    `prepare_data_parallel` diretamente quando quiser também os tempos.
    """
    data, artifacts, _ = prepare_data_parallel(config, data_dir, scene, backend=backend)
    return data, artifacts


@dataclass
class ExperimentResult:
    config: ExperimentConfig
    data: PreparedData
    model: object
    training: TrainResult
    metrics: Metrics
    predicted: np.ndarray
    device: object
    prep_timings: StageTimings
    #: Resultado do contrato de dados validado antes de construir DataLoaders.
    data_contract: dict


#: Campos de `PreparedData` que dominam a memória e não são necessários para
#: relatar um trial já avaliado. `labels` e `splits` ficam: são pequenos e vários
#: relatórios os consultam.
_HEAVY_DATA_FIELDS = ("spectra", "segments", "coords")


def result_footprint(result: "ExperimentResult") -> dict[str, float]:
    """Quantos MB de arrays um resultado está segurando, por campo.

    Serve para justificar com número — e não com intuição — quando vale a pena
    liberar um resultado em vez de retê-lo.
    """
    data = getattr(result, "data", None)
    sizes = {}
    for field in _HEAVY_DATA_FIELDS:
        array = getattr(data, field, None) if data is not None else None
        if array is not None:
            sizes[f"data.{field}"] = array.nbytes / 1e6
    model = getattr(result, "model", None)
    if model is not None:
        sizes["modelo"] = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
    if getattr(result, "predicted", None) is not None:
        sizes["predicted"] = result.predicted.nbytes / 1e6
    sizes["total"] = sum(sizes.values())
    return sizes


def release_result(result: "ExperimentResult | None", keep_model: bool = False) -> None:
    """Solta os arrays pesados de um resultado que não será mais analisado.

    Por que isso existe: uma campanha de HPO instancia dezenas de
    `ExperimentResult`, e cada um segura sua própria cópia dos dados preparados.
    Em Indian Pines são 2,5 MB por trial; em Salinas, 6,5 MB. Reter dezenas de
    trials acumula sem necessidade, e o custo cresce com o tamanho da cena.

    A prática correta é a que o orientador apontou: instanciar, treinar, extrair
    as métricas (que são kilobytes), persistir o checkpoint em disco se o modelo
    ainda for necessário, e liberar o resto antes do próximo trial.

    O que sobrevive: config, métricas, matriz de confusão, histórico de treino,
    tempos e splits — tudo o que os relatórios e os PDFs consomem.
    """
    if result is None:
        return
    data = getattr(result, "data", None)
    if data is not None:
        for field in _HEAVY_DATA_FIELDS:
            # PreparedData é frozen: a atribuição precisa contornar o __setattr__.
            object.__setattr__(data, field, None)
    if not keep_model:
        result.model = None
    result.predicted = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


def run_experiment(
    config: ExperimentConfig | None = None,
    data_dir: Path | None = None,
    backend: str = "sequential",
    verbose: bool = True,
) -> ExperimentResult:
    """Execução completa de ponta a ponta — o que o CLI chama."""
    config = config or ExperimentConfig()

    device = select_device(verbose=verbose)
    configure_determinism(config.deterministic, verbose=verbose)

    data, _, prep_timings = prepare_data_parallel(config, data_dir, backend=backend)
    data_contract = validate_prepared_data(data)
    if verbose:
        print("\n" + config.summary())
        print("Amostras:", data.splits.sizes(), "| classes:", data.n_classes)
        print("\n" + prep_timings.summary())

    loaders = build_loaders(data, config, device)

    # A semente é reposta imediatamente antes de construir o modelo: são os pesos
    # iniciais que dependem do RNG global do PyTorch.
    seed_everything(config.seed)
    # Cada arquitetura declara seu próprio layout: 1D=(1,Kb), 2D=(3,P,P) e
    # 3D=(1,B,P,P). O modelo nunca precisa inferir ou rearranjar dados sozinho.
    model, criterion, optimizer = build_model(
        config, data.input_shape(config.model_type), data.n_classes, device,
        n_segments=data.n_segments,
    )
    if verbose:
        print("\nParâmetros treináveis:", model.n_parameters)

    training = fit(model, loaders, criterion, optimizer, config, device, verbose=verbose)
    if verbose:
        print("\n" + training.summary())

    metrics, predicted = evaluate(
        model, loaders["test"], criterion, device, data.labels[data.splits.test]
    )
    if verbose:
        print("\n" + metrics.report)
        print(metrics.summary())

    return ExperimentResult(
        config=config,
        data=data,
        model=model,
        training=training,
        metrics=metrics,
        predicted=predicted,
        device=device,
        prep_timings=prep_timings,
        data_contract=data_contract,
    )

# ============================================================================
# PARALELISMO POR PARTIÇÃO DA CENA
# ============================================================================
"""Dividir a imagem em N partes e processá-las concorrentemente.

POR QUE ISTO EXISTE. O paralelismo de dois ramos (`prepare_data_parallel`) tem teto
estrutural: a segmentação é 78–93% do pré-processamento, então dois ramos
desbalanceados nunca passam de ~1,2× — medido nas três cenas. Pior, o
pré-processamento inteiro é 0,9% do tempo de execução, e o treino é o resto.

Particionar a cena ataca o lado certo. As partes têm tamanho parecido, então o teto
é ≈ N em vez de 1,2×, e o que fica paralelo é o **treino**, que é onde o tempo está.

A SEGMENTAÇÃO É FEITA UMA VEZ, NA CENA INTEIRA, antes de particionar. Isso não é
detalhe: segmentar cada parte isoladamente produziria superpixels diferentes nas
fronteiras, e os ids deixariam de ser comparáveis entre partições. Fazer uma vez
custa pouco (0,15 a 1,9 s) e mantém o contrato.

DOIS MODOS, porque respondem a perguntas diferentes:

- ``"replicado"`` — um modelo treinado com todos os dados, replicado para inferir as
  partições em paralelo. O resultado é **idêntico** ao monolítico; o que se mede é
  puro ganho de velocidade na inferência.
- ``"independente"`` — cada partição treina seu próprio modelo, com os dados dela.
  Paraleliza o treino, que é o caro, mas cada modelo vê 1/N das amostras. Como
  medimos que cortar o treino de 100% para 10% custa 14 pontos de OA, o custo em
  acurácia aqui é esperado e precisa ser quantificado, não presumido.
"""

import multiprocessing
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor


def particionar_cena(coords: np.ndarray, scene_shape: tuple[int, int],
                     n_particoes: int, modo: str = "faixas",
                     seed: int = 42) -> np.ndarray:
    """Atribui cada amostra a uma partição.

    ``"faixas"`` corta a cena em faixas horizontais. Faixas, e não um xadrez: elas
    preservam a contiguidade espacial de cada parte, que é como o processamento
    seria distribuído num cenário real — cada nó recebe um pedaço contíguo da
    imagem. O corte é por linha, com fronteiras escolhidas para equilibrar a
    **quantidade de pixels rotulados**, e não a área, porque é o número de amostras
    que determina o tempo de treino.

    ``"aleatoria"`` embaralha os pixels e distribui sem olhar a posição. **É o
    controle do experimento, não uma alternativa de produção**: como o custo
    computacional é idêntico ao das faixas, qualquer diferença de acurácia entre os
    dois modos é atribuível à localização espacial, e não à partição em si. Serve
    para testar se um eventual ganho das faixas é especialização legítima ou mais
    um atalho posicional, como o vazamento da entrada 3 do diário.
    """
    if modo not in {"faixas", "aleatoria"}:
        raise ValueError("modo de partição deve ser 'faixas' ou 'aleatoria'.")
    total = len(np.asarray(coords))
    if modo == "faixas":
        ordem = np.argsort(np.asarray(coords)[:, 0], kind="stable")
    else:
        ordem = np.random.default_rng(seed).permutation(total)
    particao = np.empty(total, dtype=np.int64)
    for indice, bloco in enumerate(np.array_split(ordem, n_particoes)):
        particao[bloco] = indice
    return particao


def _fatiar(data: "PreparedData", mascara: np.ndarray) -> "PreparedData":
    """Recorta `PreparedData` para um subconjunto de amostras, reindexando os splits.

    Os splits guardam posições absolutas; depois do recorte elas mudam. O mapa
    `posicao` traduz índice antigo → novo e descarta o que saiu da partição.
    """
    indices = np.flatnonzero(mascara)
    posicao = -np.ones(len(mascara), dtype=np.int64)
    posicao[indices] = np.arange(len(indices))
    traduz = lambda bloco: posicao[bloco][posicao[bloco] >= 0]
    return PreparedData(
        spectra=data.spectra[indices],
        labels=data.labels[indices],
        segments=None if data.segments is None else data.segments[indices],
        splits=Splits(train=traduz(data.splits.train), val=traduz(data.splits.val),
                      test=traduz(data.splits.test)),
        coords=None if data.coords is None else data.coords[indices],
        scene_shape=data.scene_shape,
    )


def _treinar_e_inferir(tarefa: tuple) -> tuple[int, np.ndarray, np.ndarray, float]:
    """Treina (quando preciso) e infere uma partição. Devolve `(k, previsto, real, s)`.

    Função de módulo, e não closure, porque precisa ser serializável para o backend
    de processos — mesma razão de `spatial_branch` na preparação dos dados.
    """
    k, pedaco, config, n_classes, n_segments, modelo_pronto = tarefa
    marca = time.perf_counter()
    device = torch.device("cpu")
    loaders = build_loaders(pedaco, config, device)
    if modelo_pronto is None:
        seed_everything(config.seed)
        modelo, criterio, otimizador = build_model(
            config, pedaco.input_shape(), n_classes, device, n_segments=n_segments)
        fit(modelo, loaders, criterio, otimizador, config, device, verbose=False)
    else:
        modelo = modelo_pronto
    verdadeiro = pedaco.labels[pedaco.splits.test]
    _, previsto = evaluate(modelo, loaders["test"], nn.CrossEntropyLoss(),
                           device, verdadeiro)
    return k, previsto, verdadeiro, time.perf_counter() - marca


@dataclass
class ResultadoParticionado:
    """Saída de uma execução particionada, com os tempos que justificam o desenho."""

    config: ExperimentConfig
    modo: str
    particao_modo: str
    executor: str
    n_particoes: int
    metrics: "Metrics"
    predicted: np.ndarray
    tempo_preparo: float
    tempo_segmentacao: float
    #: Tempo de parede do bloco que roda concorrente. No modo "independente" inclui
    #: treino e inferência; no "replicado", só a inferência — o treino é único e
    #: sequencial por definição, e somá-lo aqui mascararia o speedup real.
    tempo_paralelo: float
    #: Treino único do modo "replicado". Zero no modo "independente", onde o treino
    #: já está dentro de `tempo_paralelo`.
    tempo_treino_sequencial: float
    tempos_por_particao: list[float]
    amostras_por_particao: list[int]
    parametros_totais: int

    @property
    def soma_das_partes(self) -> float:
        """Soma dos tempos individuais das partições.

        **Atenção ao interpretar.** Quando o executor é concorrente, cada tempo foi
        cronometrado já sob disputa com os outros. Se as N partições começam e
        terminam juntas, `soma ÷ parede` tende a N por construção — e não porque o
        paralelismo esteja funcionando. Para speedup honesto, compare a parede deste
        arranjo com a de uma execução `executor="sequencial"`, que mede o custo real
        de processar as partições uma a uma.
        """
        return sum(self.tempos_por_particao)

    @property
    def tempo_total(self) -> float:
        """Parede de ponta a ponta: preparo + treino sequencial + bloco paralelo."""
        return self.tempo_preparo + self.tempo_treino_sequencial + self.tempo_paralelo

    @property
    def overhead(self) -> float:
        """Custo de coordenar: parede menos a partição mais lenta."""
        return self.tempo_paralelo - max(self.tempos_por_particao, default=0.0)

    @property
    def teto_de_amdahl(self) -> float:
        """Teto do speedup: soma das partes dividida pela mais lenta.

        Com partições equilibradas isto tende a N — é a diferença em relação ao
        paralelismo de dois ramos, onde o desequilíbrio fixava o teto em ~1,2×.
        """
        maior = max(self.tempos_por_particao, default=0.0)
        return self.soma_das_partes / maior if maior else float("nan")

    def resumo(self) -> str:
        linhas = [
            f"modo: {self.modo} | partição: {self.particao_modo} | "
            f"executor: {self.executor} | {self.n_particoes} partes",
            f"  segmentação (uma vez):  {self.tempo_segmentacao:7.3f}s",
            f"  bloco paralelo:         {self.tempo_paralelo:7.3f}s"
            + (f"  (+ treino sequencial {self.tempo_treino_sequencial:.3f}s)"
               if self.tempo_treino_sequencial else ""),
            f"  soma das partes:        {self.soma_das_partes:7.3f}s  "
            f"(speedup {self.soma_das_partes / max(self.tempo_paralelo, 1e-9):.2f}x, "
            f"teto {self.teto_de_amdahl:.2f}x)",
            f"  overhead:               {self.overhead:+7.3f}s",
            f"  tempo total:            {self.tempo_total:7.3f}s",
            f"  amostras por partição:  {self.amostras_por_particao}",
        ]
        return "\n".join(linhas)


def run_experiment_particionado(
    config: ExperimentConfig,
    n_particoes: int = 4,
    modo: str = "independente",
    particao_modo: str = "faixas",
    executor: str = "thread",
    data_dir: Path | None = None,
    backend: str = "thread",
    verbose: bool = True,
) -> ResultadoParticionado:
    """Executa o pipeline dividindo a cena em `n_particoes` processadas em paralelo.

    Devolve métricas calculadas sobre o conjunto de teste **inteiro**, remontado a
    partir das predições de cada partição — e não a média das partições, que seria
    outra coisa e não comparável com o pipeline monolítico.
    """
    if modo not in {"replicado", "independente"}:
        raise ValueError("modo deve ser 'replicado' ou 'independente'.")
    if executor not in {"sequencial", "thread", "process"}:
        raise ValueError("executor deve ser 'sequencial', 'thread' ou 'process'.")
    if executor == "process" and modo == "replicado":  # noqa: SIM102
        raise ValueError(
            "modo 'replicado' com executor 'process' exigiria serializar o modelo "
            "treinado para cada worker; use 'thread' ou o modo 'independente'."
        )
    if n_particoes < 2:
        raise ValueError("n_particoes deve ser pelo menos 2.")

    device = select_device(verbose=False)
    configure_determinism(config.deterministic, verbose=False)

    inicio = time.perf_counter()
    data, _, prep_timings = prepare_data_parallel(config, data_dir, backend=backend)
    tempo_preparo = time.perf_counter() - inicio
    validate_prepared_data(data)

    if data.coords is None:
        raise ValueError("partição espacial exige coordenadas em PreparedData.")
    particao = particionar_cena(data.coords, data.scene_shape, n_particoes,
                                particao_modo, config.seed)
    pedacos = [_fatiar(data, particao == k) for k in range(n_particoes)]

    if modo == "replicado":
        # Treina uma vez com a cena inteira; o paralelismo é só da inferência.
        marca = time.perf_counter()
        loaders = build_loaders(data, config, device)
        seed_everything(config.seed)
        modelo, criterio, otimizador = build_model(
            config, data.input_shape(), data.n_classes, device, n_segments=data.n_segments,
        )
        fit(modelo, loaders, criterio, otimizador, config, device, verbose=False)
        tempo_treino_unico = time.perf_counter() - marca
        modelos = [modelo] * n_particoes
    else:
        tempo_treino_unico = 0.0

    tarefas = [
        (k, pedacos[k], config, data.n_classes, data.n_segments,
         modelos[k] if modo == "replicado" else None)
        for k in range(n_particoes)
    ]
    marca = time.perf_counter()
    if executor == "sequencial":
        saidas = sorted(_treinar_e_inferir(t) for t in tarefas)
    elif executor == "thread":
        with ThreadPoolExecutor(max_workers=n_particoes) as pool:
            saidas = sorted(pool.map(_treinar_e_inferir, tarefas))
    else:
        # `spawn`, e não o `fork` padrão do Linux: o PyTorch recusa fork depois que
        # o autograd rodou em threads ("Unable to handle autograd's threading in
        # combination with fork-based multiprocessing"). O preço do spawn é que cada
        # worker sobe um interpretador limpo e reimporta tudo — custo que aparece
        # inteiro na parede e precisa ser reportado, não escondido.
        contexto = multiprocessing.get_context("spawn")
        with ProcessPoolExecutor(max_workers=n_particoes, mp_context=contexto) as pool:
            saidas = sorted(pool.map(_treinar_e_inferir, tarefas))
    tempo_paralelo = time.perf_counter() - marca

    previstos = np.concatenate([s[1] for s in saidas])
    verdadeiros = np.concatenate([s[2] for s in saidas])
    tempos = [s[3] for s in saidas]

    confusao = np.zeros((data.n_classes, data.n_classes), dtype=np.int64)
    np.add.at(confusao, (verdadeiros, previstos), 1)
    por_classe, media = _average_accuracy(confusao)
    acertos = int(np.trace(confusao))
    total = int(confusao.sum())
    esperado = float((confusao.sum(0) * confusao.sum(1)).sum() / (total * total))
    oa = acertos / total
    metrics = Metrics(
        overall_accuracy=oa, average_accuracy=media,
        kappa=(oa - esperado) / (1 - esperado) if esperado < 1 else 0.0,
        per_class_accuracy=por_classe, confusion=confusao,
        test_loss=float("nan"), test_accuracy=oa,
        report=f"partições={n_particoes} modo={modo} amostras de teste={total}",
    )

    resultado = ResultadoParticionado(
        config=config, modo=modo, particao_modo=particao_modo, executor=executor,
        n_particoes=n_particoes,
        metrics=metrics, predicted=previstos,
        tempo_preparo=tempo_preparo,
        tempo_segmentacao=prep_timings.stages.get("espacial", 0.0),
        tempo_paralelo=tempo_paralelo,
        tempo_treino_sequencial=tempo_treino_unico,
        tempos_por_particao=tempos,
        amostras_por_particao=[len(p.labels) for p in pedacos],
        parametros_totais=sum(m.n_parameters for m in
                              (modelos if modo == "replicado" else [])) or 0,
    )
    if verbose:
        print(resultado.resumo())
        print(f"  OA={metrics.overall_accuracy:.4f} AA={metrics.average_accuracy:.4f} "
              f"Kappa={metrics.kappa:.4f}")
    return resultado


# ============================================================================
# VARIANTS
# ============================================================================
"""As duas versões do pipeline que o trabalho compara.

`BASELINE` reproduz a implementação de referência de Sydney Matheus de Souza: fluxo
estritamente sequencial, com os mesmos hiperparâmetros do notebook original. É a
linha de base contra a qual todo ganho é medido — e ela é reproduzida **bit a bit**
por este código (mesmo histórico de perda por época, mesmos pesos finais, mesmos
OA/AA/kappa).

`OPTIMIZED` é a nossa implementação: mesmo modelo e mesmos hiperparâmetros, mas com
as etapas independentes da preparação executadas concorrentemente (ver `hsi.parallel`).

Os hiperparâmetros são idênticos nas duas de propósito. É isso que torna a comparação
interpretável: qualquer diferença de tempo vem da estratégia de execução, e as
métricas de qualidade têm de sair iguais — se não saírem, a otimização mudou o que o
código calcula, e é um bug, não um ganho.


    base = run_variant(BASELINE, dataset="PU")
    nossa = run_variant(OPTIMIZED, dataset="PU")
    print(nossa.speedup_over(base))
"""


import time
from dataclasses import dataclass, field, replace
import gc
from pathlib import Path


__all__ = ["Variant", "BASELINE", "OPTIMIZED", "VARIANTS", "VariantRun", "run_variant"]


@dataclass(frozen=True)
class Variant:
    """Uma estratégia de execução do pipeline.

    `overrides` são aplicados sobre a `ExperimentConfig` base; hoje as duas variantes
    compartilham os mesmos hiperparâmetros, e o campo existe para quando a otimização
    automática (Grid Search / Bayesiana) passar a propor configurações diferentes.
    """

    key: str
    name: str
    description: str
    backend: str
    overrides: dict = field(default_factory=dict)

    def config(self, **kwargs) -> ExperimentConfig:
        """Configuração desta variante, aceitando ajustes pontuais (dataset, épocas...)."""
        return replace(ExperimentConfig(**kwargs), **self.overrides) if self.overrides \
            else ExperimentConfig(**kwargs)


BASELINE = Variant(
    key="baseline",
    name="Baseline (Souza, 2023)",
    description=(
        "Fluxo sequencial da implementação de referência: PCA espectral, depois PCA de "
        "3 componentes, composição RGB e segmentação SEEDS, e só então o treino."
    ),
    backend="sequential",
)

OPTIMIZED = Variant(
    key="optimized",
    name="Nossa implementação",
    description=(
        "Mesmo modelo e mesmos hiperparâmetros, com os ramos espectral e espacial da "
        "preparação executados concorrentemente; sincronização apenas na fusão."
    ),
    backend="thread",
)

VARIANTS = {v.key: v for v in (BASELINE, OPTIMIZED)}


@dataclass
class VariantRun:
    """Resultado de uma variante, com os tempos separados por etapa."""

    variant: Variant
    result: ExperimentResult
    total_time: float

    @property
    def prep_time(self) -> float:
        return self.result.prep_timings.wall

    @property
    def train_time(self) -> float:
        return self.result.training.total_time

    @property
    def metrics(self):
        return self.result.metrics

    def speedup_over(self, baseline: "VariantRun") -> dict[str, float]:
        """Speedup por etapa e no total, em relação a outra execução."""
        return {
            "preparação": baseline.prep_time / self.prep_time if self.prep_time else float("nan"),
            "treino": baseline.train_time / self.train_time if self.train_time else float("nan"),
            "total": baseline.total_time / self.total_time if self.total_time else float("nan"),
        }

    def matches(self, other: "VariantRun") -> bool:
        """As duas variantes chegaram exatamente ao mesmo resultado de classificação?"""
        a, b = self.metrics, other.metrics
        return (
            a.overall_accuracy == b.overall_accuracy
            and a.average_accuracy == b.average_accuracy
            and a.kappa == b.kappa
        )

    def summary(self) -> str:
        return (
            f"{self.variant.name}\n"
            f"  preparação: {self.prep_time:7.3f}s  "
            f"(backend: {self.result.prep_timings.backend})\n"
            f"  treino:     {self.train_time:7.3f}s  "
            f"({len(self.result.training.history)} épocas)\n"
            f"  total:      {self.total_time:7.3f}s\n"
            f"  OA: {self.metrics.overall_accuracy:.6f} | "
            f"AA: {self.metrics.average_accuracy:.6f} | "
            f"kappa: {self.metrics.kappa:.6f}"
        )


def run_variant(
    variant: Variant,
    data_dir: Path | None = None,
    verbose: bool = False,
    **config_kwargs,
) -> VariantRun:
    """Executa uma variante de ponta a ponta e mede o tempo total."""
    config = variant.config(**config_kwargs)
    started = time.perf_counter()
    result = run_experiment(config, data_dir=data_dir, backend=variant.backend, verbose=verbose)
    return VariantRun(variant=variant, result=result, total_time=time.perf_counter() - started)


## 8. Executar o experimento

### 8.0 Controle de execução

A execução pesada é intencionalmente protegida. Mude `RUN_EXPERIMENT` para `True` somente quando quiser treinar; assim, `Run All` pode ser usado para revisar e validar o notebook sem iniciar um treino de 60 épocas.

In [ ]:
RUN_EXPERIMENT = True

if RUN_EXPERIMENT:
    result = run_experiment(config, backend='thread', verbose=True)
    pdf_report = generate_standalone_pdf_report(result)
    print(f"\nRelatório PDF: {pdf_report}")
    print("Experimento concluído com sucesso.")
    print(result.metrics.summary())
else:
    print('Treino não iniciado. Defina RUN_EXPERIMENT = True para executar.')


### 8.1 Painel visual do resultado

Quando o treino estiver habilitado, este painel mostra convergência, métricas globais e por classe, matriz de confusão, mapa de erros, custo de inferência/memória, perfil dos ramos paralelos e a melhor época/motivo da parada antecipada.

In [ ]:
if RUN_EXPERIMENT:
    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    evaluation_figure, per_class_metrics, extra_metrics = plot_evaluation_dashboard(result, scene_for_visuals)
    display(evaluation_figure)
    display(plot_per_class_metrics(per_class_metrics))
    display(plot_inference_profile_chart(profile_inference(result)))
    display(plot_parallel_profile(result.prep_timings))
    show_glossary('avaliacao')
else:
    print('Execute o treino para gerar o painel de avaliação.')


### 8.2 O que a CNN está usando para decidir

O painel usa um pixel do conjunto de teste: os mapas são filtros × posições espectrais ao longo das bandas.

In [ ]:
if RUN_EXPERIMENT:
    x_explain, seg_explain, true_class = explanation_sample(result, split='test', batch_index=0)
    activations, predicted_class = capture_activations(result.model, x_explain, seg_explain)
    print(f'Classe real: {true_class + 1} | classe prevista: {predicted_class + 1}')

    for figure in plot_activation_atlases(activations).values():
        display(figure)
    for figure in plot_maxpool_effect(activations).values():
        display(figure)

    cam, cam_class, cam_layer = grad_cam(result.model, x_explain, seg_explain)
    display(plot_grad_cam(cam, result.config.model_type, cam_class, cam_layer))
    display(plot_input_saliency(result.model, x_explain, seg_explain, result.config.model_type))
    display(plot_prediction_uncertainty(result, scene_for_visuals))
    show_glossary('explicabilidade')
else:
    print('Execute o treino para gerar ativações, Grad-CAM e saliência.')


### 8.4 HPO por CNN e segmentador

Habilite esta campanha para buscar automaticamente a melhor combinação. Ela sempre cobre SEEDS, SLIC, LSC e Watershed; em seguida renderiza os outputs e o dashboard do vencedor.

In [ ]:
RUN_HPO = True
HPO_TRIALS_PER_CNN = 8      # mínimo 4: um candidato por segmentador
HPO_SEARCH_EPOCHS = 10      # orçamento reduzido para a busca; retreine vencedor depois
HPO_STRATEGY = 'tpe'        # 'random' | 'tpe' (bayesiano) | 'genetic'

# Compara as três estratégias com orçamento idêntico. Custa 3x o HPO normal,
# então é opt-in separado.
RUN_STRATEGY_COMPARISON = False
STRATEGY_TRIALS = 20

if RUN_HPO:
    hpo_table, hpo_winners = automated_hpo(
        config,
        trials_per_cnn=HPO_TRIALS_PER_CNN,
        search_epochs=HPO_SEARCH_EPOCHS,
        backend='thread',
        strategy=HPO_STRATEGY,
    )
    for cnn in MODEL_TYPES:
        hpo_figure, hpo_output = plot_hpo_by_cnn(hpo_table, cnn)
        display(hpo_figure)

    hpo_compilation, hpo_winner_table = plot_hpo_compilation(hpo_table)
    display(hpo_compilation)

    # O que a busca de fato variou, eixo a eixo.
    hpo_space_figure = plot_hpo_space(hpo_table)
    display(hpo_space_figure)
    plt.close(hpo_space_figure)

    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    hpo_dashboards = {name: render_cnn_dashboard(result, scene_for_visuals) for name, result in hpo_winners.items()}
    for name, dashboard in hpo_dashboards.items():
        display_cnn_dashboard(f'{name} — vencedor HPO', dashboard)
    hpo_final_figure, hpo_final_summary, _ = plot_final_cnn_compilation(hpo_dashboards)
    display(hpo_final_figure)
    plt.close(hpo_final_figure)
    show_glossary('hpo', 'comparacao')
else:
    print('HPO desativado. Defina RUN_HPO = True para otimizar cada CNN e mostrar seus outputs.')

if RUN_STRATEGY_COMPARISON:
    strategy_comparison = compare_hpo_strategies(
        config,
        trials_per_cnn=STRATEGY_TRIALS,
        search_epochs=HPO_SEARCH_EPOCHS,
        backend='thread',
        model_types=('cnn1d',),
    )
    display(hpo_strategy_summary(strategy_comparison))
    strategy_figure = plot_hpo_strategies(strategy_comparison)
    display(strategy_figure)
    plt.close(strategy_figure)


### 8.5 Campanha auditável com múltiplas execuções

Esta é a execução MLOps para resultados comparáveis: quatro receitas por CNN, quatro segmentadores e três seeds por padrão. Todos os JSONs, checkpoints, fingerprints e relatórios ficam em `outputs/campaigns/`, com links para os runs individuais em `outputs/runs/`.

In [ ]:
RUN_AUDITED_CAMPAIGN = True
# O orçamento de screening é o mesmo definido na seção 8.4: mudar lá muda aqui,
# em vez de duas seções discordarem silenciosamente sobre o tamanho da busca.
FINAL_HPO_SEEDS = (11, 29, 42)
FINAL_HPO_EPOCHS = config.epochs  # orçamento completo, só para vencedores HPO

if RUN_AUDITED_CAMPAIGN:
    # 1) Screening: cobre todos os segmentadores e escolhe por validação.
    hpo_table, _ = automated_hpo(
        config,
        trials_per_cnn=HPO_TRIALS_PER_CNN,
        search_epochs=HPO_SEARCH_EPOCHS,
        backend='thread',
    )
    # 2) Final: repete exclusivamente o melhor HPO de cada CNN, com várias seeds.
    campaign = run_best_hpo_campaign(
        hpo_table,
        final_epochs=FINAL_HPO_EPOCHS,
        seeds=FINAL_HPO_SEEDS,
        backend='thread',
    )
    print(f"Campanha dos melhores HPO: {campaign['directory']}")
    campaign_figure = plot_campaign_analysis(campaign['records'], campaign['analysis'])
    if campaign_figure is not None:
        display(campaign_figure)
        plt.close(campaign_figure)
    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    best_hpo_results = best_hpo_results_by_cnn(campaign)
    best_hpo_dashboards = {name: render_cnn_dashboard(result, scene_for_visuals) for name, result in best_hpo_results.items()}
    for name, dashboard in best_hpo_dashboards.items():
        display_cnn_dashboard(f'{name} — melhor HPO', dashboard)
    best_hpo_figure, best_hpo_summary, _ = plot_final_cnn_compilation(best_hpo_dashboards)
    display(best_hpo_figure)
    plt.close(best_hpo_figure)
    show_glossary('campanha', 'comparacao')
else:
    print('Campanha desativada. Quando habilitada, ela usa sempre o melhor HPO de cada CNN.')


### 8.6 Explicar o resultado diretamente na imagem

Use esta seção após treinar uma CNN individual. Ela remonta tudo no espaço da cena: segmentos, classes, métricas, erros, incerteza e exemplos Grad-CAM escolhidos automaticamente.

In [ ]:
if RUN_EXPERIMENT:
    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    spatial_explanation = render_spatial_explanation(result, scene_for_visuals)
    for title, figure in spatial_explanation['figures'].items():
        print(title)
        display(figure)
        plt.close(figure)
    display(plot_segment_metrics_visual(spatial_explanation['segmentos']))
    show_glossary('espacial', 'explicabilidade')
else:
    print('Execute uma CNN na seção 8 antes de reconstruir suas decisões na imagem.')


### 8.7 HPO de segmentação + batches de 50 em 50

A busca escolhe primeiro o melhor HPO por CNN, incluindo os parâmetros do segmentador. Depois percorre uma **escada aritmética fixa**: 50, 100, 150… Cada passo acrescenta exatamente 50 amostras, isolando o efeito do batch e permitindo medir o ganho marginal de throughput, a saturação e o limite de memória sem saltos arbitrários.

O batch recomendado é o de maior throughput cuja validação fica a no máximo **0,005** (0,5 ponto percentual) do melhor batch. Início, passo, máximo, tolerância e justificativa ficam gravados no plano auditável.

In [ ]:
RUN_BATCH_SWEEP = True
# Escada uniforme: +50 por tentativa mede o ganho marginal de forma comparável.
BATCH_START = 50
BATCH_STEP = 50
BATCH_MAX = 300       # fecha a escada: 50, 100, 150, ..., 300
BATCH_QUALITY_TOLERANCE = .005  # aceita até 0,5 p.p. abaixo da melhor validação
BATCH_SWEEP_EPOCHS = 10

batch_schedule = list(range(BATCH_START, BATCH_MAX + 1, BATCH_STEP))
assert batch_schedule[-1] == BATCH_MAX, "BATCH_MAX deve fechar a escada."
print(f"Escada de batches: {batch_schedule} | tolerância: {BATCH_QUALITY_TOLERANCE:.1%}")

if RUN_BATCH_SWEEP:
    if 'hpo_table' not in globals():
        hpo_table, _ = automated_hpo(config, trials_per_cnn=8, search_epochs=globals().get('HPO_SEARCH_EPOCHS', 10), backend='thread')
    batch_sweeps = run_batch_sweeps_for_hpo_winners(
        hpo_table,
        batch_start=BATCH_START,
        batch_max=BATCH_MAX,
        step=BATCH_STEP,
        epochs=BATCH_SWEEP_EPOCHS,
        quality_tolerance=BATCH_QUALITY_TOLERANCE,
        backend='thread',
    )
    for cnn, sweep in batch_sweeps.items():
        print(f"{cnn.upper()} | artefatos: {sweep['directory']}")
        figure = plot_batch_sweep(sweep)
        display(figure)
        plt.close(figure)
        print('Batch recomendado:', sweep['winner'])
    show_glossary('batch')
else:
    print('Sweep desativado. Defina RUN_BATCH_SWEEP = True para testar batches de 50 em 50 nos vencedores HPO.')


## 9. Observar e comparar resultados

### 9.1 Registro local e promoção de modelos

Esta célula lê os runs já gravados, gera painéis comparativos e alertas quando houver. Após uma execução, filtre pelo `run_id` retornado para inspecionar um experimento específico.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

runs = filter_runs(discover_runs(), dataset=config.dataset, include_incomplete=True)
summary = runs_summary(runs)
print(summary)

if not runs:
    print('Nenhum run encontrado para os filtros selecionados.')

plot_runs_dashboard(runs)


all_runs_frame = load_all_audited_runs()
all_figures = all_run_figures(all_runs_frame)
for plot_key, figure in all_figures.items():
    display(figure)
    show_all_run_note(plot_key)
    plt.close(figure)
all_runs_pdf, _, _ = generate_all_runs_pdf()
print(f"Relatório visual consolidado de todos os runs: {all_runs_pdf}")


## Próximos ajustes

Para reduzir consumo de memória, diminua `batch_size` ou aumente `gradient_accumulation_steps`. Para promover um modelo, use `ModelRegistry(output_dir).promote(run_id)` após conferir as métricas.

Para ampliar a busca de hiperparâmetros, edite `HPO_SPACE` na seção 5.4 — acrescentar um eixo é acrescentar uma chave, e `hpo_search_space` já registra o espaço efetivo de cada CNN. Um hipertunador guiado (TPE/bayesiano) ou um algoritmo evolutivo entrariam substituindo apenas `hpo_candidates`, que é o único ponto que decide *quais* configurações treinar; `automated_hpo` e toda a auditoria seguem iguais.